In [1]:
# ============================================================
# CELL 1: NOTEBOOK BOOTSTRAP
# 04_GHSL_Grid_Year_Pipeline.ipynb
#
# Purpose:
#   - Mount Google Drive
#   - Use remote_exec.sh as the only AWS connection source
#   - Define one reusable remote execution function
#
# This cell does NOT:
#   - contain an AWS IP
#   - test SSH separately
#   - install software
#   - download or upload GHSL data
#   - start, stop, or format HDFS
#   - run Spark processing
# ============================================================

from google.colab import drive
from pathlib import Path
import subprocess


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive",
    force_remount=False,
)


# ------------------------------------------------------------
# 2. Connection helper
#
# remote_exec.sh remains the single source of truth for:
#   - current AWS public IP
#   - SSH username
#   - SSH key
#   - SSH options
# ------------------------------------------------------------

REMOTE_EXEC = Path(
    "/content/drive/MyDrive/remote_exec.sh"
)

if not REMOTE_EXEC.is_file():
    raise FileNotFoundError(
        f"Remote execution helper not found: {REMOTE_EXEC}"
    )


# ------------------------------------------------------------
# 3. Reusable remote command function
# ------------------------------------------------------------

def run_remote(
    command,
    timeout=300,
    show_output=True,
):
    """
    Execute one command on AWS through remote_exec.sh.

    Parameters
    ----------
    command : str
        Shell command to execute remotely.

    timeout : int
        Maximum notebook-side waiting time in seconds.

    show_output : bool
        Print remote stdout and stderr when True.

    Returns
    -------
    subprocess.CompletedProcess
        Completed remote command result.

    Raises
    ------
    RuntimeError
        If the remote command returns a nonzero exit code.
    """

    result = subprocess.run(
        [
            "bash",
            str(REMOTE_EXEC),
            command,
        ],
        text=True,
        capture_output=True,
        timeout=timeout,
    )

    if show_output and result.stdout:
        print(result.stdout)

    if show_output and result.stderr:
        print("REMOTE STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            "Remote command failed with return code "
            f"{result.returncode}."
        )

    return result


# ------------------------------------------------------------
# 4. Project constants
#
# These are data paths only. No processing happens here.
# ------------------------------------------------------------

HDFS_ROOT = "hdfs://localhost:9000"

HDFS_RAW_ROOT = (
    f"{HDFS_ROOT}/data/ghsl/raw/india"
)

HDFS_METADATA_CATALOG = (
    f"{HDFS_ROOT}/data/ghsl/metadata/india/"
    "geotiff_catalog"
)

HDFS_CURATED_ROOT = (
    f"{HDFS_ROOT}/data/ghsl/curated/india"
)

PILOT_CITY = "bengaluru"
PILOT_EPOCH = 1975
PILOT_PRODUCT = "population"

INPUT_RESOLUTION_METERS = 100
OUTPUT_RESOLUTION_METERS = 1000

EXPECTED_RAW_FILE_COUNT = 325


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("=" * 80)
print("NOTEBOOK 04 BOOTSTRAP COMPLETE")
print("=" * 80)
print("Remote helper:", REMOTE_EXEC)
print("Pilot city:", PILOT_CITY)
print("Pilot product:", PILOT_PRODUCT)
print("Pilot epoch:", PILOT_EPOCH)
print(
    "Target aggregation:",
    f"{INPUT_RESOLUTION_METERS} m -> "
    f"{OUTPUT_RESOLUTION_METERS} m",
)
print()
print("No remote processing was executed.")
print("CELL 1: SUCCESS")

Mounted at /content/drive
NOTEBOOK 04 BOOTSTRAP COMPLETE
Remote helper: /content/drive/MyDrive/remote_exec.sh
Pilot city: bengaluru
Pilot product: population
Pilot epoch: 1975
Target aggregation: 100 m -> 1000 m

No remote processing was executed.
CELL 1: SUCCESS


In [3]:
# ============================================================
# CELL 1A: RESTORE SSH KEY TO THE FRESH COLAB RUNTIME
#
# remote_exec.sh expects:
#   /content/hadoop-hdfs.pem
# ============================================================

from pathlib import Path
import shutil
import os


LOCAL_KEY = Path("/content/hadoop-hdfs.pem")

DRIVE_KEY_CANDIDATES = [
    Path("/content/drive/MyDrive/hadoop-hdfs.pem"),
    Path(
        "/content/drive/MyDrive/"
        "Colab Notebooks/hadoop-hdfs.pem"
    ),
]


if LOCAL_KEY.is_file():
    os.chmod(LOCAL_KEY, 0o600)

    print(
        "SSH key already available:",
        LOCAL_KEY,
    )

else:
    matching_keys = [
        path
        for path in DRIVE_KEY_CANDIDATES
        if path.is_file()
    ]

    if not matching_keys:
        raise FileNotFoundError(
            "Could not find hadoop-hdfs.pem in Google Drive.\n"
            "Checked:\n"
            + "\n".join(
                str(path)
                for path in DRIVE_KEY_CANDIDATES
            )
        )

    source_key = matching_keys[0]

    shutil.copy2(
        source_key,
        LOCAL_KEY,
    )

    os.chmod(
        LOCAL_KEY,
        0o600,
    )

    print(
        "SSH key restored from:",
        source_key,
    )

    print(
        "SSH key restored to:",
        LOCAL_KEY,
    )


# Validate permissions without testing SSH.
key_mode = oct(
    LOCAL_KEY.stat().st_mode
    & 0o777
)

print("SSH key permissions:", key_mode)

if key_mode != "0o600":
    raise RuntimeError(
        "SSH key permissions must be 0o600."
    )

print("CELL 1A: SUCCESS")

SSH key restored from: /content/drive/MyDrive/hadoop-hdfs.pem
SSH key restored to: /content/hadoop-hdfs.pem
SSH key permissions: 0o600
CELL 1A: SUCCESS


In [6]:
# ============================================================
# CELL 1B-FIX: VALIDATE EXISTING HDFS DATA
#
# HDFS is already running.
# This fixes only the incompatible file-count command.
# No data is modified.
# ============================================================

hdfs_validation_command = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME="/usr/local/hadoop"
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

echo "============================================================"
echo "HDFS PROCESS CHECK"
echo "============================================================"

jps

for daemon in NameNode DataNode SecondaryNameNode; do
    if ! jps | awk '{print $2}' | grep -qx "${daemon}"; then
        echo "ERROR: ${daemon} is not running."
        exit 20
    fi
done

echo
echo "============================================================"
echo "HDFS SAFE MODE"
echo "============================================================"

SAFE_MODE_STATUS="$(hdfs dfsadmin -safemode get 2>&1)"
echo "${SAFE_MODE_STATUS}"

if ! echo "${SAFE_MODE_STATUS}" \
    | grep -qi "Safe mode is OFF"
then
    echo "ERROR: HDFS safe mode is not OFF."
    exit 21
fi

echo
echo "============================================================"
echo "RAW GHSL INVENTORY"
echo "============================================================"

# Hadoop's `hdfs dfs -find` implementation on this host does
# not support the GNU/Unix `-type f` predicate.
#
# The raw tree contains files, not directories, with names
# ending in .tif or .tiff. Count those paths explicitly.

RAW_FILE_COUNT="$(
    hdfs dfs -find /data/ghsl/raw/india \
    | grep -Eiv '/_temporary/|/_SUCCESS$' \
    | grep -Ei '\.(tif|tiff)$' \
    | wc -l \
    | tr -d ' '
)"

RAW_LOGICAL_BYTES="$(
    hdfs dfs -du -s /data/ghsl/raw/india \
    | awk '{print $1}'
)"

echo "RAW_FILE_COUNT=${RAW_FILE_COUNT}"
echo "RAW_LOGICAL_BYTES=${RAW_LOGICAL_BYTES}"

if [ "${RAW_FILE_COUNT}" -ne 325 ]; then
    echo "ERROR: Expected 325 GeoTIFF files."
    exit 22
fi

if [ "${RAW_LOGICAL_BYTES}" -ne 7038596744 ]; then
    echo "ERROR: Unexpected raw GHSL logical byte count."
    exit 23
fi

echo
echo "============================================================"
echo "METADATA CATALOG"
echo "============================================================"

if hdfs dfs \
    -test -e /data/ghsl/metadata/india/geotiff_catalog
then
    echo "METADATA_CATALOG_EXISTS=true"
else
    echo "ERROR: Existing metadata catalog is missing."
    exit 24
fi

CATALOG_PART_COUNT="$(
    hdfs dfs \
    -ls /data/ghsl/metadata/india/geotiff_catalog \
    2>/dev/null \
    | awk '$1 ~ /^-/ && $8 ~ /part-/ {count++} END {print count+0}'
)"

echo "CATALOG_PART_COUNT=${CATALOG_PART_COUNT}"

if [ "${CATALOG_PART_COUNT}" -lt 1 ]; then
    echo "ERROR: Metadata catalog has no Parquet part files."
    exit 25
fi

echo
echo "============================================================"
echo "CELL 1B-FIX RESULT"
echo "============================================================"
echo "NameNode: RUNNING"
echo "DataNode: RUNNING"
echo "SecondaryNameNode: RUNNING"
echo "Safe mode: OFF"
echo "Raw GHSL GeoTIFF files: ${RAW_FILE_COUNT}"
echo "Raw logical bytes: ${RAW_LOGICAL_BYTES}"
echo "Metadata catalog: PRESENT"
echo "Metadata catalog part files: ${CATALOG_PART_COUNT}"
echo "CELL 1B-FIX: SUCCESS"
'''

result = run_remote(
    hdfs_validation_command,
    timeout=180,
    show_output=True,
)

print()
print("Remote return code:", result.returncode)

HDFS PROCESS CHECK
1585 NameNode
2770 Jps
1944 SecondaryNameNode
1725 DataNode

HDFS SAFE MODE
Safe mode is OFF

RAW GHSL INVENTORY
RAW_FILE_COUNT=325
RAW_LOGICAL_BYTES=7038596744

METADATA CATALOG
METADATA_CATALOG_EXISTS=true
CATALOG_PART_COUNT=0
ERROR: Metadata catalog has no Parquet part files.

REMOTE STDERR:



RuntimeError: Remote command failed with return code 25.

In [7]:
# ============================================================
# CELL 2: INSPECT EXISTING GHSL METADATA CATALOG
#
# Purpose:
#   - Read the existing HDFS Parquet catalog with Spark
#   - Print its exact schema
#   - Show representative records
#   - Show distinct product/epoch combinations when available
#
# No AWS IP and no separate SSH test.
# ============================================================

import base64
import textwrap


# ------------------------------------------------------------
# Remote Spark program
# ------------------------------------------------------------

catalog_program = textwrap.dedent(
    r'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )


    spark = None

    try:
        spark = (
            SparkSession.builder
            .appName("GHSL-Catalog-Inspection")
            .getOrCreate()
        )

        spark.sparkContext.setLogLevel("WARN")

        print("=" * 80)
        print("READING EXISTING GHSL METADATA CATALOG")
        print("=" * 80)
        print("Catalog:", HDFS_CATALOG)

        catalog_df = spark.read.parquet(
            HDFS_CATALOG
        )

        catalog_df.cache()

        row_count = catalog_df.count()
        column_names = catalog_df.columns

        print()
        print("=" * 80)
        print("CATALOG SUMMARY")
        print("=" * 80)
        print("Row count:", row_count)
        print("Column count:", len(column_names))
        print("Columns:", column_names)

        if row_count != 325:
            raise RuntimeError(
                f"Expected 325 catalog records, found {row_count}."
            )

        print()
        print("=" * 80)
        print("CATALOG SCHEMA")
        print("=" * 80)

        catalog_df.printSchema()

        print()
        print("=" * 80)
        print("REPRESENTATIVE RECORDS")
        print("=" * 80)

        catalog_df.show(
            10,
            truncate=False,
            vertical=True,
        )


        # ----------------------------------------------------
        # Detect likely semantic columns from the real schema
        # ----------------------------------------------------

        lowercase_lookup = {
            column.lower(): column
            for column in column_names
        }

        product_candidates = [
            "product",
            "product_name",
            "dataset",
            "dataset_name",
            "ghsl_product",
        ]

        epoch_candidates = [
            "epoch",
            "year",
            "reference_year",
            "temporal_epoch",
        ]

        path_candidates = [
            "path",
            "source_path",
            "file_path",
            "hdfs_path",
        ]

        product_column = next(
            (
                lowercase_lookup[name]
                for name in product_candidates
                if name in lowercase_lookup
            ),
            None,
        )

        epoch_column = next(
            (
                lowercase_lookup[name]
                for name in epoch_candidates
                if name in lowercase_lookup
            ),
            None,
        )

        path_column = next(
            (
                lowercase_lookup[name]
                for name in path_candidates
                if name in lowercase_lookup
            ),
            None,
        )

        print()
        print("=" * 80)
        print("DETECTED CATALOG COLUMNS")
        print("=" * 80)
        print("Product column:", product_column)
        print("Epoch column:", epoch_column)
        print("Path column:", path_column)


        # ----------------------------------------------------
        # Product and epoch inventory
        # ----------------------------------------------------

        if product_column is not None:
            print()
            print("=" * 80)
            print("PRODUCT INVENTORY")
            print("=" * 80)

            (
                catalog_df
                .groupBy(product_column)
                .agg(
                    F.count("*").alias(
                        "file_count"
                    )
                )
                .orderBy(product_column)
                .show(
                    100,
                    truncate=False,
                )
            )

        if (
            product_column is not None
            and epoch_column is not None
        ):
            print()
            print("=" * 80)
            print("PRODUCT AND EPOCH INVENTORY")
            print("=" * 80)

            (
                catalog_df
                .groupBy(
                    product_column,
                    epoch_column,
                )
                .agg(
                    F.count("*").alias(
                        "file_count"
                    )
                )
                .orderBy(
                    product_column,
                    epoch_column,
                )
                .show(
                    500,
                    truncate=False,
                )
            )


        # ----------------------------------------------------
        # Locate the 1975 population records
        # ----------------------------------------------------

        if (
            product_column is not None
            and epoch_column is not None
        ):
            population_1975_df = (
                catalog_df
                .filter(
                    F.lower(
                        F.col(product_column)
                    ).contains("population")
                )
                .filter(
                    F.col(epoch_column).cast("int")
                    == 1975
                )
            )

            population_1975_count = (
                population_1975_df.count()
            )

            print()
            print("=" * 80)
            print("1975 POPULATION CATALOG RECORDS")
            print("=" * 80)
            print(
                "Matching records:",
                population_1975_count,
            )

            if path_column is not None:
                (
                    population_1975_df
                    .orderBy(path_column)
                    .show(
                        100,
                        truncate=False,
                        vertical=True,
                    )
                )
            else:
                population_1975_df.show(
                    100,
                    truncate=False,
                    vertical=True,
                )


        catalog_df.unpersist()

        print()
        print("=" * 80)
        print("CELL 2 REMOTE RESULT")
        print("=" * 80)
        print("Metadata catalog read: PASS")
        print("Catalog row count: 325")
        print("No raster processing performed.")
        print("CELL 2: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ------------------------------------------------------------
# Transfer the small program through remote_exec.sh
#
# Base64 avoids nested shell quoting problems.
# ------------------------------------------------------------

encoded_program = base64.b64encode(
    catalog_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell02_catalog_inspection.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "
    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "
    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "
    "spark-submit "
    "--master local[1] "
    "--driver-memory 2g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    "--conf spark.default.parallelism=2 "
    f"{remote_script}"
)


# ------------------------------------------------------------
# Execute through the helper defined in Cell 1
# ------------------------------------------------------------

result = run_remote(
    remote_command,
    timeout=900,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 2 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 2: SUCCESS")

Remote Python syntax: PASS
READING EXISTING GHSL METADATA CATALOG
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog

CATALOG SUMMARY
Row count: 325
Column count: 30
Columns: ['temporal_value', 'filename', 'path', 'driver', 'fileSize', 'width', 'height', 'pixel_count', 'numBands', 'band_data_types', 'band_nodata_values', 'srid', 'crs', 'resolution_x', 'resolution_y', 'min_x', 'min_y', 'max_x', 'max_y', 'geoTransform', 'cornerCoordinates', 'bands', 'overviews', 'overview_count', 'isTiled', 'compression', 'metadata', 'catalog_created_utc', 'product', 'temporal_type']

CATALOG SCHEMA
root
 |-- temporal_value: integer (nullable = true)
 |-- filename: string (nullable = true)
 |-- path: string (nullable = true)
 |-- driver: string (nullable = true)
 |-- fileSize: long (nullable = true)
 |-- width: integer (nullable = true)
 |-- height: integer (nullable = true)
 |-- pixel_count: long (nullable = true)
 |-- numBands: integer (nullable = true)
 |-- band_data_types: array 

In [8]:
# ============================================================
# CELL 3: IDENTIFY BENGALURU 1975 POPULATION SHARDS
#
# Primary tools:
#   - HDFS: stores the metadata catalog
#   - Spark: filters and evaluates catalog records
#   - Sedona: CRS transformation and spatial intersection
#
# This cell is read-only.
# It does not decode or process raster pixels.
# ============================================================

import base64
import textwrap


# ------------------------------------------------------------
# Remote Spark + Sedona program
# ------------------------------------------------------------

cell03_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )

    PILOT_CITY = "bengaluru"
    PILOT_EPOCH = 1975
    PILOT_PRODUCT = "population"

    # Bengaluru pilot extent in WGS 84 longitude/latitude.
    PILOT_MIN_LON = 77.35
    PILOT_MIN_LAT = 12.75
    PILOT_MAX_LON = 77.85
    PILOT_MAX_LAT = 13.25


    spark = None

    try:
        # ----------------------------------------------------
        # Create Spark + Sedona context
        # ----------------------------------------------------

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Population-Shard-Selection"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")

        print("=" * 80)
        print("CELL 3: BENGALURU SHARD SELECTION")
        print("=" * 80)
        print("City:", PILOT_CITY)
        print("Product:", PILOT_PRODUCT)
        print("Epoch:", PILOT_EPOCH)
        print(
            "WGS84 pilot extent:",
            (
                PILOT_MIN_LON,
                PILOT_MIN_LAT,
                PILOT_MAX_LON,
                PILOT_MAX_LAT,
            ),
        )


        # ----------------------------------------------------
        # Read the existing catalog from HDFS
        # ----------------------------------------------------

        catalog_df = spark.read.parquet(
            HDFS_CATALOG
        )

        population_1975_df = (
            catalog_df
            .filter(
                F.col("product")
                == PILOT_PRODUCT
            )
            .filter(
                F.col("temporal_value")
                == PILOT_EPOCH
            )
            .select(
                "filename",
                "path",
                "fileSize",
                "width",
                "height",
                "resolution_x",
                "resolution_y",
                "min_x",
                "min_y",
                "max_x",
                "max_y",
                "geoTransform",
                "crs",
            )
        )

        population_1975_count = (
            population_1975_df.count()
        )

        print()
        print("1975 population catalog records:",
              population_1975_count)

        if population_1975_count != 15:
            raise RuntimeError(
                "Expected 15 population records for 1975, "
                f"found {population_1975_count}."
            )


        # ----------------------------------------------------
        # Register the filtered catalog as a Spark SQL view
        # ----------------------------------------------------

        population_1975_df.createOrReplaceTempView(
            "population_1975_catalog"
        )


        # ----------------------------------------------------
        # Construct the Bengaluru pilot polygon
        #
        # The polygon begins in EPSG:4326 and is transformed
        # by Sedona into ESRI:54009, World Mollweide, matching
        # the GHSL raster coordinates.
        # ----------------------------------------------------

        pilot_wkt = (
            "POLYGON(("
            f"{PILOT_MIN_LON} {PILOT_MIN_LAT},"
            f"{PILOT_MAX_LON} {PILOT_MIN_LAT},"
            f"{PILOT_MAX_LON} {PILOT_MAX_LAT},"
            f"{PILOT_MIN_LON} {PILOT_MAX_LAT},"
            f"{PILOT_MIN_LON} {PILOT_MIN_LAT}"
            "))"
        )

        spark.sql(
            f"""
            CREATE OR REPLACE TEMP VIEW
                bengaluru_pilot
            AS
            SELECT
                ST_Transform(
                    ST_GeomFromWKT('{pilot_wkt}'),
                    'EPSG:4326',
                    'ESRI:54009'
                ) AS pilot_geometry
            """
        )


        # ----------------------------------------------------
        # Show transformed pilot geometry
        # ----------------------------------------------------

        print()
        print("=" * 80)
        print("BENGALURU PILOT GEOMETRY")
        print("=" * 80)

        pilot_geometry_df = spark.sql(
            """
            SELECT
                ST_AsText(
                    pilot_geometry
                ) AS pilot_mollweide_wkt
            FROM bengaluru_pilot
            """
        )

        pilot_geometry_df.show(
            truncate=False,
        )


        # ----------------------------------------------------
        # Sedona spatial intersection
        #
        # Each catalog row becomes its Mollweide raster
        # footprint using ST_PolygonFromEnvelope.
        # Sedona then intersects the footprint against the
        # transformed Bengaluru pilot polygon.
        # ----------------------------------------------------

        selected_shards_df = spark.sql(
            """
            SELECT
                c.filename,
                c.path,
                c.fileSize,
                c.width,
                c.height,
                c.resolution_x,
                c.resolution_y,
                c.min_x,
                c.min_y,
                c.max_x,
                c.max_y,
                c.geoTransform,

                ST_AsText(
                    ST_PolygonFromEnvelope(
                        c.min_x,
                        c.min_y,
                        c.max_x,
                        c.max_y
                    )
                ) AS shard_footprint_wkt

            FROM population_1975_catalog AS c

            CROSS JOIN bengaluru_pilot AS b

            WHERE ST_Intersects(
                ST_PolygonFromEnvelope(
                    c.min_x,
                    c.min_y,
                    c.max_x,
                    c.max_y
                ),
                b.pilot_geometry
            )

            ORDER BY c.filename
            """
        )


        # ----------------------------------------------------
        # Validate selected population shards
        # ----------------------------------------------------

        selected_count = selected_shards_df.count()

        print()
        print("=" * 80)
        print("SELECTED BENGALURU POPULATION SHARDS")
        print("=" * 80)
        print("Selected shard count:", selected_count)

        selected_shards_df.show(
            20,
            truncate=False,
            vertical=True,
        )

        if selected_count < 1:
            raise RuntimeError(
                "No 1975 population shard intersects the "
                "Bengaluru pilot extent."
            )

        if selected_count > 4:
            raise RuntimeError(
                "The Bengaluru pilot unexpectedly intersects "
                f"{selected_count} shards. Review the CRS "
                "transformation before raster processing."
            )


        # ----------------------------------------------------
        # Validate raster metadata for selected shards
        # ----------------------------------------------------

        invalid_metadata_count = (
            selected_shards_df
            .filter(
                (F.col("width") <= 0)
                | (F.col("height") <= 0)
                | (F.col("resolution_x") != 100.0)
                | (F.col("resolution_y") != 100.0)
                | F.col("path").isNull()
            )
            .count()
        )

        if invalid_metadata_count != 0:
            raise RuntimeError(
                "One or more selected shards have invalid "
                "raster metadata."
            )


        # ----------------------------------------------------
        # Compact result for the next notebook cell
        # ----------------------------------------------------

        print()
        print("=" * 80)
        print("SELECTED HDFS PATHS")
        print("=" * 80)

        (
            selected_shards_df
            .select(
                "path",
            )
            .show(
                20,
                truncate=False,
            )
        )


        print()
        print("=" * 80)
        print("CELL 3 REMOTE RESULT")
        print("=" * 80)
        print("Catalog source: HDFS Parquet")
        print("Catalog filtering: Spark")
        print("CRS transformation: Sedona")
        print("Spatial intersection: Sedona")
        print(
            "Selected 1975 population shards:",
            selected_count,
        )
        print("Raster pixel processing: NOT STARTED")
        print("CELL 3: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ------------------------------------------------------------
# Transfer the small remote program safely
# ------------------------------------------------------------

encoded_program = base64.b64encode(
    cell03_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell03_bengaluru_shards.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 2g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    "--conf spark.default.parallelism=2 "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ------------------------------------------------------------
# Execute through remote_exec.sh
# ------------------------------------------------------------

result = run_remote(
    remote_command,
    timeout=1200,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 3 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 3: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 3: BENGALURU SHARD SELECTION
City: bengaluru
Product: population
Epoch: 1975
WGS84 pilot extent: (77.35, 12.75, 77.85, 13.25)

1975 population catalog records: 15

BENGALURU PILOT GEOMETRY
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|pilot_mollweide_wkt                                                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|POLYGON ((7633672.199624

Notebook bootstrap                         COMPLETE
SSH key restoration                       COMPLETE
HDFS restart and validation               COMPLETE
Spark metadata catalog inspection         COMPLETE
Bengaluru boundary transformation         COMPLETE
1975 population shard selection           COMPLETE
Raster pixel processing                   NOT STARTED

In [9]:
# ============================================================
# CELL 4: READ AND CLIP BENGALURU 1975 POPULATION RASTER
#
# Production tools:
#   - HDFS: source GeoTIFF
#   - Spark binaryFile: distributed file ingestion
#   - Sedona RS_FromGeoTiff: raster decoding
#   - Sedona RS_Clip: spatial clipping
#   - Sedona raster metadata functions: validation
#
# Python only submits the remote Spark/Sedona program.
#
# This cell does NOT:
#   - use NumPy
#   - use Rasterio
#   - use a Python raster UDF
#   - aggregate population yet
#   - write output to HDFS
# ============================================================

import base64
import textwrap


cell04_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    SOURCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/raw/india/population/epoch=1975/"
        "ghsl_india_population_1975_100m_"
        "r2023a-0000016384-0000008192.tif"
    )

    PILOT_MIN_LON = 77.35
    PILOT_MIN_LAT = 12.75
    PILOT_MAX_LON = 77.85
    PILOT_MAX_LAT = 13.25

    EXPECTED_SOURCE_WIDTH = 8192
    EXPECTED_SOURCE_HEIGHT = 8192
    EXPECTED_SOURCE_BANDS = 1

    INVALID_SENTINEL = -200.0


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Population-Clip-Validation"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 4: BENGALURU POPULATION RASTER CLIP")
        print("=" * 80)
        print("Source:", SOURCE_PATH)
        print(
            "Pilot extent:",
            (
                PILOT_MIN_LON,
                PILOT_MIN_LAT,
                PILOT_MAX_LON,
                PILOT_MAX_LAT,
            ),
        )


        # ====================================================
        # VERIFY REQUIRED SEDONA FUNCTIONS
        # ====================================================

        required_functions = [
            "rs_fromgeotiff",
            "rs_clip",
            "rs_width",
            "rs_height",
            "rs_numbands",
            "rs_scalex",
            "rs_scaley",
            "rs_upperleftx",
            "rs_upperlefty",
        ]

        available_functions = {
            row["function"]
            for row in spark.sql(
                "SHOW FUNCTIONS"
            ).collect()
        }

        missing_functions = [
            function_name
            for function_name in required_functions
            if function_name not in available_functions
        ]

        if missing_functions:
            raise RuntimeError(
                "Required Sedona raster functions are missing: "
                + ", ".join(missing_functions)
            )

        print("Required Sedona raster functions: PRESENT")


        # ====================================================
        # CREATE THE BENGALURU PILOT GEOMETRY
        #
        # Input coordinates are WGS 84 longitude/latitude.
        # Sedona transforms the geometry to World Mollweide,
        # matching the GHSL raster.
        # ====================================================

        pilot_wkt = (
            "POLYGON(("
            f"{PILOT_MIN_LON} {PILOT_MIN_LAT},"
            f"{PILOT_MAX_LON} {PILOT_MIN_LAT},"
            f"{PILOT_MAX_LON} {PILOT_MAX_LAT},"
            f"{PILOT_MIN_LON} {PILOT_MAX_LAT},"
            f"{PILOT_MIN_LON} {PILOT_MIN_LAT}"
            "))"
        )

        spark.sql(
            f"""
            CREATE OR REPLACE TEMP VIEW
                bengaluru_pilot
            AS
            SELECT
                ST_Transform(
                    ST_GeomFromWKT(
                        '{pilot_wkt}'
                    ),
                    'EPSG:4326',
                    'ESRI:54009'
                ) AS pilot_geometry
            """
        )


        # ====================================================
        # READ THE SELECTED GEOTIFF DIRECTLY FROM HDFS
        #
        # Spark reads exactly one selected file.
        # Sedona converts its binary content to a raster.
        # ====================================================

        source_binary_df = (
            spark.read
            .format("binaryFile")
            .load(SOURCE_PATH)
            .select(
                "path",
                F.col("length").alias("source_bytes"),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        source_binary_count = source_binary_df.count()

        if source_binary_count != 1:
            raise RuntimeError(
                "Expected one source GeoTIFF, found "
                f"{source_binary_count}."
            )

        source_binary_df.createOrReplaceTempView(
            "population_source"
        )


        # ====================================================
        # VALIDATE SOURCE RASTER METADATA
        #
        # No pixel values are transferred to Python.
        # Only scalar metadata is returned.
        # ====================================================

        source_metadata_df = spark.sql(
            """
            SELECT
                path,
                source_bytes,
                RS_Width(
                    source_raster
                ) AS source_width,
                RS_Height(
                    source_raster
                ) AS source_height,
                RS_NumBands(
                    source_raster
                ) AS source_band_count,
                RS_ScaleX(
                    source_raster
                ) AS source_scale_x,
                RS_ScaleY(
                    source_raster
                ) AS source_scale_y,
                RS_UpperLeftX(
                    source_raster
                ) AS source_upper_left_x,
                RS_UpperLeftY(
                    source_raster
                ) AS source_upper_left_y
            FROM population_source
            """
        )

        source_metadata = (
            source_metadata_df.first()
        )

        print()
        print("=" * 80)
        print("SOURCE RASTER METADATA")
        print("=" * 80)

        source_metadata_df.show(
            truncate=False,
            vertical=True,
        )

        if (
            source_metadata["source_width"]
            != EXPECTED_SOURCE_WIDTH
        ):
            raise RuntimeError(
                "Unexpected source raster width."
            )

        if (
            source_metadata["source_height"]
            != EXPECTED_SOURCE_HEIGHT
        ):
            raise RuntimeError(
                "Unexpected source raster height."
            )

        if (
            source_metadata["source_band_count"]
            != EXPECTED_SOURCE_BANDS
        ):
            raise RuntimeError(
                "Unexpected source raster band count."
            )

        if (
            abs(
                float(
                    source_metadata["source_scale_x"]
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected source X resolution."
            )

        if (
            abs(
                abs(
                    float(
                        source_metadata[
                            "source_scale_y"
                        ]
                    )
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected source Y resolution."
            )


        # ====================================================
        # CLIP THE SOURCE RASTER INSIDE SEDONA/JVM
        #
        # Parameters:
        #   raster             source raster
        #   1                  band index
        #   pilot_geometry     Mollweide pilot polygon
        #   false              pixel-center selection
        #   -200.0             output NoData value
        #   true               crop output to pilot extent
        #   false              fail if geometry does not overlap
        #
        # The clipped raster remains inside Spark/Sedona.
        # ====================================================

        clipped_raster_df = spark.sql(
            f"""
            SELECT
                p.path,
                p.source_bytes,

                RS_Clip(
                    p.source_raster,
                    1,
                    b.pilot_geometry,
                    false,
                    {INVALID_SENTINEL},
                    true,
                    false
                ) AS clipped_raster

            FROM population_source AS p

            CROSS JOIN bengaluru_pilot AS b
            """
        )

        clipped_raster_df.createOrReplaceTempView(
            "bengaluru_population_clip"
        )


        # ====================================================
        # VALIDATE CLIPPED RASTER METADATA
        # ====================================================

        clipped_metadata_df = spark.sql(
            """
            SELECT
                path,
                source_bytes,

                RS_Width(
                    clipped_raster
                ) AS clipped_width,

                RS_Height(
                    clipped_raster
                ) AS clipped_height,

                RS_NumBands(
                    clipped_raster
                ) AS clipped_band_count,

                RS_ScaleX(
                    clipped_raster
                ) AS clipped_scale_x,

                RS_ScaleY(
                    clipped_raster
                ) AS clipped_scale_y,

                RS_UpperLeftX(
                    clipped_raster
                ) AS clipped_upper_left_x,

                RS_UpperLeftY(
                    clipped_raster
                ) AS clipped_upper_left_y

            FROM bengaluru_population_clip
            """
        )

        clipped_metadata = (
            clipped_metadata_df.first()
        )

        print()
        print("=" * 80)
        print("CLIPPED BENGALURU RASTER METADATA")
        print("=" * 80)

        clipped_metadata_df.show(
            truncate=False,
            vertical=True,
        )


        # ====================================================
        # CLIP VALIDATION
        # ====================================================

        clipped_width = int(
            clipped_metadata["clipped_width"]
        )

        clipped_height = int(
            clipped_metadata["clipped_height"]
        )

        clipped_band_count = int(
            clipped_metadata[
                "clipped_band_count"
            ]
        )

        clipped_pixel_count = (
            clipped_width
            * clipped_height
        )

        source_pixel_count = (
            EXPECTED_SOURCE_WIDTH
            * EXPECTED_SOURCE_HEIGHT
        )

        reduction_factor = (
            source_pixel_count
            / clipped_pixel_count
        )


        if clipped_width <= 0:
            raise RuntimeError(
                "Clipped raster width is not positive."
            )

        if clipped_height <= 0:
            raise RuntimeError(
                "Clipped raster height is not positive."
            )

        if clipped_width >= EXPECTED_SOURCE_WIDTH:
            raise RuntimeError(
                "Clipping did not reduce raster width."
            )

        if clipped_height >= EXPECTED_SOURCE_HEIGHT:
            raise RuntimeError(
                "Clipping did not reduce raster height."
            )

        if clipped_band_count != 1:
            raise RuntimeError(
                "Clipped raster does not have one band."
            )

        if (
            abs(
                float(
                    clipped_metadata[
                        "clipped_scale_x"
                    ]
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Clipped raster X resolution changed."
            )

        if (
            abs(
                abs(
                    float(
                        clipped_metadata[
                            "clipped_scale_y"
                        ]
                    )
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Clipped raster Y resolution changed."
            )


        print()
        print("=" * 80)
        print("CLIP SIZE SUMMARY")
        print("=" * 80)
        print(
            "Source dimensions:",
            EXPECTED_SOURCE_WIDTH,
            "x",
            EXPECTED_SOURCE_HEIGHT,
        )
        print(
            "Source pixel count:",
            source_pixel_count,
        )
        print(
            "Clipped dimensions:",
            clipped_width,
            "x",
            clipped_height,
        )
        print(
            "Clipped pixel count:",
            clipped_pixel_count,
        )
        print(
            "Pixel-count reduction factor:",
            round(reduction_factor, 2),
        )


        print()
        print("=" * 80)
        print("CELL 4 REMOTE RESULT")
        print("=" * 80)
        print("HDFS binary GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Sedona spatial clipping: PASS")
        print("Source dimensions: 8192 x 8192")
        print(
            "Clipped dimensions:",
            f"{clipped_width} x {clipped_height}",
        )
        print("Resolution preserved: 100 m")
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("HDFS output written: NO")
        print("CELL 4: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell04_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell04_bengaluru_clip.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# RUN THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 4 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 4: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 4: BENGALURU POPULATION RASTER CLIP
Source: hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif
Pilot extent: (77.35, 12.75, 77.85, 13.25)
Required Sedona raster functions: PRESENT

SOURCE RASTER METADATA
-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------
 path                | hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif 
 source_bytes        | 75930256                                                                                                                         
 source_width        | 8192                               

RuntimeError: Remote command failed with return code 1.

In [10]:
# ============================================================
# CELL 4B: CRS-SAFE BENGALURU RASTER CLIP
#
# Replaces the failed Cell 4.
#
# Production tools:
#   - HDFS: source GeoTIFF
#   - Spark binaryFile: file ingestion
#   - Sedona RS_FromGeoTiff: raster decoding
#   - Sedona RS_Clip: raster clipping
#
# No Python raster UDF, NumPy or Rasterio.
# ============================================================

import base64
import textwrap


cell04b_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    SOURCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/raw/india/population/epoch=1975/"
        "ghsl_india_population_1975_100m_"
        "r2023a-0000016384-0000008192.tif"
    )

    # These Mollweide coordinates were produced successfully
    # by Sedona in Cell 3 from the Bengaluru WGS84 pilot box.
    #
    # Use the complete envelope of the transformed polygon.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454

    INVALID_SENTINEL = -200.0

    EXPECTED_SOURCE_WIDTH = 8192
    EXPECTED_SOURCE_HEIGHT = 8192


    spark = None

    try:
        # ----------------------------------------------------
        # Spark + Sedona context
        # ----------------------------------------------------

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "CRS-Safe-Population-Clip"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 4B: CRS-SAFE BENGALURU RASTER CLIP")
        print("=" * 80)
        print("Source:", SOURCE_PATH)
        print(
            "Pilot Mollweide envelope:",
            (
                PILOT_MIN_X,
                PILOT_MIN_Y,
                PILOT_MAX_X,
                PILOT_MAX_Y,
            ),
        )


        # ----------------------------------------------------
        # Read one GeoTIFF directly from HDFS
        # ----------------------------------------------------

        source_df = (
            spark.read
            .format("binaryFile")
            .load(SOURCE_PATH)
            .select(
                "path",
                F.col("length").alias(
                    "source_bytes"
                ),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "population_source"
        )


        # ----------------------------------------------------
        # Create raster and pilot envelopes
        #
        # ST_PolygonFromEnvelope creates a geometry from the
        # numeric Mollweide coordinates without asking Sedona
        # to transform the geometry again.
        # ----------------------------------------------------

        geometry_df = spark.sql(
            f"""
            SELECT
                path,
                source_bytes,
                source_raster,

                RS_Envelope(
                    source_raster
                ) AS raster_envelope,

                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                ) AS pilot_envelope

            FROM population_source
            """
        )

        geometry_df.createOrReplaceTempView(
            "clip_inputs"
        )


        # ----------------------------------------------------
        # Validate the numeric geometry intersection first
        # ----------------------------------------------------

        intersection_df = spark.sql(
            """
            SELECT
                path,
                source_bytes,

                ST_AsText(
                    raster_envelope
                ) AS raster_envelope_wkt,

                ST_AsText(
                    pilot_envelope
                ) AS pilot_envelope_wkt,

                ST_Intersects(
                    raster_envelope,
                    pilot_envelope
                ) AS envelopes_intersect

            FROM clip_inputs
            """
        )

        print()
        print("=" * 80)
        print("PRE-CLIP INTERSECTION VALIDATION")
        print("=" * 80)

        intersection_df.show(
            truncate=False,
            vertical=True,
        )

        intersection_row = intersection_df.first()

        if intersection_row["envelopes_intersect"] is not True:
            raise RuntimeError(
                "The numeric Bengaluru pilot envelope does not "
                "intersect the source raster envelope."
            )


        # ----------------------------------------------------
        # Clip inside Sedona
        #
        # The pilot envelope is already expressed in the same
        # Mollweide coordinate space as the raster.
        #
        # lenient=true prevents a complete Spark-stage failure
        # if this Sedona version still rejects the clip.
        # A null result is checked explicitly afterward.
        # ----------------------------------------------------

        clipped_df = spark.sql(
            f"""
            SELECT
                path,
                source_bytes,

                RS_Clip(
                    source_raster,
                    1,
                    pilot_envelope,
                    false,
                    {INVALID_SENTINEL},
                    true,
                    true
                ) AS clipped_raster

            FROM clip_inputs

            WHERE ST_Intersects(
                raster_envelope,
                pilot_envelope
            )
            """
        )

        clipped_df.createOrReplaceTempView(
            "bengaluru_population_clip"
        )


        # ----------------------------------------------------
        # Validate clipping result
        # ----------------------------------------------------

        metadata_df = spark.sql(
            """
            SELECT
                path,
                source_bytes,

                CASE
                    WHEN clipped_raster IS NULL
                    THEN true
                    ELSE false
                END AS clip_is_null,

                RS_Width(
                    clipped_raster
                ) AS clipped_width,

                RS_Height(
                    clipped_raster
                ) AS clipped_height,

                RS_NumBands(
                    clipped_raster
                ) AS clipped_band_count,

                RS_ScaleX(
                    clipped_raster
                ) AS clipped_scale_x,

                RS_ScaleY(
                    clipped_raster
                ) AS clipped_scale_y,

                RS_UpperLeftX(
                    clipped_raster
                ) AS clipped_upper_left_x,

                RS_UpperLeftY(
                    clipped_raster
                ) AS clipped_upper_left_y

            FROM bengaluru_population_clip
            """
        )

        print()
        print("=" * 80)
        print("CLIPPED RASTER METADATA")
        print("=" * 80)

        metadata_df.show(
            truncate=False,
            vertical=True,
        )

        row = metadata_df.first()

        if row["clip_is_null"]:
            raise RuntimeError(
                "RS_Clip returned NULL despite the confirmed "
                "numeric envelope intersection. The next safe "
                "method would use a pixel-window raster crop."
            )

        clipped_width = int(
            row["clipped_width"]
        )

        clipped_height = int(
            row["clipped_height"]
        )

        if not (
            1 <= clipped_width
            < EXPECTED_SOURCE_WIDTH
        ):
            raise RuntimeError(
                "Invalid clipped raster width: "
                f"{clipped_width}"
            )

        if not (
            1 <= clipped_height
            < EXPECTED_SOURCE_HEIGHT
        ):
            raise RuntimeError(
                "Invalid clipped raster height: "
                f"{clipped_height}"
            )

        if int(row["clipped_band_count"]) != 1:
            raise RuntimeError(
                "Expected one band in clipped raster."
            )

        if (
            abs(
                float(row["clipped_scale_x"])
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Clipped X resolution changed."
            )

        if (
            abs(
                abs(
                    float(row["clipped_scale_y"])
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Clipped Y resolution changed."
            )


        source_pixel_count = (
            EXPECTED_SOURCE_WIDTH
            * EXPECTED_SOURCE_HEIGHT
        )

        clipped_pixel_count = (
            clipped_width
            * clipped_height
        )

        reduction_factor = (
            source_pixel_count
            / clipped_pixel_count
        )


        print()
        print("=" * 80)
        print("CLIP SUMMARY")
        print("=" * 80)
        print(
            "Source dimensions:",
            "8192 x 8192",
        )
        print(
            "Clipped dimensions:",
            f"{clipped_width} x {clipped_height}",
        )
        print(
            "Clipped pixel count:",
            clipped_pixel_count,
        )
        print(
            "Pixel-count reduction factor:",
            round(reduction_factor, 2),
        )


        print()
        print("=" * 80)
        print("CELL 4B REMOTE RESULT")
        print("=" * 80)
        print("HDFS GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Numeric envelope intersection: PASS")
        print("Sedona raster clipping: PASS")
        print("Resolution preserved: 100 m")
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("HDFS output written: NO")
        print("CELL 4B: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


encoded_program = base64.b64encode(
    cell04b_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell04b_crs_safe_clip.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 4B NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 4B: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 4B: CRS-SAFE BENGALURU RASTER CLIP
Source: hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif
Pilot Mollweide envelope: (7624177.350136818, 1571479.656526347, 7683017.20414696, 1632690.7573197454)

PRE-CLIP INTERSECTION VALIDATION
-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 path                | hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif                                                                    
 source_bytes        | 75930256                   

RuntimeError: Remote command failed with return code 1.

In [12]:
# ============================================================
# CELL 4C: VERIFY SEDONA TILE GEOREFERENCING
#
# Purpose:
#   - Read the selected 1975 population shard from HDFS
#   - Split it into 256 x 256 Sedona raster tiles
#   - Inspect only the first 5 tile envelopes and metadata
#
# No Python raster UDF, NumPy, Rasterio, or HDFS write.
# ============================================================

import base64
import textwrap


cell04c_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    SOURCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/raw/india/population/epoch=1975/"
        "ghsl_india_population_1975_100m_"
        "r2023a-0000016384-0000008192.tif"
    )

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    spark = None

    try:
        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-Tile-"
                "Georeferencing-Diagnostic"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")

        print("=" * 80)
        print("CELL 4C: SEDONA TILE GEOREFERENCING")
        print("=" * 80)

        source_df = (
            spark.read
            .format("binaryFile")
            .load(SOURCE_PATH)
            .select(
                "path",
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "population_source"
        )

        tiles_df = spark.sql(
            f"""
            SELECT
                path,
                posexplode(
                    RS_Tile(
                        source_raster,
                        ARRAY(1),
                        {TILE_WIDTH},
                        {TILE_HEIGHT},
                        false
                    )
                ) AS (
                    tile_index,
                    tile
                )
            FROM population_source
            """
        )

        tiles_df.createOrReplaceTempView(
            "population_tiles"
        )

        expected_tile_columns = (
            8192 // TILE_WIDTH
        )

        expected_tile_rows = (
            8192 // TILE_HEIGHT
        )

        expected_tile_count = (
            expected_tile_columns
            * expected_tile_rows
        )

        actual_tile_count = tiles_df.count()

        print("Expected tile columns:", expected_tile_columns)
        print("Expected tile rows:", expected_tile_rows)
        print("Expected tile count:", expected_tile_count)
        print("Actual tile count:", actual_tile_count)

        if actual_tile_count != expected_tile_count:
            raise RuntimeError(
                "Unexpected tile count. Expected "
                f"{expected_tile_count}, found "
                f"{actual_tile_count}."
            )

        tile_metadata_df = spark.sql(
            f"""
            SELECT
                tile_index,

                CAST(
                    FLOOR(
                        tile_index
                        / {expected_tile_columns}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index
                    % {expected_tile_columns}
                    AS INT
                ) AS tile_col,

                RS_Width(
                    tile
                ) AS tile_width,

                RS_Height(
                    tile
                ) AS tile_height,

                RS_UpperLeftX(
                    tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    tile
                ) AS tile_scale_y,

                ST_AsText(
                    RS_Envelope(tile)
                ) AS tile_envelope_wkt

            FROM population_tiles

            ORDER BY tile_index
            """
        )

        print()
        print("=" * 80)
        print("FIRST FIVE TILE RECORDS")
        print("=" * 80)

        tile_metadata_df.show(
            5,
            truncate=False,
            vertical=True,
        )

        first_tile = tile_metadata_df.first()

        if first_tile["tile_width"] != TILE_WIDTH:
            raise RuntimeError(
                "Unexpected first-tile width."
            )

        if first_tile["tile_height"] != TILE_HEIGHT:
            raise RuntimeError(
                "Unexpected first-tile height."
            )

        if (
            abs(
                float(
                    first_tile["tile_scale_x"]
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Tile X resolution is not 100 m."
            )

        if (
            abs(
                abs(
                    float(
                        first_tile["tile_scale_y"]
                    )
                )
                - 100.0
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Tile Y resolution is not 100 m."
            )

        print()
        print("=" * 80)
        print("CELL 4C REMOTE RESULT")
        print("=" * 80)
        print("HDFS GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Sedona 256 x 256 tiling: PASS")
        print("Tile count: 1024")
        print("Tile georeferencing: PRESERVED")
        print("Python raster UDF used: NO")
        print("HDFS output written: NO")
        print("CELL 4C: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


encoded_program = base64.b64encode(
    cell04c_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell04c_tile_diagnostic.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 4C NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 4C: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 4C: SEDONA TILE GEOREFERENCING
Expected tile columns: 32
Expected tile rows: 32
Expected tile count: 1024
Actual tile count: 1024

FIRST FIVE TILE RECORDS
-RECORD 0------------------------------------------------------------------------------------------------------------
 tile_index        | 0                                                                                               
 tile_row          | 0                                                                                               
 tile_col          | 0                                                                                               
 tile_width        | 256                                                                                             
 tile_height       | 256                              

In [13]:
# ============================================================
# CELL 5: SELECT BENGALURU-INTERSECTING POPULATION TILES
#
# Tools:
#   - HDFS: source GeoTIFF
#   - Spark: ingestion, tiling rows, filtering, validation
#   - Sedona: decoding, raster tiling, tile envelopes,
#             spatial intersection
#
# No NumPy, Rasterio, or Python raster UDF.
# No HDFS output is written yet.
# ============================================================

import base64
import textwrap


cell05_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    SOURCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/raw/india/population/epoch=1975/"
        "ghsl_india_population_1975_100m_"
        "r2023a-0000016384-0000008192.tif"
    )

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    SOURCE_WIDTH = 8192
    SOURCE_HEIGHT = 8192

    TILE_COLUMNS = SOURCE_WIDTH // TILE_WIDTH
    TILE_ROWS = SOURCE_HEIGHT // TILE_HEIGHT

    # Mollweide pilot envelope verified in Cells 3 and 4B.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Population-Tile-Selection"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 5: BENGALURU TILE SELECTION")
        print("=" * 80)
        print("Source:", SOURCE_PATH)
        print(
            "Source tile layout:",
            f"{TILE_COLUMNS} x {TILE_ROWS}",
        )
        print(
            "Pilot envelope:",
            (
                PILOT_MIN_X,
                PILOT_MIN_Y,
                PILOT_MAX_X,
                PILOT_MAX_Y,
            ),
        )


        # ====================================================
        # READ AND DECODE THE SOURCE RASTER
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(SOURCE_PATH)
            .select(
                "path",
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "population_source"
        )


        # ====================================================
        # SPLIT INTO 256 x 256 GEOREFERENCED SEDONA TILES
        #
        # ARRAY(1) explicitly selects the first raster band.
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path,

                tile_index,

                CAST(
                    FLOOR(
                        tile_index / {TILE_COLUMNS}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {TILE_COLUMNS}
                    AS INT
                ) AS tile_col,

                tile

            FROM population_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded AS tile_index, tile
            """
        )

        all_tile_count = tiles_df.count()

        if all_tile_count != 1024:
            raise RuntimeError(
                "Expected 1024 source tiles, found "
                f"{all_tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "population_tiles"
        )


        # ====================================================
        # CREATE PILOT ENVELOPE IN MOLLWEIDE COORDINATES
        # ====================================================

        pilot_df = spark.sql(
            f"""
            SELECT
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                ) AS pilot_envelope
            """
        )

        pilot_df.createOrReplaceTempView(
            "bengaluru_pilot"
        )


        # ====================================================
        # SELECT ONLY SPATIALLY INTERSECTING TILES
        #
        # Each tile keeps its georeferenced raster envelope.
        # ====================================================

        selected_tiles_df = spark.sql(
            """
            SELECT
                t.path,
                t.tile_index,
                t.tile_row,
                t.tile_col,
                t.tile,

                RS_Width(
                    t.tile
                ) AS tile_width,

                RS_Height(
                    t.tile
                ) AS tile_height,

                RS_UpperLeftX(
                    t.tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    t.tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    t.tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    t.tile
                ) AS tile_scale_y,

                RS_Envelope(
                    t.tile
                ) AS tile_envelope

            FROM population_tiles AS t

            CROSS JOIN bengaluru_pilot AS b

            WHERE ST_Intersects(
                RS_Envelope(t.tile),
                b.pilot_envelope
            )
            """
        )


        # ====================================================
        # VALIDATE SELECTION
        # ====================================================

        selected_tile_count = (
            selected_tiles_df.count()
        )

        print()
        print("=" * 80)
        print("BENGALURU-INTERSECTING TILES")
        print("=" * 80)
        print("Source tiles:", all_tile_count)
        print(
            "Selected tiles:",
            selected_tile_count,
        )


        if selected_tile_count < 6:
            raise RuntimeError(
                "Too few Bengaluru tiles were selected: "
                f"{selected_tile_count}."
            )

        if selected_tile_count > 20:
            raise RuntimeError(
                "Too many Bengaluru tiles were selected: "
                f"{selected_tile_count}."
            )


        invalid_tile_count = (
            selected_tiles_df
            .filter(
                (F.col("tile_width") != TILE_WIDTH)
                | (F.col("tile_height") != TILE_HEIGHT)
                | (
                    F.abs(
                        F.col("tile_scale_x")
                        - F.lit(100.0)
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.abs(
                            F.col("tile_scale_y")
                        )
                        - F.lit(100.0)
                    )
                    > F.lit(1e-9)
                )
            )
            .count()
        )

        if invalid_tile_count != 0:
            raise RuntimeError(
                "Selected tiles failed metadata validation."
            )


        selected_summary_df = (
            selected_tiles_df
            .agg(
                F.min("tile_row").alias(
                    "minimum_tile_row"
                ),
                F.max("tile_row").alias(
                    "maximum_tile_row"
                ),
                F.min("tile_col").alias(
                    "minimum_tile_col"
                ),
                F.max("tile_col").alias(
                    "maximum_tile_col"
                ),
                F.countDistinct(
                    "tile_row"
                ).alias(
                    "selected_tile_rows"
                ),
                F.countDistinct(
                    "tile_col"
                ).alias(
                    "selected_tile_columns"
                ),
            )
        )

        summary = selected_summary_df.first()

        print()
        print("=" * 80)
        print("SELECTED TILE RANGE")
        print("=" * 80)
        selected_summary_df.show(
            truncate=False,
            vertical=True,
        )


        print()
        print("=" * 80)
        print("SELECTED TILE METADATA")
        print("=" * 80)

        (
            selected_tiles_df
            .select(
                "tile_index",
                "tile_row",
                "tile_col",
                "tile_width",
                "tile_height",
                "tile_upper_left_x",
                "tile_upper_left_y",
                F.expr(
                    "ST_AsText(tile_envelope)"
                ).alias(
                    "tile_envelope_wkt"
                ),
            )
            .orderBy(
                "tile_row",
                "tile_col",
            )
            .show(
                50,
                truncate=False,
                vertical=True,
            )
        )


        selected_pixel_count = (
            selected_tile_count
            * TILE_WIDTH
            * TILE_HEIGHT
        )

        source_pixel_count = (
            SOURCE_WIDTH
            * SOURCE_HEIGHT
        )

        reduction_factor = (
            source_pixel_count
            / selected_pixel_count
        )


        print()
        print("=" * 80)
        print("WORKING-SET REDUCTION")
        print("=" * 80)
        print(
            "Source pixel count:",
            source_pixel_count,
        )
        print(
            "Selected tile pixel capacity:",
            selected_pixel_count,
        )
        print(
            "Reduction factor:",
            round(reduction_factor, 2),
        )


        print()
        print("=" * 80)
        print("CELL 5 REMOTE RESULT")
        print("=" * 80)
        print("HDFS GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Sedona 256 x 256 tiling: PASS")
        print("Sedona spatial tile selection: PASS")
        print(
            "Selected tile rows:",
            (
                summary["minimum_tile_row"],
                summary["maximum_tile_row"],
            ),
        )
        print(
            "Selected tile columns:",
            (
                summary["minimum_tile_col"],
                summary["maximum_tile_col"],
            ),
        )
        print(
            "Selected tile count:",
            selected_tile_count,
        )
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("HDFS output written: NO")
        print("CELL 5: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

encoded_program = base64.b64encode(
    cell05_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell05_bengaluru_tiles.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 5 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 5: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 5: BENGALURU TILE SELECTION
Source: hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif
Source tile layout: 32 x 32
Pilot envelope: (7624177.350136818, 1571479.656526347, 7683017.20414696, 1632690.7573197454)

BENGALURU-INTERSECTING TILES
Source tiles: 1024
Selected tiles: 12

SELECTED TILE RANGE
-RECORD 0--------------------
 minimum_tile_row      | 28  
 maximum_tile_row      | 31  
 minimum_tile_col      | 13  
 maximum_tile_col      | 15  
 selected_tile_rows    | 4   
 selected_tile_columns | 3   


SELECTED TILE METADATA
-RECORD 0------------------------------------------------------------------------------------------------------------
 tile_index        | 909                                               

In [14]:
# ============================================================
# CELL 6: WRITE BENGALURU 1975 SELECTED-TILE CHECKPOINT
#
# Tools:
#   - HDFS: source and checkpoint storage
#   - Spark: ingestion, filtering, Parquet write/read
#   - Sedona: raster decode, tiling, envelopes, GeoTIFF encode
#
# Output:
#   /data/ghsl/curated/pilots/bengaluru/
#   product=population/epoch=1975/selected_tiles
#
# Python only submits the Spark/Sedona program.
# No NumPy, Rasterio, or Python raster UDF.
# ============================================================

import base64
import textwrap


cell06_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    SOURCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/raw/india/population/epoch=1975/"
        "ghsl_india_population_1975_100m_"
        "r2023a-0000016384-0000008192.tif"
    )

    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=1975/"
        "selected_tiles"
    )

    CITY = "bengaluru"
    PRODUCT = "population"
    EPOCH = 1975

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    SOURCE_WIDTH = 8192
    SOURCE_HEIGHT = 8192
    TILE_COLUMNS = 32

    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454

    EXPECTED_SELECTED_TILES = 12


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Selected-Tile-Checkpoint"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 6: WRITE SELECTED-TILE CHECKPOINT")
        print("=" * 80)
        print("Source:", SOURCE_PATH)
        print("Output:", OUTPUT_PATH)


        # ====================================================
        # READ AND DECODE THE SOURCE GEOTIFF
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(SOURCE_PATH)
            .select(
                "path",
                F.col("length").alias(
                    "source_file_bytes"
                ),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "population_source"
        )


        # ====================================================
        # SPLIT THE RASTER INTO GEOREFERENCED TILES
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path AS source_path,
                source_file_bytes,

                tile_index,

                CAST(
                    FLOOR(
                        tile_index / {TILE_COLUMNS}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {TILE_COLUMNS}
                    AS INT
                ) AS tile_col,

                tile

            FROM population_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded AS tile_index, tile
            """
        )

        tiles_df.createOrReplaceTempView(
            "population_tiles"
        )


        # ====================================================
        # DEFINE BENGALURU PILOT ENVELOPE
        # ====================================================

        pilot_df = spark.sql(
            f"""
            SELECT
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                ) AS pilot_envelope
            """
        )

        pilot_df.createOrReplaceTempView(
            "bengaluru_pilot"
        )


        # ====================================================
        # SELECT 12 INTERSECTING TILES
        #
        # RS_AsGeoTiff converts each Sedona raster tile to
        # portable binary before Parquet persistence.
        # ====================================================

        checkpoint_df = spark.sql(
            f"""
            SELECT
                '{CITY}' AS city,
                '{PRODUCT}' AS product,
                CAST({EPOCH} AS INT) AS epoch,

                t.source_path,
                t.source_file_bytes,

                t.tile_index,
                t.tile_row,
                t.tile_col,

                RS_Width(
                    t.tile
                ) AS tile_width,

                RS_Height(
                    t.tile
                ) AS tile_height,

                RS_UpperLeftX(
                    t.tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    t.tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    t.tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    t.tile
                ) AS tile_scale_y,

                ST_AsText(
                    RS_Envelope(t.tile)
                ) AS tile_envelope_wkt,

                RS_AsGeoTiff(
                    t.tile,
                    'LZW',
                    1.0
                ) AS tile_geotiff

            FROM population_tiles AS t

            CROSS JOIN bengaluru_pilot AS b

            WHERE ST_Intersects(
                RS_Envelope(t.tile),
                b.pilot_envelope
            )
            """
        )


        # ====================================================
        # PRE-WRITE VALIDATION
        # ====================================================

        prewrite_count = checkpoint_df.count()

        print()
        print("Pre-write selected tile count:", prewrite_count)

        if prewrite_count != EXPECTED_SELECTED_TILES:
            raise RuntimeError(
                "Expected 12 selected tiles before writing, "
                f"found {prewrite_count}."
            )


        invalid_prewrite_count = (
            checkpoint_df
            .filter(
                F.col("tile_geotiff").isNull()
                | (
                    F.length("tile_geotiff")
                    <= F.lit(0)
                )
                | (
                    F.col("tile_width")
                    != F.lit(TILE_WIDTH)
                )
                | (
                    F.col("tile_height")
                    != F.lit(TILE_HEIGHT)
                )
            )
            .count()
        )

        if invalid_prewrite_count != 0:
            raise RuntimeError(
                "One or more selected tiles failed "
                "pre-write validation."
            )


        # ====================================================
        # WRITE CHECKPOINT TO HDFS PARQUET
        #
        # Overwrite is intentional because this path represents
        # a reproducible product/city/epoch checkpoint.
        # ====================================================

        (
            checkpoint_df
            .orderBy(
                "tile_row",
                "tile_col",
            )
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )

        print("HDFS Parquet checkpoint write: COMPLETE")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        readback_count = readback_df.count()

        print("Read-back tile count:", readback_count)

        if readback_count != EXPECTED_SELECTED_TILES:
            raise RuntimeError(
                "Expected 12 checkpoint rows after read-back, "
                f"found {readback_count}."
            )


        duplicate_count = (
            readback_df
            .groupBy(
                "tile_index",
                "tile_row",
                "tile_col",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "Duplicate selected tiles found after read-back."
            )


        invalid_binary_count = (
            readback_df
            .filter(
                F.col("tile_geotiff").isNull()
                | (
                    F.length("tile_geotiff")
                    <= F.lit(0)
                )
            )
            .count()
        )

        if invalid_binary_count != 0:
            raise RuntimeError(
                "Invalid GeoTIFF binary found after read-back."
            )


        # ====================================================
        # DECODE THE CHECKPOINTED TILE BYTES WITH SEDONA
        #
        # This verifies that the persisted binary can be used
        # independently by the next Spark/Sedona job.
        # ====================================================

        decoded_readback_df = (
            readback_df
            .withColumn(
                "decoded_tile",
                F.expr(
                    "RS_FromGeoTiff(tile_geotiff)"
                ),
            )
            .select(
                "tile_index",
                "tile_row",
                "tile_col",
                F.expr(
                    "RS_Width(decoded_tile)"
                ).alias(
                    "decoded_width"
                ),
                F.expr(
                    "RS_Height(decoded_tile)"
                ).alias(
                    "decoded_height"
                ),
                F.expr(
                    "RS_UpperLeftX(decoded_tile)"
                ).alias(
                    "decoded_upper_left_x"
                ),
                F.expr(
                    "RS_UpperLeftY(decoded_tile)"
                ).alias(
                    "decoded_upper_left_y"
                ),
                F.length(
                    "tile_geotiff"
                ).alias(
                    "tile_geotiff_bytes"
                ),
            )
        )


        invalid_decode_count = (
            decoded_readback_df
            .filter(
                (F.col("decoded_width") != TILE_WIDTH)
                | (
                    F.col("decoded_height")
                    != TILE_HEIGHT
                )
            )
            .count()
        )

        if invalid_decode_count != 0:
            raise RuntimeError(
                "One or more checkpointed tiles could not be "
                "decoded correctly by Sedona."
            )


        # ====================================================
        # DISPLAY COMPACT CHECKPOINT INVENTORY
        # ====================================================

        print()
        print("=" * 80)
        print("CHECKPOINT TILE INVENTORY")
        print("=" * 80)

        (
            decoded_readback_df
            .orderBy(
                "tile_row",
                "tile_col",
            )
            .show(
                20,
                truncate=False,
            )
        )


        summary = (
            readback_df
            .agg(
                F.count("*").alias(
                    "tile_count"
                ),
                F.sum(
                    F.length("tile_geotiff")
                ).alias(
                    "checkpoint_binary_bytes"
                ),
                F.min("tile_row").alias(
                    "minimum_tile_row"
                ),
                F.max("tile_row").alias(
                    "maximum_tile_row"
                ),
                F.min("tile_col").alias(
                    "minimum_tile_col"
                ),
                F.max("tile_col").alias(
                    "maximum_tile_col"
                ),
            )
            .first()
        )


        print()
        print("=" * 80)
        print("CELL 6 REMOTE RESULT")
        print("=" * 80)
        print("HDFS source read: PASS")
        print("Spark/Sedona tile selection: PASS")
        print("Sedona GeoTIFF encoding: PASS")
        print("HDFS Parquet checkpoint write: PASS")
        print("HDFS Parquet read-back: PASS")
        print("Sedona checkpoint decode: PASS")
        print("Checkpoint rows:", summary["tile_count"])
        print(
            "Checkpoint binary bytes:",
            summary["checkpoint_binary_bytes"],
        )
        print(
            "Tile rows:",
            (
                summary["minimum_tile_row"],
                summary["maximum_tile_row"],
            ),
        )
        print(
            "Tile columns:",
            (
                summary["minimum_tile_col"],
                summary["maximum_tile_col"],
            ),
        )
        print("Output:", OUTPUT_PATH)
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 6: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER AND EXECUTE THROUGH remote_exec.sh
# ============================================================

encoded_program = base64.b64encode(
    cell06_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell06_tile_checkpoint.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=1 "
    "--conf spark.default.parallelism=1 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 6 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 6: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 6: WRITE SELECTED-TILE CHECKPOINT
Source: hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/selected_tiles

Pre-write selected tile count: 12
Traceback (most recent call last):
  File "/home/ubuntu/ghsl_notebook04_cell06_tile_checkpoint.py", line 265, in <module>
    .count()
     ^^^^^^^
  File "/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/sql/dataframe.py", line 1239, in count
  File "/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1322, in __call__
  File "/home/ubuntu/ghsl-sedona-env/l

RuntimeError: Remote command failed with return code 1.

In [15]:
# ============================================================
# CELL 7: BENGALURU 1975 POPULATION, 100 M TO 1 KM
#
# This replaces the failed raster-checkpoint approach.
#
# Production tools:
#   - HDFS: source GeoTIFF and output Parquet
#   - Spark binaryFile: source ingestion
#   - Apache Sedona:
#       RS_FromGeoTiff
#       RS_Tile
#       RS_Envelope
#       RS_BandAsArray
#   - Spark SQL:
#       pixel expansion
#       finite/valid filtering
#       pilot-boundary filtering
#       100 m to 1 km aggregation
#       validation
#       Parquet write and read-back
#
# Python is orchestration only.
#
# This cell does NOT use:
#   - Python raster UDF
#   - NumPy
#   - Rasterio
#   - RS_AsGeoTiff
#   - RS_Clip
# ============================================================

import base64
import textwrap


cell07_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    SOURCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/raw/india/population/epoch=1975/"
        "ghsl_india_population_1975_100m_"
        "r2023a-0000016384-0000008192.tif"
    )

    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=1975/"
        "grid_1km"
    )

    CITY = "bengaluru"
    PRODUCT = "population"
    EPOCH = 1975

    SOURCE_WIDTH = 8192
    SOURCE_HEIGHT = 8192

    SOURCE_RESOLUTION = 100.0
    TARGET_RESOLUTION = 1000.0

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    TILE_COLUMNS = (
        SOURCE_WIDTH // TILE_WIDTH
    )

    EXPECTED_SOURCE_TILES = 1024
    EXPECTED_SELECTED_TILES = 12

    PIXELS_PER_1KM_AXIS = int(
        TARGET_RESOLUTION
        / SOURCE_RESOLUTION
    )

    EXPECTED_PIXELS_PER_FULL_1KM_CELL = (
        PIXELS_PER_1KM_AXIS
        * PIXELS_PER_1KM_AXIS
    )

    INVALID_SENTINEL = -200.0

    # Bengaluru pilot envelope in World Mollweide coordinates.
    #
    # These values were established and spatially verified in
    # Cells 3, 4B and 5.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Population-100m-to-1km"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 7: BENGALURU POPULATION 100 M TO 1 KM")
        print("=" * 80)
        print("Source:", SOURCE_PATH)
        print("Output:", OUTPUT_PATH)
        print("City:", CITY)
        print("Product:", PRODUCT)
        print("Epoch:", EPOCH)
        print(
            "Input resolution:",
            SOURCE_RESOLUTION,
            "m",
        )
        print(
            "Output resolution:",
            TARGET_RESOLUTION,
            "m",
        )
        print(
            "Expected source pixels per complete 1 km cell:",
            EXPECTED_PIXELS_PER_FULL_1KM_CELL,
        )


        # ====================================================
        # VERIFY REQUIRED SEDONA FUNCTIONS
        # ====================================================

        required_functions = [
            "rs_fromgeotiff",
            "rs_tile",
            "rs_envelope",
            "rs_bandasarray",
            "rs_width",
            "rs_height",
            "rs_upperleftx",
            "rs_upperlefty",
            "rs_scalex",
            "rs_scaley",
        ]

        available_functions = {
            row["function"].lower()
            for row in spark.sql(
                "SHOW FUNCTIONS"
            ).collect()
        }

        missing_functions = [
            function_name
            for function_name in required_functions
            if function_name not in available_functions
        ]

        if missing_functions:
            raise RuntimeError(
                "Required Sedona functions are missing: "
                + ", ".join(missing_functions)
            )

        print("Required Sedona functions: PRESENT")


        # ====================================================
        # READ EXACTLY ONE SOURCE GEOTIFF FROM HDFS
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(SOURCE_PATH)
            .select(
                "path",
                F.col("length").alias(
                    "source_file_bytes"
                ),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        source_count = source_df.count()

        if source_count != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF, "
                f"found {source_count}."
            )

        source_df.createOrReplaceTempView(
            "population_source"
        )


        # ====================================================
        # VALIDATE SOURCE METADATA
        # ====================================================

        source_metadata_df = spark.sql(
            """
            SELECT
                path,
                source_file_bytes,

                RS_Width(
                    source_raster
                ) AS source_width,

                RS_Height(
                    source_raster
                ) AS source_height,

                RS_ScaleX(
                    source_raster
                ) AS source_scale_x,

                RS_ScaleY(
                    source_raster
                ) AS source_scale_y,

                RS_UpperLeftX(
                    source_raster
                ) AS source_upper_left_x,

                RS_UpperLeftY(
                    source_raster
                ) AS source_upper_left_y

            FROM population_source
            """
        )

        source_metadata = (
            source_metadata_df.first()
        )

        if (
            int(source_metadata["source_width"])
            != SOURCE_WIDTH
        ):
            raise RuntimeError(
                "Unexpected source width."
            )

        if (
            int(source_metadata["source_height"])
            != SOURCE_HEIGHT
        ):
            raise RuntimeError(
                "Unexpected source height."
            )

        if (
            abs(
                float(
                    source_metadata[
                        "source_scale_x"
                    ]
                )
                - SOURCE_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected source X resolution."
            )

        if (
            abs(
                abs(
                    float(
                        source_metadata[
                            "source_scale_y"
                        ]
                    )
                )
                - SOURCE_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected source Y resolution."
            )

        print("Source raster metadata: PASS")


        # ====================================================
        # SPLIT SOURCE INTO 256 x 256 SEDONA TILES
        #
        # ARRAY(1) explicitly selects population band 1.
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path AS source_path,
                source_file_bytes,

                tile_index,

                CAST(
                    FLOOR(
                        tile_index / {TILE_COLUMNS}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {TILE_COLUMNS}
                    AS INT
                ) AS tile_col,

                tile

            FROM population_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded
            AS tile_index, tile
            """
        )

        all_tile_count = tiles_df.count()

        if (
            all_tile_count
            != EXPECTED_SOURCE_TILES
        ):
            raise RuntimeError(
                "Expected "
                f"{EXPECTED_SOURCE_TILES} source tiles, "
                f"found {all_tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "population_tiles"
        )

        print(
            "Sedona source tile count:",
            all_tile_count,
        )


        # ====================================================
        # CREATE BENGALURU PILOT ENVELOPE
        # ====================================================

        pilot_df = spark.sql(
            f"""
            SELECT
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                ) AS pilot_envelope
            """
        )

        pilot_df.createOrReplaceTempView(
            "bengaluru_pilot"
        )


        # ====================================================
        # SELECT THE 12 INTERSECTING TILES
        #
        # Tile metadata is retained because it is needed to
        # reconstruct every 100 m pixel's Mollweide position.
        # ====================================================

        selected_tiles_df = spark.sql(
            """
            SELECT
                t.source_path,
                t.source_file_bytes,

                t.tile_index,
                t.tile_row,
                t.tile_col,

                RS_Width(
                    t.tile
                ) AS tile_width,

                RS_Height(
                    t.tile
                ) AS tile_height,

                RS_UpperLeftX(
                    t.tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    t.tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    t.tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    t.tile
                ) AS tile_scale_y,

                t.tile

            FROM population_tiles AS t

            CROSS JOIN bengaluru_pilot AS b

            WHERE ST_Intersects(
                RS_Envelope(t.tile),
                b.pilot_envelope
            )
            """
        )

        selected_tile_count = (
            selected_tiles_df.count()
        )

        if (
            selected_tile_count
            != EXPECTED_SELECTED_TILES
        ):
            raise RuntimeError(
                "Expected "
                f"{EXPECTED_SELECTED_TILES} Bengaluru tiles, "
                f"found {selected_tile_count}."
            )

        print(
            "Selected Bengaluru tiles:",
            selected_tile_count,
        )

        selected_tiles_df.createOrReplaceTempView(
            "selected_population_tiles"
        )


        # ====================================================
        # EXTRACT BAND VALUES INSIDE SEDONA
        #
        # RS_BandAsArray returns ARRAY<DOUBLE>.
        #
        # Spark posexplode creates:
        #   pixel_index     zero-based index inside the tile
        #   population_value
        #
        # No raster object crosses into a Python UDF.
        # ====================================================

        pixels_df = spark.sql(
            """
            SELECT
                source_path,
                source_file_bytes,

                tile_index,
                tile_row,
                tile_col,

                tile_width,
                tile_height,

                tile_upper_left_x,
                tile_upper_left_y,
                tile_scale_x,
                tile_scale_y,

                pixel_index,

                CAST(
                    FLOOR(
                        pixel_index / tile_width
                    )
                    AS INT
                ) AS local_pixel_row,

                CAST(
                    pixel_index % tile_width
                    AS INT
                ) AS local_pixel_col,

                CAST(
                    population_value
                    AS DOUBLE
                ) AS population_value

            FROM selected_population_tiles

            LATERAL VIEW posexplode(
                RS_BandAsArray(
                    tile,
                    1
                )
            ) exploded
            AS pixel_index, population_value
            """
        )

        pixels_df.createOrReplaceTempView(
            "selected_population_pixels"
        )


        # ====================================================
        # RECONSTRUCT 100 M PIXEL CENTRES
        #
        # X coordinate:
        #   upper-left X + (column + 0.5) * positive X scale
        #
        # Y coordinate:
        #   upper-left Y + (row + 0.5) * negative Y scale
        #
        # Global source row/column are retained for lineage.
        # ====================================================

        positioned_pixels_df = spark.sql(
            f"""
            SELECT
                source_path,

                tile_index,
                tile_row,
                tile_col,

                pixel_index,
                local_pixel_row,
                local_pixel_col,

                CAST(
                    tile_col * {TILE_WIDTH}
                    + local_pixel_col
                    AS INT
                ) AS global_100m_col,

                CAST(
                    tile_row * {TILE_HEIGHT}
                    + local_pixel_row
                    AS INT
                ) AS global_100m_row,

                (
                    tile_upper_left_x
                    + (
                        local_pixel_col + 0.5
                    ) * tile_scale_x
                ) AS pixel_center_x,

                (
                    tile_upper_left_y
                    + (
                        local_pixel_row + 0.5
                    ) * tile_scale_y
                ) AS pixel_center_y,

                population_value

            FROM selected_population_pixels
            """
        )

        positioned_pixels_df.createOrReplaceTempView(
            "positioned_population_pixels"
        )


        # ====================================================
        # CLASSIFY PILOT PIXELS AND VALID VALUES
        #
        # Pixel-center filtering ensures the output represents
        # only the requested Bengaluru pilot envelope, not the
        # complete 12-tile bounding block.
        #
        # Valid population rule:
        #   - pixel centre inside pilot envelope
        #   - value is not NULL
        #   - value is not NaN
        #   - value is finite
        #   - value is not -200
        #   - value is nonnegative
        #
        # ABS(value) <= max finite double rejects +/- Infinity.
        # ====================================================

        classified_pixels_df = spark.sql(
            f"""
            SELECT
                source_path,

                tile_index,
                tile_row,
                tile_col,

                global_100m_col,
                global_100m_row,

                pixel_center_x,
                pixel_center_y,

                population_value,

                CASE
                    WHEN
                        pixel_center_x >= {PILOT_MIN_X}
                        AND pixel_center_x < {PILOT_MAX_X}
                        AND pixel_center_y >= {PILOT_MIN_Y}
                        AND pixel_center_y < {PILOT_MAX_Y}
                    THEN true
                    ELSE false
                END AS inside_pilot,

                CASE
                    WHEN
                        population_value IS NOT NULL
                        AND NOT isnan(population_value)
                        AND ABS(population_value)
                            <= 1.7976931348623157E308
                        AND population_value
                            <> {INVALID_SENTINEL}
                        AND population_value >= 0.0
                    THEN true
                    ELSE false
                END AS is_valid_population

            FROM positioned_population_pixels
            """
        )

        classified_pixels_df.createOrReplaceTempView(
            "classified_population_pixels"
        )


        # ====================================================
        # PIXEL-LEVEL QUALITY SUMMARY
        # ====================================================

        pixel_quality_df = spark.sql(
            """
            SELECT
                COUNT(*) AS selected_tile_pixel_capacity,

                SUM(
                    CASE
                        WHEN inside_pilot
                        THEN 1
                        ELSE 0
                    END
                ) AS pilot_pixel_count,

                SUM(
                    CASE
                        WHEN
                            inside_pilot
                            AND is_valid_population
                        THEN 1
                        ELSE 0
                    END
                ) AS valid_pilot_pixel_count,

                SUM(
                    CASE
                        WHEN
                            inside_pilot
                            AND NOT is_valid_population
                        THEN 1
                        ELSE 0
                    END
                ) AS excluded_pilot_pixel_count,

                SUM(
                    CASE
                        WHEN
                            inside_pilot
                            AND is_valid_population
                            AND population_value = 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS zero_population_pixel_count,

                SUM(
                    CASE
                        WHEN
                            inside_pilot
                            AND is_valid_population
                            AND population_value > 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS positive_population_pixel_count

            FROM classified_population_pixels
            """
        )

        pixel_quality = pixel_quality_df.first()

        selected_capacity = int(
            pixel_quality[
                "selected_tile_pixel_capacity"
            ]
        )

        pilot_pixel_count = int(
            pixel_quality["pilot_pixel_count"]
        )

        valid_pilot_pixel_count = int(
            pixel_quality[
                "valid_pilot_pixel_count"
            ]
        )

        excluded_pilot_pixel_count = int(
            pixel_quality[
                "excluded_pilot_pixel_count"
            ]
        )


        expected_selected_capacity = (
            EXPECTED_SELECTED_TILES
            * TILE_WIDTH
            * TILE_HEIGHT
        )

        if (
            selected_capacity
            != expected_selected_capacity
        ):
            raise RuntimeError(
                "Unexpected selected-tile pixel capacity. "
                f"Expected {expected_selected_capacity}, "
                f"found {selected_capacity}."
            )

        if pilot_pixel_count <= 0:
            raise RuntimeError(
                "No pixel centres fall inside the pilot extent."
            )

        if (
            valid_pilot_pixel_count
            + excluded_pilot_pixel_count
            != pilot_pixel_count
        ):
            raise RuntimeError(
                "Pilot valid/excluded pixel identity failed."
            )


        print()
        print("=" * 80)
        print("PIXEL QUALITY SUMMARY")
        print("=" * 80)

        pixel_quality_df.show(
            truncate=False,
            vertical=True,
        )


        # ====================================================
        # BUILD CANONICAL 1 KM CELL IDENTIFIERS
        #
        # The World Mollweide grid is aligned to 1,000 m
        # multiples. Cell indices are derived from the pixel
        # centre coordinates rather than tile boundaries.
        #
        # This is essential because 256 is not divisible by 10.
        # A 1 km cell may cross a 256-pixel tile boundary.
        # ====================================================

        valid_pixels_df = spark.sql(
            f"""
            SELECT
                source_path,

                global_100m_col,
                global_100m_row,

                pixel_center_x,
                pixel_center_y,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {TARGET_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_col,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {TARGET_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_row,

                population_value

            FROM classified_population_pixels

            WHERE
                inside_pilot
                AND is_valid_population
            """
        )

        valid_pixels_df.createOrReplaceTempView(
            "valid_population_pixels"
        )


        # ====================================================
        # AGGREGATE 100 M PIXELS TO 1 KM CELLS WITH SPARK SQL
        #
        # No Python-side pixel calculations occur.
        # ====================================================

        population_1km_df = spark.sql(
            f"""
            SELECT
                '{CITY}' AS city,
                '{PRODUCT}' AS product,
                CAST({EPOCH} AS INT) AS epoch,

                grid_1km_col,
                grid_1km_row,

                CONCAT(
                    'MOLL_1KM_',
                    CAST(grid_1km_col AS STRING),
                    '_',
                    CAST(grid_1km_row AS STRING)
                ) AS grid_cell_id,

                CAST(
                    grid_1km_col
                    * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_x,

                CAST(
                    (
                        grid_1km_col + 1
                    ) * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_x,

                CAST(
                    grid_1km_row
                    * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_y,

                CAST(
                    (
                        grid_1km_row + 1
                    ) * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_y,

                COUNT(*) AS valid_100m_pixels,

                CAST(
                    COUNT(*) / 100.0
                    AS DOUBLE
                ) AS valid_coverage_fraction,

                SUM(
                    population_value
                ) AS population_total,

                AVG(
                    population_value
                ) AS population_mean_100m,

                MIN(
                    population_value
                ) AS population_min_100m,

                MAX(
                    population_value
                ) AS population_max_100m,

                SUM(
                    CASE
                        WHEN population_value = 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS zero_population_pixels,

                SUM(
                    CASE
                        WHEN population_value > 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS positive_population_pixels,

                MIN(source_path) AS source_path,

                '{SOURCE_RESOLUTION}' AS input_resolution_m,
                '{TARGET_RESOLUTION}' AS output_resolution_m,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM valid_population_pixels

            GROUP BY
                grid_1km_col,
                grid_1km_row
            """
        )


        # ====================================================
        # GRID-LEVEL VALIDATION
        # ====================================================

        grid_cell_count = (
            population_1km_df.count()
        )

        if grid_cell_count <= 0:
            raise RuntimeError(
                "No 1 km population cells were produced."
            )


        duplicate_grid_cell_count = (
            population_1km_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_grid_cell_count != 0:
            raise RuntimeError(
                "Duplicate 1 km grid cells were produced."
            )


        invalid_grid_count = (
            population_1km_df
            .filter(
                F.col("population_total").isNull()
                | F.isnan("population_total")
                | (
                    F.col("population_total")
                    < F.lit(0.0)
                )
                | F.col(
                    "valid_100m_pixels"
                ).isNull()
                | (
                    F.col("valid_100m_pixels")
                    <= F.lit(0)
                )
                | (
                    F.col("valid_100m_pixels")
                    > F.lit(
                        EXPECTED_PIXELS_PER_FULL_1KM_CELL
                    )
                )
                | (
                    F.col("valid_coverage_fraction")
                    <= F.lit(0.0)
                )
                | (
                    F.col("valid_coverage_fraction")
                    > F.lit(1.0)
                )
            )
            .count()
        )

        if invalid_grid_count != 0:
            raise RuntimeError(
                "One or more 1 km cells failed validation."
            )


        # ====================================================
        # CONSERVATION VALIDATION
        #
        # Sum of valid source pixels must equal sum of the
        # aggregated 1 km population totals.
        # ====================================================

        source_population_total = (
            valid_pixels_df
            .agg(
                F.sum(
                    "population_value"
                ).alias("total")
            )
            .first()["total"]
        )

        aggregated_population_total = (
            population_1km_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        if source_population_total is None:
            raise RuntimeError(
                "Source population total is NULL."
            )

        if aggregated_population_total is None:
            raise RuntimeError(
                "Aggregated population total is NULL."
            )

        source_population_total = float(
            source_population_total
        )

        aggregated_population_total = float(
            aggregated_population_total
        )

        absolute_difference = abs(
            source_population_total
            - aggregated_population_total
        )

        allowed_difference = max(
            1e-6,
            abs(source_population_total) * 1e-12,
        )

        if absolute_difference > allowed_difference:
            raise RuntimeError(
                "Population conservation failed. "
                f"Source total: {source_population_total}; "
                "aggregated total: "
                f"{aggregated_population_total}; "
                f"difference: {absolute_difference}."
            )


        print()
        print("=" * 80)
        print("1 KM AGGREGATION SUMMARY")
        print("=" * 80)
        print("1 km grid cells:", grid_cell_count)
        print(
            "Valid 100 m pilot pixels:",
            valid_pilot_pixel_count,
        )
        print(
            "Excluded pilot pixels:",
            excluded_pilot_pixel_count,
        )
        print(
            "Source population total:",
            source_population_total,
        )
        print(
            "Aggregated population total:",
            aggregated_population_total,
        )
        print(
            "Conservation difference:",
            absolute_difference,
        )


        # ====================================================
        # DISPLAY SAMPLE RECORDS
        # ====================================================

        print()
        print("=" * 80)
        print("SAMPLE 1 KM POPULATION RECORDS")
        print("=" * 80)

        (
            population_1km_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "valid_100m_pixels",
                "valid_coverage_fraction",
                "population_total",
                "positive_population_pixels",
                "grid_min_x",
                "grid_min_y",
            )
            .orderBy(
                F.col(
                    "population_total"
                ).desc(),
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE SCALAR 1 KM DATASET TO HDFS PARQUET
        #
        # No raster objects or raster bytes are written.
        # ====================================================

        (
            population_1km_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )

        print("HDFS Parquet write: COMPLETE")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if readback_count != grid_cell_count:
            raise RuntimeError(
                "HDFS read-back row count mismatch. "
                f"Expected {grid_cell_count}, "
                f"found {readback_count}."
            )


        readback_duplicate_count = (
            readback_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if readback_duplicate_count != 0:
            raise RuntimeError(
                "Read-back dataset contains duplicate cells."
            )


        readback_population_total = (
            readback_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        readback_population_total = float(
            readback_population_total
        )

        readback_difference = abs(
            readback_population_total
            - aggregated_population_total
        )

        if readback_difference > allowed_difference:
            raise RuntimeError(
                "HDFS read-back population total mismatch."
            )


        # ====================================================
        # OUTPUT COVERAGE SUMMARY
        # ====================================================

        coverage_summary_df = (
            readback_df
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),
                F.sum(
                    F.when(
                        F.col(
                            "valid_100m_pixels"
                        )
                        == F.lit(
                            EXPECTED_PIXELS_PER_FULL_1KM_CELL
                        ),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "complete_1km_cells"
                ),
                F.sum(
                    F.when(
                        F.col(
                            "valid_100m_pixels"
                        )
                        < F.lit(
                            EXPECTED_PIXELS_PER_FULL_1KM_CELL
                        ),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "partial_1km_cells"
                ),
                F.min(
                    "valid_coverage_fraction"
                ).alias(
                    "minimum_coverage"
                ),
                F.max(
                    "valid_coverage_fraction"
                ).alias(
                    "maximum_coverage"
                ),
                F.avg(
                    "valid_coverage_fraction"
                ).alias(
                    "average_coverage"
                ),
            )
        )


        print()
        print("=" * 80)
        print("OUTPUT COVERAGE SUMMARY")
        print("=" * 80)

        coverage_summary_df.show(
            truncate=False,
            vertical=True,
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 7 REMOTE RESULT")
        print("=" * 80)
        print("HDFS GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Sedona 256 x 256 tiling: PASS")
        print("Sedona Bengaluru tile selection: PASS")
        print("Sedona band-array extraction: PASS")
        print("Spark SQL pixel filtering: PASS")
        print("Spark SQL 100 m to 1 km aggregation: PASS")
        print("Population conservation: PASS")
        print("HDFS Parquet write: PASS")
        print("HDFS Parquet read-back: PASS")
        print("Output grid cells:", readback_count)
        print(
            "Population total:",
            readback_population_total,
        )
        print("Output:", OUTPUT_PATH)
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("RS_AsGeoTiff used: NO")
        print("RS_Clip used: NO")
        print("CELL 7: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER THE REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell07_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell07_population_1km.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 7 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 7: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 7: BENGALURU POPULATION 100 M TO 1 KM
Source: hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=1975/ghsl_india_population_1975_100m_r2023a-0000016384-0000008192.tif
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/grid_1km
City: bengaluru
Product: population
Epoch: 1975
Input resolution: 100.0 m
Output resolution: 1000.0 m
Expected source pixels per complete 1 km cell: 100
Required Sedona functions: PRESENT
Source raster metadata: PASS
Sedona source tile count: 1024
Selected Bengaluru tiles: 12

PIXEL QUALITY SUMMARY
-RECORD 0---------------------------------
 selected_tile_pixel_capacity    | 786432 
 pilot_pixel_count               | 359856 
 valid_pilot_pixel_count         | 359856 
 excluded_pilot_pixel_count      | 0      
 zero

What completed

Using Apache Spark, Apache Sedona, and HDFS:

The 1975 population GeoTIFF was read from HDFS.
Sedona decoded the raster and created 1,024 georeferenced tiles.
Sedona selected the 12 tiles covering the Bengaluru pilot.
RS_BandAsArray exposed the selected tile pixels to Spark SQL.
Spark processed 359,856 pilot pixels and aggregated them into 3,658 one-kilometre cells.
All 359,856 pilot pixels were valid. No pixels were excluded.
The output passed write and read-back validation in HDFS.
Numerical validation
Metric	ResultSelected tile capacity	786,432 pixels
Pixels inside pilot	359,856
Valid pixels	359,856
Zero-population pixels	324,953
Positive-population pixels	34,903
1 km cells	3,658
Complete 1 km cells	3,480
Partial boundary cells	178
Minimum boundary coverage	40%
Average coverage	98.38%
Population total	3,060,319.09

The input population total was 3,060,319.0921731517, while the aggregated total was 3,060,319.0921731493. The difference was only 2.33 × 10⁻⁹, so population conservation passed.

Output produced

The scalar Parquet dataset is stored at:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/grid_1km

This output is persistent and can be used without rereading the original population GeoTIFF.

Stack usage

This run achieved the architecture you wanted:

Plain Text
HDFS
→ Spark binaryFile
→ Sedona raster decoding
→ Sedona tiling
→ Sedona band extraction
→ Spark SQL filtering and aggregation
→ HDFS Parquet

No Python raster UDF, NumPy, Rasterio, RS_Clip, or RS_AsGeoTiff was used.

Next step

The next dataset should be SMOD 1975 for the same Bengaluru pilot.

SMOD is already at 1 km resolution, so the next stage will:

Identify the one 1975 SMOD GeoTIFF from the metadata catalog.
Read it from HDFS using Spark/Sedona.
Select only the Bengaluru 1 km cells.
Extract the SMOD settlement class per cell.
create the same grid_cell_id;
write the SMOD scalar records to HDFS;
join SMOD with the persistent population output.

We should not add built surface yet. First, complete and validate:

Plain Text
Bengaluru 1975 population + SMOD

In [16]:
# ============================================================
# CELL 8: BENGALURU 1975 SMOD 1 KM DATASET
#
# Pipeline:
#   HDFS metadata catalog
#   -> Spark identifies the 1975 SMOD source
#   -> Spark binaryFile reads the GeoTIFF
#   -> Sedona decodes and tiles the raster
#   -> Sedona RS_BandAsArray extracts SMOD values
#   -> Spark SQL selects Bengaluru cells
#   -> Spark writes scalar Parquet to HDFS
#
# No NumPy, Rasterio, Python raster UDF, RS_Clip,
# or RS_AsGeoTiff.
# ============================================================

import base64
import textwrap


cell08_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )

    POPULATION_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=1975/"
        "grid_1km"
    )

    SMOD_OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/"
        "epoch=1975/"
        "grid_1km"
    )

    CITY = "bengaluru"
    PRODUCT = "smod"
    EPOCH = 1975

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    EXPECTED_RESOLUTION = 1000.0
    EXPECTED_POPULATION_CELLS = 3658

    # Verified World Mollweide pilot envelope.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-SMOD-1km"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 8: BENGALURU 1975 SMOD 1 KM")
        print("=" * 80)
        print("Catalog:", HDFS_CATALOG)
        print("Population reference:", POPULATION_PATH)
        print("SMOD output:", SMOD_OUTPUT_PATH)


        # ====================================================
        # DISCOVER THE SMOD SOURCE FROM THE HDFS CATALOG
        #
        # Only one small catalog row is collected to the
        # Python driver. No raster data is collected.
        # ====================================================

        catalog_df = spark.read.parquet(
            HDFS_CATALOG
        )

        smod_catalog_df = (
            catalog_df
            .filter(
                F.lower(
                    F.col("product")
                ) == F.lit("smod")
            )
            .filter(
                F.col(
                    "temporal_value"
                ).cast("int")
                == F.lit(EPOCH)
            )
            .select(
                "filename",
                "path",
                "fileSize",
                "width",
                "height",
                "resolution_x",
                "resolution_y",
                "min_x",
                "min_y",
                "max_x",
                "max_y",
            )
        )

        smod_catalog_count = (
            smod_catalog_df.count()
        )

        print(
            "1975 SMOD catalog records:",
            smod_catalog_count,
        )

        if smod_catalog_count != 1:
            print("Matching catalog records:")
            smod_catalog_df.show(
                20,
                truncate=False,
                vertical=True,
            )

            raise RuntimeError(
                "Expected exactly one 1975 SMOD source, "
                f"found {smod_catalog_count}."
            )

        smod_catalog_row = (
            smod_catalog_df.first()
        )

        smod_source_path = (
            smod_catalog_row["path"]
        )

        print("SMOD filename:",
              smod_catalog_row["filename"])
        print("SMOD source:", smod_source_path)
        print(
            "Catalog dimensions:",
            smod_catalog_row["width"],
            "x",
            smod_catalog_row["height"],
        )
        print(
            "Catalog resolution:",
            smod_catalog_row["resolution_x"],
            "x",
            smod_catalog_row["resolution_y"],
        )


        # ====================================================
        # READ AND DECODE THE SMOD GEOTIFF FROM HDFS
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(smod_source_path)
            .select(
                "path",
                F.col("length").alias(
                    "source_file_bytes"
                ),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one SMOD GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "smod_source"
        )


        # ====================================================
        # VALIDATE SOURCE RASTER METADATA
        # ====================================================

        source_metadata_df = spark.sql(
            """
            SELECT
                path,
                source_file_bytes,

                RS_Width(
                    source_raster
                ) AS source_width,

                RS_Height(
                    source_raster
                ) AS source_height,

                RS_NumBands(
                    source_raster
                ) AS source_band_count,

                RS_ScaleX(
                    source_raster
                ) AS source_scale_x,

                RS_ScaleY(
                    source_raster
                ) AS source_scale_y,

                RS_UpperLeftX(
                    source_raster
                ) AS source_upper_left_x,

                RS_UpperLeftY(
                    source_raster
                ) AS source_upper_left_y

            FROM smod_source
            """
        )

        source_metadata = (
            source_metadata_df.first()
        )

        source_width = int(
            source_metadata["source_width"]
        )

        source_height = int(
            source_metadata["source_height"]
        )

        source_scale_x = float(
            source_metadata["source_scale_x"]
        )

        source_scale_y = float(
            source_metadata["source_scale_y"]
        )

        if (
            int(
                source_metadata[
                    "source_band_count"
                ]
            )
            != 1
        ):
            raise RuntimeError(
                "Expected one SMOD raster band."
            )

        if (
            abs(
                source_scale_x
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected SMOD X resolution: "
                f"{source_scale_x}"
            )

        if (
            abs(
                abs(source_scale_y)
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected SMOD Y resolution: "
                f"{source_scale_y}"
            )

        print()
        print("=" * 80)
        print("SMOD SOURCE METADATA")
        print("=" * 80)

        source_metadata_df.show(
            truncate=False,
            vertical=True,
        )

        print("SMOD source metadata: PASS")


        # ====================================================
        # DETERMINE THE TILE LAYOUT
        # ====================================================

        tile_columns = (
            source_width + TILE_WIDTH - 1
        ) // TILE_WIDTH

        tile_rows = (
            source_height + TILE_HEIGHT - 1
        ) // TILE_HEIGHT

        expected_tile_count = (
            tile_columns * tile_rows
        )

        print(
            "SMOD tile layout:",
            f"{tile_columns} x {tile_rows}",
        )
        print(
            "Expected SMOD tiles:",
            expected_tile_count,
        )


        # ====================================================
        # TILE THE SMOD RASTER WITH SEDONA
        #
        # Edge tiles are allowed to be smaller than 256 pixels.
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path AS source_path,
                source_file_bytes,

                tile_index,

                CAST(
                    FLOOR(
                        tile_index / {tile_columns}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {tile_columns}
                    AS INT
                ) AS tile_col,

                tile

            FROM smod_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded
            AS tile_index, tile
            """
        )

        tile_count = tiles_df.count()

        if tile_count != expected_tile_count:
            raise RuntimeError(
                "Unexpected SMOD tile count. "
                f"Expected {expected_tile_count}, "
                f"found {tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "smod_tiles"
        )

        print("Sedona SMOD tile count:", tile_count)


        # ====================================================
        # SELECT ONLY TILES INTERSECTING BENGALURU
        # ====================================================

        pilot_df = spark.sql(
            f"""
            SELECT
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                ) AS pilot_envelope
            """
        )

        pilot_df.createOrReplaceTempView(
            "bengaluru_pilot"
        )


        selected_tiles_df = spark.sql(
            """
            SELECT
                t.source_path,
                t.tile_index,
                t.tile_row,
                t.tile_col,

                RS_Width(
                    t.tile
                ) AS tile_width,

                RS_Height(
                    t.tile
                ) AS tile_height,

                RS_UpperLeftX(
                    t.tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    t.tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    t.tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    t.tile
                ) AS tile_scale_y,

                t.tile

            FROM smod_tiles AS t

            CROSS JOIN bengaluru_pilot AS b

            WHERE ST_Intersects(
                RS_Envelope(t.tile),
                b.pilot_envelope
            )
            """
        )

        selected_tile_count = (
            selected_tiles_df.count()
        )

        if selected_tile_count < 1:
            raise RuntimeError(
                "No SMOD tile intersects Bengaluru."
            )

        print(
            "Selected Bengaluru SMOD tiles:",
            selected_tile_count,
        )

        selected_tiles_df.createOrReplaceTempView(
            "selected_smod_tiles"
        )


        # ====================================================
        # EXTRACT SMOD VALUES WITH SEDONA
        #
        # pixel_index is zero-based within each selected tile.
        # ====================================================

        pixel_df = spark.sql(
            """
            SELECT
                source_path,
                tile_index,
                tile_row,
                tile_col,

                tile_width,
                tile_height,

                tile_upper_left_x,
                tile_upper_left_y,

                tile_scale_x,
                tile_scale_y,

                pixel_index,

                CAST(
                    FLOOR(
                        pixel_index / tile_width
                    )
                    AS INT
                ) AS local_pixel_row,

                CAST(
                    pixel_index % tile_width
                    AS INT
                ) AS local_pixel_col,

                CAST(
                    smod_value
                    AS DOUBLE
                ) AS smod_value

            FROM selected_smod_tiles

            LATERAL VIEW posexplode(
                RS_BandAsArray(
                    tile,
                    1
                )
            ) exploded
            AS pixel_index, smod_value
            """
        )

        pixel_df.createOrReplaceTempView(
            "smod_pixels"
        )


        # ====================================================
        # POSITION EACH 1 KM PIXEL AND CREATE GRID IDENTIFIERS
        #
        # grid_cell_id uses the same World Mollweide 1 km
        # indexing convention used by Cell 7.
        # ====================================================

        positioned_df = spark.sql(
            f"""
            SELECT
                source_path,

                (
                    tile_upper_left_x
                    + (
                        local_pixel_col + 0.5
                    ) * tile_scale_x
                ) AS pixel_center_x,

                (
                    tile_upper_left_y
                    + (
                        local_pixel_row + 0.5
                    ) * tile_scale_y
                ) AS pixel_center_y,

                smod_value

            FROM smod_pixels
            """
        )

        positioned_df.createOrReplaceTempView(
            "positioned_smod_pixels"
        )


        smod_grid_df = spark.sql(
            f"""
            SELECT
                '{CITY}' AS city,
                '{PRODUCT}' AS product,
                CAST({EPOCH} AS INT) AS epoch,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {EXPECTED_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_col,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {EXPECTED_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_row,

                CONCAT(
                    'MOLL_1KM_',
                    CAST(
                        FLOOR(
                            pixel_center_x
                            / {EXPECTED_RESOLUTION}
                        )
                        AS STRING
                    ),
                    '_',
                    CAST(
                        FLOOR(
                            pixel_center_y
                            / {EXPECTED_RESOLUTION}
                        )
                        AS STRING
                    )
                ) AS grid_cell_id,

                CAST(
                    smod_value
                    AS INT
                ) AS smod_class,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {EXPECTED_RESOLUTION}
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_x,

                CAST(
                    (
                        FLOOR(
                            pixel_center_x
                            / {EXPECTED_RESOLUTION}
                        ) + 1
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_x,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {EXPECTED_RESOLUTION}
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_y,

                CAST(
                    (
                        FLOOR(
                            pixel_center_y
                            / {EXPECTED_RESOLUTION}
                        ) + 1
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_y,

                source_path,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM positioned_smod_pixels

            WHERE
                pixel_center_x >= {PILOT_MIN_X}
                AND pixel_center_x < {PILOT_MAX_X}
                AND pixel_center_y >= {PILOT_MIN_Y}
                AND pixel_center_y < {PILOT_MAX_Y}

                AND smod_value IS NOT NULL
                AND NOT isnan(smod_value)
                AND ABS(smod_value)
                    <= 1.7976931348623157E308
                AND smod_value >= 0.0
            """
        )


        # ====================================================
        # VALIDATE SMOD GRID
        # ====================================================

        smod_grid_count = (
            smod_grid_df.count()
        )

        if smod_grid_count <= 0:
            raise RuntimeError(
                "No Bengaluru SMOD cells were produced."
            )


        duplicate_count = (
            smod_grid_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "Duplicate SMOD grid cells were produced."
            )


        noninteger_smod_count = (
            smod_grid_df
            .filter(
                F.col("smod_class").isNull()
            )
            .count()
        )

        if noninteger_smod_count != 0:
            raise RuntimeError(
                "Invalid SMOD classes were produced."
            )


        print()
        print("=" * 80)
        print("SMOD CLASS DISTRIBUTION")
        print("=" * 80)

        (
            smod_grid_df
            .groupBy("smod_class")
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                )
            )
            .orderBy("smod_class")
            .show(
                100,
                truncate=False,
            )
        )


        # ====================================================
        # COMPARE SMOD COVERAGE AGAINST POPULATION CELLS
        # ====================================================

        population_df = spark.read.parquet(
            POPULATION_PATH
        )

        population_count = (
            population_df.count()
        )

        if (
            population_count
            != EXPECTED_POPULATION_CELLS
        ):
            raise RuntimeError(
                "Unexpected population grid count. "
                f"Expected {EXPECTED_POPULATION_CELLS}, "
                f"found {population_count}."
            )


        overlap_count = (
            population_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                smod_grid_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="inner",
            )
            .count()
        )


        population_without_smod = (
            population_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                smod_grid_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )


        print()
        print("=" * 80)
        print("POPULATION AND SMOD COVERAGE")
        print("=" * 80)
        print(
            "Population grid cells:",
            population_count,
        )
        print(
            "SMOD pilot cells:",
            smod_grid_count,
        )
        print(
            "Population-SMOD overlapping cells:",
            overlap_count,
        )
        print(
            "Population cells without SMOD:",
            population_without_smod,
        )

        if population_without_smod != 0:
            raise RuntimeError(
                "One or more population grid cells do not "
                "have a corresponding SMOD cell."
            )


        # ====================================================
        # WRITE SCALAR SMOD PARQUET TO HDFS
        # ====================================================

        (
            smod_grid_df
            .repartition(2)
            .write
            .mode("overwrite")
            .parquet(SMOD_OUTPUT_PATH)
        )

        print("HDFS SMOD Parquet write: COMPLETE")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            SMOD_OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if readback_count != smod_grid_count:
            raise RuntimeError(
                "SMOD read-back row count mismatch."
            )


        readback_duplicate_count = (
            readback_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if readback_duplicate_count != 0:
            raise RuntimeError(
                "SMOD read-back contains duplicate cells."
            )


        # ====================================================
        # DISPLAY SAMPLE
        # ====================================================

        print()
        print("=" * 80)
        print("SAMPLE SMOD RECORDS")
        print("=" * 80)

        (
            readback_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "smod_class",
                "grid_min_x",
                "grid_min_y",
            )
            .orderBy(
                "grid_1km_row",
                "grid_1km_col",
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 8 REMOTE RESULT")
        print("=" * 80)
        print("SMOD source discovery from catalog: PASS")
        print("HDFS SMOD GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Sedona SMOD tiling: PASS")
        print("Sedona band-array extraction: PASS")
        print("Spark SQL Bengaluru filtering: PASS")
        print("Spark SQL grid identifier creation: PASS")
        print("Population-SMOD coverage validation: PASS")
        print("HDFS SMOD Parquet write: PASS")
        print("HDFS SMOD Parquet read-back: PASS")
        print("SMOD output cells:", readback_count)
        print(
            "Population cells covered by SMOD:",
            overlap_count,
        )
        print("Output:", SMOD_OUTPUT_PATH)
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("RS_Clip used: NO")
        print("RS_AsGeoTiff used: NO")
        print("CELL 8: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell08_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell08_smod_1975.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 8 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 8: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 8: BENGALURU 1975 SMOD 1 KM
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Population reference: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/grid_1km
SMOD output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1975/grid_1km
1975 SMOD catalog records: 1
SMOD filename: ghsl_india_smod_1975_1000m_r2023a.tif
SMOD source: hdfs://localhost:9000/data/ghsl/raw/india/smod/epoch=1975/ghsl_india_smod_1975_1000m_r2023a.tif
Catalog dimensions: 2902 x 3179
Catalog resolution: 1000.0 x 1000.0

SMOD SOURCE METADATA
-RECORD 0--------------------------------------------------------------------------------------------------------------
 path                | hdfs://localhost:9000/data/ghsl/raw/india/smod/epoch=1

In [17]:
# ============================================================
# CELL 9: JOIN BENGALURU 1975 POPULATION + SMOD
#
# Input datasets:
#   1. Population aggregated from 100 m to 1 km
#   2. SMOD settlement class at 1 km
#
# Output:
#   First Bengaluru grid-year analytical dataset
#
# Primary tools:
#   - HDFS: persistent input and output
#   - Spark DataFrames / Spark SQL: validation and join
#   - Sedona: creation of 1 km cell geometry
#
# No source GeoTIFF is reread.
# No NumPy, Rasterio, or Python UDF is used.
# ============================================================

import base64
import textwrap


cell09_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        IntegerType,
        LongType,
        DoubleType,
        StringType,
    )


    # ========================================================
    # CONFIGURATION
    # ========================================================

    POPULATION_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=1975/"
        "grid_1km"
    )

    SMOD_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/"
        "epoch=1975/"
        "grid_1km"
    )

    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "grid_year/"
        "epoch=1975"
    )

    CITY = "bengaluru"
    EPOCH = 1975

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_POPULATION_TOTAL = (
        3060319.0921731493
    )

    COMPLETE_CELL_PIXELS = 100
    TARGET_RESOLUTION_METERS = 1000.0


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-Grid-Year-Join"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 9: BENGALURU 1975 GRID-YEAR JOIN")
        print("=" * 80)
        print("Population input:", POPULATION_PATH)
        print("SMOD input:", SMOD_PATH)
        print("Output:", OUTPUT_PATH)


        # ====================================================
        # READ THE TWO PERSISTENT HDFS PARQUET DATASETS
        # ====================================================

        population_raw_df = spark.read.parquet(
            POPULATION_PATH
        )

        smod_raw_df = spark.read.parquet(
            SMOD_PATH
        )

        print()
        print("=" * 80)
        print("INPUT SCHEMAS")
        print("=" * 80)

        print("Population schema:")
        population_raw_df.printSchema()

        print("SMOD schema:")
        smod_raw_df.printSchema()


        # ====================================================
        # SELECT AND NORMALIZE REQUIRED COLUMNS
        #
        # Aliases prevent ambiguous column names after joining.
        # ====================================================

        population_df = (
            population_raw_df
            .select(
                F.col("grid_cell_id").cast(
                    StringType()
                ).alias("grid_cell_id"),

                F.col("epoch").cast(
                    IntegerType()
                ).alias("epoch"),

                F.col("grid_1km_col").cast(
                    LongType()
                ).alias("population_grid_col"),

                F.col("grid_1km_row").cast(
                    LongType()
                ).alias("population_grid_row"),

                F.col("grid_min_x").cast(
                    DoubleType()
                ).alias("population_grid_min_x"),

                F.col("grid_max_x").cast(
                    DoubleType()
                ).alias("population_grid_max_x"),

                F.col("grid_min_y").cast(
                    DoubleType()
                ).alias("population_grid_min_y"),

                F.col("grid_max_y").cast(
                    DoubleType()
                ).alias("population_grid_max_y"),

                F.col("valid_100m_pixels").cast(
                    LongType()
                ).alias("valid_100m_pixels"),

                F.col(
                    "valid_coverage_fraction"
                ).cast(
                    DoubleType()
                ).alias(
                    "population_coverage_fraction"
                ),

                F.col("population_total").cast(
                    DoubleType()
                ).alias("population_total"),

                F.col(
                    "population_mean_100m"
                ).cast(
                    DoubleType()
                ).alias(
                    "population_mean_100m"
                ),

                F.col(
                    "population_min_100m"
                ).cast(
                    DoubleType()
                ).alias(
                    "population_min_100m"
                ),

                F.col(
                    "population_max_100m"
                ).cast(
                    DoubleType()
                ).alias(
                    "population_max_100m"
                ),

                F.col(
                    "zero_population_pixels"
                ).cast(
                    LongType()
                ).alias(
                    "zero_population_pixels"
                ),

                F.col(
                    "positive_population_pixels"
                ).cast(
                    LongType()
                ).alias(
                    "positive_population_pixels"
                ),

                F.col("source_path").alias(
                    "population_source_path"
                ),
            )
        )


        smod_df = (
            smod_raw_df
            .select(
                F.col("grid_cell_id").cast(
                    StringType()
                ).alias("grid_cell_id"),

                F.col("epoch").cast(
                    IntegerType()
                ).alias("epoch"),

                F.col("grid_1km_col").cast(
                    LongType()
                ).alias("smod_grid_col"),

                F.col("grid_1km_row").cast(
                    LongType()
                ).alias("smod_grid_row"),

                F.col("grid_min_x").cast(
                    DoubleType()
                ).alias("smod_grid_min_x"),

                F.col("grid_max_x").cast(
                    DoubleType()
                ).alias("smod_grid_max_x"),

                F.col("grid_min_y").cast(
                    DoubleType()
                ).alias("smod_grid_min_y"),

                F.col("grid_max_y").cast(
                    DoubleType()
                ).alias("smod_grid_max_y"),

                F.col("smod_class").cast(
                    IntegerType()
                ).alias("smod_class"),

                F.col("source_path").alias(
                    "smod_source_path"
                ),
            )
        )


        # ====================================================
        # INPUT VALIDATION
        # ====================================================

        population_count = population_df.count()
        smod_count = smod_df.count()

        print()
        print("=" * 80)
        print("INPUT ROW COUNTS")
        print("=" * 80)
        print("Population rows:", population_count)
        print("SMOD rows:", smod_count)

        if population_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected population input count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {population_count}."
            )

        if smod_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected SMOD input count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {smod_count}."
            )


        def duplicate_key_count(dataframe):
            return (
                dataframe
                .groupBy(
                    "grid_cell_id",
                    "epoch",
                )
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )


        population_duplicate_count = (
            duplicate_key_count(population_df)
        )

        smod_duplicate_count = (
            duplicate_key_count(smod_df)
        )

        if population_duplicate_count != 0:
            raise RuntimeError(
                "Population input contains duplicate "
                "grid_cell_id/epoch keys."
            )

        if smod_duplicate_count != 0:
            raise RuntimeError(
                "SMOD input contains duplicate "
                "grid_cell_id/epoch keys."
            )


        incorrect_population_epoch_count = (
            population_df
            .filter(
                F.col("epoch") != F.lit(EPOCH)
            )
            .count()
        )

        incorrect_smod_epoch_count = (
            smod_df
            .filter(
                F.col("epoch") != F.lit(EPOCH)
            )
            .count()
        )

        if (
            incorrect_population_epoch_count != 0
            or incorrect_smod_epoch_count != 0
        ):
            raise RuntimeError(
                "One or more input rows have an "
                "unexpected epoch."
            )


        # ====================================================
        # VERIFY TWO-WAY KEY COVERAGE BEFORE JOINING
        # ====================================================

        population_without_smod = (
            population_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                smod_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )

        smod_without_population = (
            smod_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                population_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )

        print()
        print("=" * 80)
        print("TWO-WAY KEY COVERAGE")
        print("=" * 80)
        print(
            "Population cells without SMOD:",
            population_without_smod,
        )
        print(
            "SMOD cells without population:",
            smod_without_population,
        )

        if population_without_smod != 0:
            raise RuntimeError(
                "Population contains cells without SMOD."
            )

        if smod_without_population != 0:
            raise RuntimeError(
                "SMOD contains cells without population."
            )


        # ====================================================
        # STRICT INNER JOIN
        # ====================================================

        joined_raw_df = (
            population_df.alias("p")
            .join(
                smod_df.alias("s"),
                on=[
                    F.col("p.grid_cell_id")
                    == F.col("s.grid_cell_id"),

                    F.col("p.epoch")
                    == F.col("s.epoch"),
                ],
                how="inner",
            )
        )


        # ====================================================
        # VALIDATE GRID COORDINATE AGREEMENT
        #
        # The join key matching is not sufficient by itself.
        # Both datasets must independently report identical
        # row, column and numeric bounds.
        # ====================================================

        coordinate_mismatch_count = (
            joined_raw_df
            .filter(
                (
                    F.col("p.population_grid_col")
                    != F.col("s.smod_grid_col")
                )
                | (
                    F.col("p.population_grid_row")
                    != F.col("s.smod_grid_row")
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_min_x"
                        )
                        - F.col(
                            "s.smod_grid_min_x"
                        )
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_max_x"
                        )
                        - F.col(
                            "s.smod_grid_max_x"
                        )
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_min_y"
                        )
                        - F.col(
                            "s.smod_grid_min_y"
                        )
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_max_y"
                        )
                        - F.col(
                            "s.smod_grid_max_y"
                        )
                    )
                    > F.lit(1e-9)
                )
            )
            .count()
        )

        if coordinate_mismatch_count != 0:
            raise RuntimeError(
                "Population and SMOD grid coordinates "
                "do not agree for "
                f"{coordinate_mismatch_count} cells."
            )

        print("Grid coordinate agreement: PASS")


        # ====================================================
        # CREATE ANALYTICAL GRID-YEAR RECORDS
        #
        # Settlement labels follow the broad GHSL SMOD class
        # hierarchy represented in the input values:
        #
        #   10      Water
        #   11-13   Rural domain
        #   21-23   Urban cluster domain
        #   30      Urban centre
        #
        # The original numeric SMOD class is always preserved.
        # ====================================================

        grid_year_df = (
            joined_raw_df
            .select(
                F.lit(CITY).alias("city"),

                F.col(
                    "p.grid_cell_id"
                ).alias("grid_cell_id"),

                F.col("p.epoch").alias("epoch"),

                F.col(
                    "p.population_grid_col"
                ).alias("grid_1km_col"),

                F.col(
                    "p.population_grid_row"
                ).alias("grid_1km_row"),

                F.col(
                    "p.population_grid_min_x"
                ).alias("grid_min_x"),

                F.col(
                    "p.population_grid_max_x"
                ).alias("grid_max_x"),

                F.col(
                    "p.population_grid_min_y"
                ).alias("grid_min_y"),

                F.col(
                    "p.population_grid_max_y"
                ).alias("grid_max_y"),

                F.col(
                    "p.valid_100m_pixels"
                ).alias("valid_100m_pixels"),

                F.col(
                    "p.population_coverage_fraction"
                ).alias(
                    "population_coverage_fraction"
                ),

                F.col(
                    "p.population_total"
                ).alias("population_total"),

                F.col(
                    "p.population_mean_100m"
                ).alias("population_mean_100m"),

                F.col(
                    "p.population_min_100m"
                ).alias("population_min_100m"),

                F.col(
                    "p.population_max_100m"
                ).alias("population_max_100m"),

                F.col(
                    "p.zero_population_pixels"
                ).alias(
                    "zero_population_pixels"
                ),

                F.col(
                    "p.positive_population_pixels"
                ).alias(
                    "positive_population_pixels"
                ),

                F.col(
                    "s.smod_class"
                ).alias("smod_class"),

                F.when(
                    F.col("s.smod_class")
                    == F.lit(10),
                    F.lit("water"),
                )
                .when(
                    F.col("s.smod_class").between(
                        11,
                        13,
                    ),
                    F.lit("rural"),
                )
                .when(
                    F.col("s.smod_class").between(
                        21,
                        23,
                    ),
                    F.lit("urban_cluster"),
                )
                .when(
                    F.col("s.smod_class")
                    == F.lit(30),
                    F.lit("urban_centre"),
                )
                .otherwise(
                    F.lit("other_or_unknown")
                )
                .alias("settlement_group"),

                F.when(
                    F.col("s.smod_class").isin(
                        21,
                        22,
                        23,
                        30,
                    ),
                    F.lit(True),
                )
                .otherwise(
                    F.lit(False)
                )
                .alias("is_urban"),

                F.when(
                    F.col("s.smod_class")
                    == F.lit(30),
                    F.lit(True),
                )
                .otherwise(
                    F.lit(False)
                )
                .alias("is_urban_centre"),

                F.when(
                    F.col("p.valid_100m_pixels")
                    == F.lit(COMPLETE_CELL_PIXELS),
                    F.lit(True),
                )
                .otherwise(
                    F.lit(False)
                )
                .alias("is_complete_population_cell"),

                F.when(
                    (
                        F.col(
                            "p.valid_100m_pixels"
                        )
                        == F.lit(
                            COMPLETE_CELL_PIXELS
                        )
                    )
                    & (
                        F.col(
                            "p.population_coverage_fraction"
                        )
                        == F.lit(1.0)
                    ),
                    F.lit("complete"),
                )
                .otherwise(
                    F.lit(
                        "partial_pilot_boundary"
                    )
                )
                .alias("population_quality_flag"),

                F.col(
                    "p.population_source_path"
                ).alias("population_source_path"),

                F.col(
                    "s.smod_source_path"
                ).alias("smod_source_path"),
            )
        )


        # ====================================================
        # CREATE 1 KM CELL GEOMETRY WITH SEDONA
        #
        # Geometry remains in World Mollweide coordinates.
        # WKB is used for persistence because Parquet handles
        # binary columns reliably.
        # ====================================================

        grid_year_df.createOrReplaceTempView(
            "grid_year_scalar"
        )

        grid_year_with_geometry_df = spark.sql(
            """
            SELECT
                *,

                ST_AsBinary(
                    ST_PolygonFromEnvelope(
                        grid_min_x,
                        grid_min_y,
                        grid_max_x,
                        grid_max_y
                    )
                ) AS geometry_wkb,

                'ESRI:54009' AS geometry_crs,

                1000.0 AS grid_resolution_m,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM grid_year_scalar
            """
        )


        # ====================================================
        # OUTPUT VALIDATION
        # ====================================================

        output_count = (
            grid_year_with_geometry_df.count()
        )

        if output_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected joined output count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {output_count}."
            )


        output_duplicate_count = (
            grid_year_with_geometry_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if output_duplicate_count != 0:
            raise RuntimeError(
                "Joined output contains duplicate cells."
            )


        invalid_output_count = (
            grid_year_with_geometry_df
            .filter(
                F.col("grid_cell_id").isNull()
                | F.col("population_total").isNull()
                | F.isnan("population_total")
                | (
                    F.col("population_total")
                    < F.lit(0.0)
                )
                | F.col("smod_class").isNull()
                | F.col("settlement_group").isNull()
                | F.col("geometry_wkb").isNull()
                | (
                    F.length("geometry_wkb")
                    <= F.lit(0)
                )
                | F.col(
                    "population_quality_flag"
                ).isNull()
            )
            .count()
        )

        if invalid_output_count != 0:
            raise RuntimeError(
                "One or more joined records failed "
                "required-field validation."
            )


        unknown_settlement_count = (
            grid_year_with_geometry_df
            .filter(
                F.col("settlement_group")
                == F.lit("other_or_unknown")
            )
            .count()
        )

        if unknown_settlement_count != 0:
            raise RuntimeError(
                "Unexpected SMOD classes generated "
                f"{unknown_settlement_count} unknown "
                "settlement records."
            )


        # ====================================================
        # POPULATION CONSERVATION AFTER JOIN
        # ====================================================

        joined_population_total = (
            grid_year_with_geometry_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        joined_population_total = float(
            joined_population_total
        )

        absolute_difference = abs(
            joined_population_total
            - EXPECTED_POPULATION_TOTAL
        )

        allowed_difference = max(
            1e-6,
            abs(EXPECTED_POPULATION_TOTAL)
            * 1e-12,
        )

        if absolute_difference > allowed_difference:
            raise RuntimeError(
                "Population total changed during join. "
                f"Expected {EXPECTED_POPULATION_TOTAL}, "
                f"found {joined_population_total}, "
                f"difference {absolute_difference}."
            )


        # ====================================================
        # ANALYTICAL SUMMARIES
        # ====================================================

        print()
        print("=" * 80)
        print("SETTLEMENT GROUP SUMMARY")
        print("=" * 80)

        settlement_summary_df = (
            grid_year_with_geometry_df
            .groupBy(
                "settlement_group",
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_total"
                ).alias(
                    "population_total"
                ),

                F.avg(
                    "population_total"
                ).alias(
                    "average_population_per_cell"
                ),

                F.sum(
                    F.when(
                        F.col(
                            "population_total"
                        )
                        > F.lit(0.0),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "populated_grid_cells"
                ),
            )
            .orderBy(
                "settlement_group",
            )
        )

        settlement_summary_df.show(
            20,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("QUALITY SUMMARY")
        print("=" * 80)

        quality_summary_df = (
            grid_year_with_geometry_df
            .groupBy(
                "population_quality_flag",
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),
                F.sum(
                    "valid_100m_pixels"
                ).alias(
                    "valid_100m_pixel_count"
                ),
                F.sum(
                    "population_total"
                ).alias(
                    "population_total"
                ),
            )
            .orderBy(
                "population_quality_flag",
            )
        )

        quality_summary_df.show(
            20,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("TOP 25 POPULATION CELLS")
        print("=" * 80)

        (
            grid_year_with_geometry_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "population_total",
                "valid_100m_pixels",
                "population_coverage_fraction",
                "smod_class",
                "settlement_group",
                "is_urban",
                "population_quality_flag",
            )
            .orderBy(
                F.col(
                    "population_total"
                ).desc(),
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE GRID-YEAR DATASET TO HDFS
        # ====================================================

        (
            grid_year_with_geometry_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )

        print("HDFS grid-year Parquet write: COMPLETE")


        # ====================================================
        # HDFS READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if readback_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Grid-year read-back count mismatch. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {readback_count}."
            )


        readback_duplicate_count = (
            readback_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if readback_duplicate_count != 0:
            raise RuntimeError(
                "Grid-year read-back contains duplicates."
            )


        readback_population_total = (
            readback_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        readback_population_total = float(
            readback_population_total
        )

        readback_difference = abs(
            readback_population_total
            - joined_population_total
        )

        if readback_difference > allowed_difference:
            raise RuntimeError(
                "Grid-year HDFS read-back changed "
                "the population total."
            )


        complete_cell_count = (
            readback_df
            .filter(
                F.col(
                    "population_quality_flag"
                )
                == F.lit("complete")
            )
            .count()
        )

        partial_cell_count = (
            readback_df
            .filter(
                F.col(
                    "population_quality_flag"
                )
                == F.lit(
                    "partial_pilot_boundary"
                )
            )
            .count()
        )

        urban_cell_count = (
            readback_df
            .filter(
                F.col("is_urban")
            )
            .count()
        )

        urban_centre_cell_count = (
            readback_df
            .filter(
                F.col("is_urban_centre")
            )
            .count()
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 9 REMOTE RESULT")
        print("=" * 80)
        print("Population HDFS Parquet read: PASS")
        print("SMOD HDFS Parquet read: PASS")
        print("Two-way key coverage: PASS")
        print("Grid coordinate agreement: PASS")
        print("Spark one-to-one join: PASS")
        print("Settlement-group derivation: PASS")
        print("Sedona 1 km geometry creation: PASS")
        print("Population conservation after join: PASS")
        print("HDFS grid-year Parquet write: PASS")
        print("HDFS grid-year Parquet read-back: PASS")
        print("Grid-year rows:", readback_count)
        print("Complete population cells:",
              complete_cell_count)
        print("Partial boundary cells:",
              partial_cell_count)
        print("Urban cells:", urban_cell_count)
        print(
            "Urban-centre cells:",
            urban_centre_cell_count,
        )
        print(
            "Population total:",
            readback_population_total,
        )
        print("Output:", OUTPUT_PATH)
        print("Source GeoTIFF reread: NO")
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 9: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell09_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell09_grid_year_join.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 2g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1200,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 9 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 9: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 9: BENGALURU 1975 GRID-YEAR JOIN
Population input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=1975/grid_1km
SMOD input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1975/grid_1km
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975

INPUT SCHEMAS
Population schema:
root
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- epoch: integer (nullable = true)
 |-- grid_1km_col: long (nullable = true)
 |-- grid_1km_row: long (nullable = true)
 |-- grid_cell_id: string (nullable = true)
 |-- grid_min_x: double (nullable = true)
 |-- grid_max_x: double (nullable = true)
 |-- grid_min_y: double (nullable = true)
 |-- grid_max_y: double (nullable = true)
 |-- valid_100m_

Expected result

The joined output should contain:

3,658 grid-year rows
3,480 complete population cells
178 partial pilot-boundary cells
population total approximately 3,060,319.092
all grid cells assigned a valid SMOD settlement group
a binary World Mollweide polygon for every 1 km cell

The output will be stored at:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975

This cell does not reread either GeoTIFF. It joins only the two compact, validated HDFS Parquet products generated by Cells 7 and 8.

Excellent. Cell 9 succeeded fully, and the joined Bengaluru 1975 grid-year dataset is now complete and persistent in HDFS.

Final joined result
Metric	ResultGrid-year rows	3,658
Complete 1 km cells	3,480
Partial pilot-boundary cells	178
Urban cells	398
Urban-centre cells	202
Population total	3,060,319.092

The population and SMOD datasets had exact two-way key coverage, matching grid coordinates, and no unmatched cells. Population conservation also passed after the join.

1975 settlement pattern
Settlement group	Cells	Population	Average per cellRural	3,257	290,911.89	89.32
Urban cluster	196	290,050.98	1,479.85
Urban centre	202	2,479,356.22	12,274.04
Water	3	0	0

Approximately 81.0% of the pilot population was located in SMOD urban-centre cells in 1975, while urban-centre cells represented only about 5.5% of the pilot grid cells. This indicates strong population concentration in the historical urban core.

Persistent analytical output
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975

Each row now contains:

canonical grid_cell_id
1 km Mollweide coordinates and geometry
population total and 100 m statistics
population coverage and quality flag
numeric SMOD class
broader settlement group
urban and urban-centre flags
source lineage.
Next step

We should now repeat the same proven pipeline for Bengaluru 2020, not add built surface yet:

Plain Text
Population 2020
→ SMOD 2020
→ joined 2020 grid-year dataset
→ compare 1975 versus 2020

That end-to-end comparison will verify that identical grid_cell_id values persist across time and allow us to calculate:

population change;
SMOD class transitions;
rural-to-urban conversion;
growth of urban-centre cells;
outward expansion of populated cells.

After the 1975-to-2020 transition works, we should add built surface to both endpoint years. This gives us an early project result before processing every intermediate epoch.


In [19]:
# ============================================================
# CELL 10: BENGALURU 2020 POPULATION, 100 M TO 1 KM
#
# Pipeline:
#   HDFS catalog -> Spark shard discovery
#   -> HDFS GeoTIFF -> Sedona decode and tiling
#   -> Sedona band extraction
#   -> Spark SQL finite-value filtering
#   -> Spark SQL 100 m to 1 km aggregation
#   -> HDFS Parquet
#
# No hard-coded source filename.
# No Python raster UDF, NumPy, Rasterio, RS_Clip,
# or RS_AsGeoTiff.
# ============================================================

import base64
import textwrap


cell10_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )

    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=2020/"
        "grid_1km"
    )

    CITY = "bengaluru"
    PRODUCT = "population"
    EPOCH = 2020

    SOURCE_WIDTH = 8192
    SOURCE_HEIGHT = 8192

    SOURCE_RESOLUTION = 100.0
    TARGET_RESOLUTION = 1000.0

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    TILE_COLUMNS = SOURCE_WIDTH // TILE_WIDTH

    EXPECTED_SOURCE_TILES = 1024
    EXPECTED_SELECTED_TILES = 12
    EXPECTED_GRID_CELLS = 3658

    INVALID_SENTINEL = -200.0

    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-2020-"
                "Population-100m-to-1km"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 10: BENGALURU 2020 POPULATION")
        print("=" * 80)
        print("Catalog:", HDFS_CATALOG)
        print("Output:", OUTPUT_PATH)


        # ====================================================
        # DISCOVER THE 2020 POPULATION SHARD
        #
        # First filter the catalog to the 15 population files.
        # Then use the catalog envelopes and Sedona to select
        # the shard intersecting Bengaluru.
        # ====================================================

        catalog_df = (
            spark.read
            .parquet(HDFS_CATALOG)
            .filter(
                F.lower(F.col("product"))
                == F.lit(PRODUCT)
            )
            .filter(
                F.col("temporal_value").cast("int")
                == F.lit(EPOCH)
            )
        )

        catalog_count = catalog_df.count()

        if catalog_count != 15:
            raise RuntimeError(
                "Expected 15 population catalog records for "
                f"2020, found {catalog_count}."
            )

        catalog_df.createOrReplaceTempView(
            "population_2020_catalog"
        )

        selected_sources_df = spark.sql(
            f"""
            SELECT
                filename,
                path,
                fileSize,
                width,
                height,
                resolution_x,
                resolution_y,
                min_x,
                min_y,
                max_x,
                max_y

            FROM population_2020_catalog

            WHERE ST_Intersects(
                ST_PolygonFromEnvelope(
                    min_x,
                    min_y,
                    max_x,
                    max_y
                ),
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                )
            )

            ORDER BY filename
            """
        )

        selected_source_count = (
            selected_sources_df.count()
        )

        print(
            "2020 Bengaluru source shards:",
            selected_source_count,
        )

        selected_sources_df.show(
            10,
            truncate=False,
            vertical=True,
        )

        if selected_source_count != 1:
            raise RuntimeError(
                "Expected exactly one 2020 population shard "
                "for Bengaluru, found "
                f"{selected_source_count}."
            )

        selected_source = (
            selected_sources_df.first()
        )

        source_path = selected_source["path"]

        if (
            int(selected_source["width"])
            != SOURCE_WIDTH
            or int(selected_source["height"])
            != SOURCE_HEIGHT
        ):
            raise RuntimeError(
                "Unexpected 2020 population shard dimensions."
            )

        print("Selected source:", source_path)


        # ====================================================
        # READ AND DECODE THE SELECTED GEOTIFF
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(source_path)
            .select(
                "path",
                F.col("length").alias(
                    "source_file_bytes"
                ),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "population_source"
        )


        # ====================================================
        # VALIDATE SOURCE METADATA
        # ====================================================

        source_metadata = spark.sql(
            """
            SELECT
                RS_Width(source_raster)
                    AS source_width,
                RS_Height(source_raster)
                    AS source_height,
                RS_NumBands(source_raster)
                    AS source_band_count,
                RS_ScaleX(source_raster)
                    AS source_scale_x,
                RS_ScaleY(source_raster)
                    AS source_scale_y
            FROM population_source
            """
        ).first()

        if (
            int(source_metadata["source_width"])
            != SOURCE_WIDTH
            or int(source_metadata["source_height"])
            != SOURCE_HEIGHT
            or int(source_metadata["source_band_count"])
            != 1
        ):
            raise RuntimeError(
                "2020 source metadata validation failed."
            )

        if (
            abs(
                float(source_metadata["source_scale_x"])
                - SOURCE_RESOLUTION
            )
            > 1e-9
            or abs(
                abs(
                    float(
                        source_metadata["source_scale_y"]
                    )
                )
                - SOURCE_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "2020 population resolution is not 100 m."
            )

        print("Source metadata: PASS")


        # ====================================================
        # TILE THE SOURCE RASTER
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path AS source_path,

                tile_index,

                CAST(
                    FLOOR(
                        tile_index / {TILE_COLUMNS}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {TILE_COLUMNS}
                    AS INT
                ) AS tile_col,

                tile

            FROM population_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded
            AS tile_index, tile
            """
        )

        all_tile_count = tiles_df.count()

        if all_tile_count != EXPECTED_SOURCE_TILES:
            raise RuntimeError(
                "Expected 1024 source tiles, found "
                f"{all_tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "population_tiles"
        )


        # ====================================================
        # SELECT BENGALURU-INTERSECTING TILES
        # ====================================================

        selected_tiles_df = spark.sql(
            f"""
            SELECT
                source_path,
                tile_index,
                tile_row,
                tile_col,

                RS_Width(tile)
                    AS tile_width,

                RS_Height(tile)
                    AS tile_height,

                RS_UpperLeftX(tile)
                    AS tile_upper_left_x,

                RS_UpperLeftY(tile)
                    AS tile_upper_left_y,

                RS_ScaleX(tile)
                    AS tile_scale_x,

                RS_ScaleY(tile)
                    AS tile_scale_y,

                tile

            FROM population_tiles

            WHERE ST_Intersects(
                RS_Envelope(tile),
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                )
            )
            """
        )

        selected_tile_count = (
            selected_tiles_df.count()
        )

        if (
            selected_tile_count
            != EXPECTED_SELECTED_TILES
        ):
            raise RuntimeError(
                "Expected 12 selected tiles, found "
                f"{selected_tile_count}."
            )

        print(
            "Selected Bengaluru tiles:",
            selected_tile_count,
        )

        selected_tiles_df.createOrReplaceTempView(
            "selected_population_tiles"
        )


        # ====================================================
        # EXTRACT PIXELS WITH SEDONA
        # ====================================================

        pixel_df = spark.sql(
            """
            SELECT
                source_path,
                tile_index,
                tile_row,
                tile_col,

                tile_width,
                tile_height,

                tile_upper_left_x,
                tile_upper_left_y,

                tile_scale_x,
                tile_scale_y,

                pixel_index,

                CAST(
                    FLOOR(
                        pixel_index / tile_width
                    )
                    AS INT
                ) AS local_pixel_row,

                CAST(
                    pixel_index % tile_width
                    AS INT
                ) AS local_pixel_col,

                CAST(
                    population_value
                    AS DOUBLE
                ) AS population_value

            FROM selected_population_tiles

            LATERAL VIEW posexplode(
                RS_BandAsArray(
                    tile,
                    1
                )
            ) exploded
            AS pixel_index, population_value
            """
        )

        pixel_df.createOrReplaceTempView(
            "population_pixels"
        )


        # ====================================================
        # POSITION AND CLASSIFY PIXELS
        # ====================================================

        classified_df = spark.sql(
            f"""
            SELECT
                source_path,

                (
                    tile_upper_left_x
                    + (
                        local_pixel_col + 0.5
                    ) * tile_scale_x
                ) AS pixel_center_x,

                (
                    tile_upper_left_y
                    + (
                        local_pixel_row + 0.5
                    ) * tile_scale_y
                ) AS pixel_center_y,

                population_value,

                CASE
                    WHEN
                        population_value IS NOT NULL
                        AND NOT isnan(population_value)
                        AND ABS(population_value)
                            <= 1.7976931348623157E308
                        AND population_value
                            <> {INVALID_SENTINEL}
                        AND population_value >= 0.0
                    THEN true
                    ELSE false
                END AS is_valid_population

            FROM population_pixels
            """
        )

        classified_df.createOrReplaceTempView(
            "classified_population_pixels"
        )


        # ====================================================
        # SELECT VALID PILOT PIXELS
        # ====================================================

        valid_pixels_df = spark.sql(
            f"""
            SELECT
                source_path,

                pixel_center_x,
                pixel_center_y,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {TARGET_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_col,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {TARGET_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_row,

                population_value

            FROM classified_population_pixels

            WHERE
                pixel_center_x >= {PILOT_MIN_X}
                AND pixel_center_x < {PILOT_MAX_X}
                AND pixel_center_y >= {PILOT_MIN_Y}
                AND pixel_center_y < {PILOT_MAX_Y}
                AND is_valid_population
            """
        )

        valid_pixels_df.createOrReplaceTempView(
            "valid_population_pixels"
        )

        valid_pixel_count = (
            valid_pixels_df.count()
        )

        if valid_pixel_count <= 0:
            raise RuntimeError(
                "No valid 2020 pilot pixels were found."
            )

        print(
            "Valid 2020 pilot pixels:",
            valid_pixel_count,
        )


        # ====================================================
        # AGGREGATE 100 M PIXELS TO 1 KM
        # ====================================================

        population_1km_df = spark.sql(
            f"""
            SELECT
                '{CITY}' AS city,
                '{PRODUCT}' AS product,
                CAST({EPOCH} AS INT) AS epoch,

                grid_1km_col,
                grid_1km_row,

                CONCAT(
                    'MOLL_1KM_',
                    CAST(grid_1km_col AS STRING),
                    '_',
                    CAST(grid_1km_row AS STRING)
                ) AS grid_cell_id,

                CAST(
                    grid_1km_col
                    * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_x,

                CAST(
                    (
                        grid_1km_col + 1
                    ) * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_x,

                CAST(
                    grid_1km_row
                    * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_y,

                CAST(
                    (
                        grid_1km_row + 1
                    ) * {TARGET_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_y,

                COUNT(*) AS valid_100m_pixels,

                CAST(
                    COUNT(*) / 100.0
                    AS DOUBLE
                ) AS valid_coverage_fraction,

                SUM(
                    population_value
                ) AS population_total,

                AVG(
                    population_value
                ) AS population_mean_100m,

                MIN(
                    population_value
                ) AS population_min_100m,

                MAX(
                    population_value
                ) AS population_max_100m,

                SUM(
                    CASE
                        WHEN population_value = 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS zero_population_pixels,

                SUM(
                    CASE
                        WHEN population_value > 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS positive_population_pixels,

                MIN(source_path)
                    AS source_path,

                '100.0'
                    AS input_resolution_m,

                '1000.0'
                    AS output_resolution_m,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM valid_population_pixels

            GROUP BY
                grid_1km_col,
                grid_1km_row
            """
        )


        # ====================================================
        # VALIDATE 1 KM OUTPUT
        # ====================================================

        grid_cell_count = (
            population_1km_df.count()
        )

        if grid_cell_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 2020 grid-cell count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {grid_cell_count}."
            )


        duplicate_count = (
            population_1km_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "Duplicate 2020 grid cells were produced."
            )


        invalid_grid_count = (
            population_1km_df
            .filter(
                F.col("population_total").isNull()
                | F.isnan("population_total")
                | (
                    F.col("population_total")
                    < F.lit(0.0)
                )
                | (
                    F.col("valid_100m_pixels")
                    <= F.lit(0)
                )
                | (
                    F.col("valid_100m_pixels")
                    > F.lit(100)
                )
                | (
                    F.col(
                        "valid_coverage_fraction"
                    )
                    <= F.lit(0.0)
                )
                | (
                    F.col(
                        "valid_coverage_fraction"
                    )
                    > F.lit(1.0)
                )
            )
            .count()
        )

        if invalid_grid_count != 0:
            raise RuntimeError(
                "One or more 2020 grid cells failed "
                "validation."
            )


        # ====================================================
        # POPULATION CONSERVATION
        # ====================================================

        source_total = float(
            valid_pixels_df
            .agg(
                F.sum(
                    "population_value"
                ).alias("total")
            )
            .first()["total"]
        )

        aggregated_total = float(
            population_1km_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        conservation_difference = abs(
            source_total - aggregated_total
        )

        allowed_difference = max(
            1e-6,
            abs(source_total) * 1e-12,
        )

        if (
            conservation_difference
            > allowed_difference
        ):
            raise RuntimeError(
                "2020 population conservation failed. "
                f"Source={source_total}, "
                f"aggregated={aggregated_total}, "
                f"difference={conservation_difference}."
            )


        # ====================================================
        # CONFIRM GRID IDS MATCH 1975 POPULATION
        # ====================================================

        population_1975_path = (
            "hdfs://localhost:9000"
            "/data/ghsl/curated/pilots/bengaluru/"
            "product=population/"
            "epoch=1975/"
            "grid_1km"
        )

        population_1975_df = (
            spark.read
            .parquet(population_1975_path)
            .select("grid_cell_id")
        )

        population_2020_keys_df = (
            population_1km_df
            .select("grid_cell_id")
        )

        missing_from_2020 = (
            population_1975_df
            .join(
                population_2020_keys_df,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        new_in_2020 = (
            population_2020_keys_df
            .join(
                population_1975_df,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        print()
        print("=" * 80)
        print("1975-2020 GRID KEY COMPARISON")
        print("=" * 80)
        print(
            "1975 cells missing from 2020:",
            missing_from_2020,
        )
        print(
            "2020 cells absent from 1975:",
            new_in_2020,
        )

        if (
            missing_from_2020 != 0
            or new_in_2020 != 0
        ):
            raise RuntimeError(
                "The 2020 grid keys do not match 1975."
            )


        # ====================================================
        # DISPLAY SUMMARY
        # ====================================================

        coverage_summary_df = (
            population_1km_df
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),
                F.sum(
                    F.when(
                        F.col(
                            "valid_100m_pixels"
                        )
                        == F.lit(100),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "complete_cells"
                ),
                F.sum(
                    F.when(
                        F.col(
                            "valid_100m_pixels"
                        )
                        < F.lit(100),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "partial_cells"
                ),
                F.sum(
                    "population_total"
                ).alias(
                    "population_total"
                ),
                F.avg(
                    "population_total"
                ).alias(
                    "average_population_per_cell"
                ),
                F.max(
                    "population_total"
                ).alias(
                    "maximum_population_cell"
                ),
            )
        )

        print()
        print("=" * 80)
        print("2020 POPULATION SUMMARY")
        print("=" * 80)

        coverage_summary_df.show(
            truncate=False,
            vertical=True,
        )


        print()
        print("=" * 80)
        print("TOP 25 POPULATION CELLS")
        print("=" * 80)

        (
            population_1km_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "valid_100m_pixels",
                "valid_coverage_fraction",
                "population_total",
                "positive_population_pixels",
            )
            .orderBy(
                F.col(
                    "population_total"
                ).desc()
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE HDFS PARQUET
        # ====================================================

        (
            population_1km_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )

        print("HDFS 2020 population write: COMPLETE")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if readback_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "2020 population read-back count mismatch."
            )

        readback_total = float(
            readback_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        if (
            abs(
                readback_total
                - aggregated_total
            )
            > allowed_difference
        ):
            raise RuntimeError(
                "2020 population read-back total mismatch."
            )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 10 REMOTE RESULT")
        print("=" * 80)
        print("2020 source discovery: PASS")
        print("HDFS GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Sedona raster tiling: PASS")
        print("Sedona Bengaluru tile selection: PASS")
        print("Sedona band-array extraction: PASS")
        print("Spark SQL 100 m to 1 km aggregation: PASS")
        print("Population conservation: PASS")
        print("1975-2020 grid-key agreement: PASS")
        print("HDFS Parquet write: PASS")
        print("HDFS Parquet read-back: PASS")
        print("Output grid cells:", readback_count)
        print("Population total:", readback_total)
        print("Output:", OUTPUT_PATH)
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 10: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

encoded_program = base64.b64encode(
    cell10_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell10_population_2020.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 10 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 10: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 10: BENGALURU 2020 POPULATION
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=2020/grid_1km
2020 Bengaluru source shards: 1
-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------
 filename     | ghsl_india_population_2020_100m_r2023a-0000016384-0000008192.tif                                                                 
 path         | hdfs://localhost:9000/data/ghsl/raw/india/population/epoch=2020/ghsl_india_population_2020_100m_r2023a-0000016384-0000008192.tif 
 fileSize     | 99636178                                                                                                   

Cell 10 succeeded completely. The 2020 population dataset now matches the 1975 grid exactly, so the temporal comparison foundation is valid.

Cell 10 results
Metric	2020 resultGrid cells	3,658
Complete cells	3,480
Partial boundary cells	178
Valid pilot pixels	359,856
Population total	14,608,885.86
Average population per 1 km cell	3,993.68
Maximum population in one 1 km cell	76,534.22
1975 cells missing from 2020	0
2020 cells absent from 1975	0

The output passed source discovery, Sedona decoding and tiling, Spark SQL aggregation, population conservation, exact 1975–2020 key agreement, HDFS write, and HDFS read-back. No Python raster UDF, NumPy, or Rasterio was used.

Preliminary endpoint change

Population within the fixed Bengaluru pilot increased from approximately 3.06 million in 1975 to 14.61 million in 2020:

Absolute increase: approximately 11.55 million
Growth multiple: approximately 4.77×
Percentage increase: approximately 377.36%

This is a fixed-grid comparison because all 3,658 grid IDs match across the two years.

Persistent 2020 output
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=2020/grid_1km

In [20]:
# ============================================================
# CELL 11: BENGALURU 2020 SMOD 1 KM DATASET
#
# Pipeline:
#   HDFS metadata catalog
#   -> Spark identifies the 2020 SMOD source
#   -> Spark binaryFile reads the GeoTIFF
#   -> Sedona decodes and tiles the raster
#   -> Sedona RS_BandAsArray extracts SMOD values
#   -> Spark SQL selects Bengaluru cells
#   -> Spark validates against population 2020
#   -> Spark writes scalar Parquet to HDFS
#
# No NumPy, Rasterio, Python raster UDF, RS_Clip,
# or RS_AsGeoTiff.
# ============================================================

import base64
import textwrap


cell11_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )

    POPULATION_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=2020/"
        "grid_1km"
    )

    SMOD_1975_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/"
        "epoch=1975/"
        "grid_1km"
    )

    SMOD_OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/"
        "epoch=2020/"
        "grid_1km"
    )

    CITY = "bengaluru"
    PRODUCT = "smod"
    EPOCH = 2020

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    EXPECTED_RESOLUTION = 1000.0
    EXPECTED_POPULATION_CELLS = 3658

    # Verified World Mollweide Bengaluru pilot envelope.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-2020-SMOD-1km"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 11: BENGALURU 2020 SMOD 1 KM")
        print("=" * 80)
        print("Catalog:", HDFS_CATALOG)
        print("Population reference:", POPULATION_PATH)
        print("1975 SMOD reference:", SMOD_1975_PATH)
        print("2020 SMOD output:", SMOD_OUTPUT_PATH)


        # ====================================================
        # DISCOVER THE 2020 SMOD SOURCE FROM THE CATALOG
        # ====================================================

        catalog_df = spark.read.parquet(
            HDFS_CATALOG
        )

        smod_catalog_df = (
            catalog_df
            .filter(
                F.lower(
                    F.col("product")
                )
                == F.lit(PRODUCT)
            )
            .filter(
                F.col(
                    "temporal_value"
                ).cast("int")
                == F.lit(EPOCH)
            )
            .select(
                "filename",
                "path",
                "fileSize",
                "width",
                "height",
                "resolution_x",
                "resolution_y",
                "min_x",
                "min_y",
                "max_x",
                "max_y",
            )
        )

        smod_catalog_count = (
            smod_catalog_df.count()
        )

        print(
            "2020 SMOD catalog records:",
            smod_catalog_count,
        )

        if smod_catalog_count != 1:
            print("Matching catalog records:")

            smod_catalog_df.show(
                20,
                truncate=False,
                vertical=True,
            )

            raise RuntimeError(
                "Expected exactly one 2020 SMOD source, "
                f"found {smod_catalog_count}."
            )

        smod_catalog_row = (
            smod_catalog_df.first()
        )

        smod_source_path = (
            smod_catalog_row["path"]
        )

        print(
            "SMOD filename:",
            smod_catalog_row["filename"],
        )

        print(
            "SMOD source:",
            smod_source_path,
        )

        print(
            "Catalog dimensions:",
            smod_catalog_row["width"],
            "x",
            smod_catalog_row["height"],
        )

        print(
            "Catalog resolution:",
            smod_catalog_row["resolution_x"],
            "x",
            smod_catalog_row["resolution_y"],
        )


        # ====================================================
        # READ AND DECODE THE SMOD GEOTIFF FROM HDFS
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(smod_source_path)
            .select(
                "path",

                F.col("length").alias(
                    "source_file_bytes"
                ),

                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias(
                    "source_raster"
                ),
            )
        )

        source_count = source_df.count()

        if source_count != 1:
            raise RuntimeError(
                "Expected exactly one SMOD GeoTIFF, "
                f"found {source_count}."
            )

        source_df.createOrReplaceTempView(
            "smod_source"
        )


        # ====================================================
        # VALIDATE SOURCE RASTER METADATA
        # ====================================================

        source_metadata_df = spark.sql(
            """
            SELECT
                path,
                source_file_bytes,

                RS_Width(
                    source_raster
                ) AS source_width,

                RS_Height(
                    source_raster
                ) AS source_height,

                RS_NumBands(
                    source_raster
                ) AS source_band_count,

                RS_ScaleX(
                    source_raster
                ) AS source_scale_x,

                RS_ScaleY(
                    source_raster
                ) AS source_scale_y,

                RS_UpperLeftX(
                    source_raster
                ) AS source_upper_left_x,

                RS_UpperLeftY(
                    source_raster
                ) AS source_upper_left_y

            FROM smod_source
            """
        )

        source_metadata = (
            source_metadata_df.first()
        )

        source_width = int(
            source_metadata["source_width"]
        )

        source_height = int(
            source_metadata["source_height"]
        )

        source_band_count = int(
            source_metadata["source_band_count"]
        )

        source_scale_x = float(
            source_metadata["source_scale_x"]
        )

        source_scale_y = float(
            source_metadata["source_scale_y"]
        )

        if source_width <= 0:
            raise RuntimeError(
                "SMOD source width is not positive."
            )

        if source_height <= 0:
            raise RuntimeError(
                "SMOD source height is not positive."
            )

        if source_band_count != 1:
            raise RuntimeError(
                "Expected one SMOD raster band."
            )

        if (
            abs(
                source_scale_x
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected SMOD X resolution: "
                f"{source_scale_x}"
            )

        if (
            abs(
                abs(source_scale_y)
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Unexpected SMOD Y resolution: "
                f"{source_scale_y}"
            )

        if (
            int(smod_catalog_row["width"])
            != source_width
        ):
            raise RuntimeError(
                "Catalog and decoded SMOD widths differ."
            )

        if (
            int(smod_catalog_row["height"])
            != source_height
        ):
            raise RuntimeError(
                "Catalog and decoded SMOD heights differ."
            )

        print()
        print("=" * 80)
        print("SMOD SOURCE METADATA")
        print("=" * 80)

        source_metadata_df.show(
            truncate=False,
            vertical=True,
        )

        print("SMOD source metadata: PASS")


        # ====================================================
        # DETERMINE TILE LAYOUT
        # ====================================================

        tile_columns = (
            source_width + TILE_WIDTH - 1
        ) // TILE_WIDTH

        tile_rows = (
            source_height + TILE_HEIGHT - 1
        ) // TILE_HEIGHT

        expected_tile_count = (
            tile_columns * tile_rows
        )

        print(
            "SMOD tile layout:",
            f"{tile_columns} x {tile_rows}",
        )

        print(
            "Expected SMOD tiles:",
            expected_tile_count,
        )


        # ====================================================
        # TILE THE SMOD RASTER WITH SEDONA
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path AS source_path,
                source_file_bytes,

                tile_index,

                CAST(
                    FLOOR(
                        tile_index / {tile_columns}
                    )
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {tile_columns}
                    AS INT
                ) AS tile_col,

                tile

            FROM smod_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded
            AS tile_index, tile
            """
        )

        tile_count = tiles_df.count()

        if tile_count != expected_tile_count:
            raise RuntimeError(
                "Unexpected SMOD tile count. "
                f"Expected {expected_tile_count}, "
                f"found {tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "smod_tiles"
        )

        print(
            "Sedona SMOD tile count:",
            tile_count,
        )


        # ====================================================
        # SELECT ONLY TILES INTERSECTING BENGALURU
        # ====================================================

        pilot_df = spark.sql(
            f"""
            SELECT
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                ) AS pilot_envelope
            """
        )

        pilot_df.createOrReplaceTempView(
            "bengaluru_pilot"
        )

        selected_tiles_df = spark.sql(
            """
            SELECT
                t.source_path,
                t.source_file_bytes,

                t.tile_index,
                t.tile_row,
                t.tile_col,

                RS_Width(
                    t.tile
                ) AS tile_width,

                RS_Height(
                    t.tile
                ) AS tile_height,

                RS_UpperLeftX(
                    t.tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    t.tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    t.tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    t.tile
                ) AS tile_scale_y,

                t.tile

            FROM smod_tiles AS t

            CROSS JOIN bengaluru_pilot AS b

            WHERE ST_Intersects(
                RS_Envelope(t.tile),
                b.pilot_envelope
            )
            """
        )

        selected_tile_count = (
            selected_tiles_df.count()
        )

        if selected_tile_count < 1:
            raise RuntimeError(
                "No 2020 SMOD tile intersects Bengaluru."
            )

        if selected_tile_count > 4:
            raise RuntimeError(
                "Unexpectedly large number of Bengaluru "
                f"SMOD tiles: {selected_tile_count}."
            )

        print(
            "Selected Bengaluru SMOD tiles:",
            selected_tile_count,
        )

        selected_tiles_df.createOrReplaceTempView(
            "selected_smod_tiles"
        )


        # ====================================================
        # EXTRACT SMOD PIXEL VALUES WITH SEDONA
        # ====================================================

        pixel_df = spark.sql(
            """
            SELECT
                source_path,

                tile_index,
                tile_row,
                tile_col,

                tile_width,
                tile_height,

                tile_upper_left_x,
                tile_upper_left_y,

                tile_scale_x,
                tile_scale_y,

                pixel_index,

                CAST(
                    FLOOR(
                        pixel_index / tile_width
                    )
                    AS INT
                ) AS local_pixel_row,

                CAST(
                    pixel_index % tile_width
                    AS INT
                ) AS local_pixel_col,

                CAST(
                    smod_value
                    AS DOUBLE
                ) AS smod_value

            FROM selected_smod_tiles

            LATERAL VIEW posexplode(
                RS_BandAsArray(
                    tile,
                    1
                )
            ) exploded
            AS pixel_index, smod_value
            """
        )

        pixel_df.createOrReplaceTempView(
            "smod_pixels"
        )


        # ====================================================
        # POSITION EACH NATIVE 1 KM SMOD PIXEL
        # ====================================================

        positioned_df = spark.sql(
            """
            SELECT
                source_path,

                tile_index,
                tile_row,
                tile_col,

                pixel_index,
                local_pixel_row,
                local_pixel_col,

                (
                    tile_upper_left_x
                    + (
                        local_pixel_col + 0.5
                    ) * tile_scale_x
                ) AS pixel_center_x,

                (
                    tile_upper_left_y
                    + (
                        local_pixel_row + 0.5
                    ) * tile_scale_y
                ) AS pixel_center_y,

                smod_value

            FROM smod_pixels
            """
        )

        positioned_df.createOrReplaceTempView(
            "positioned_smod_pixels"
        )


        # ====================================================
        # CREATE THE BENGALURU 2020 SMOD GRID
        #
        # The same World Mollweide grid identifier convention
        # is used by population 1975, SMOD 1975 and population
        # 2020.
        # ====================================================

        smod_grid_df = spark.sql(
            f"""
            SELECT
                '{CITY}' AS city,
                '{PRODUCT}' AS product,
                CAST({EPOCH} AS INT) AS epoch,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {EXPECTED_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_col,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {EXPECTED_RESOLUTION}
                    )
                    AS BIGINT
                ) AS grid_1km_row,

                CONCAT(
                    'MOLL_1KM_',

                    CAST(
                        FLOOR(
                            pixel_center_x
                            / {EXPECTED_RESOLUTION}
                        )
                        AS STRING
                    ),

                    '_',

                    CAST(
                        FLOOR(
                            pixel_center_y
                            / {EXPECTED_RESOLUTION}
                        )
                        AS STRING
                    )
                ) AS grid_cell_id,

                CAST(
                    smod_value
                    AS INT
                ) AS smod_class,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {EXPECTED_RESOLUTION}
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_x,

                CAST(
                    (
                        FLOOR(
                            pixel_center_x
                            / {EXPECTED_RESOLUTION}
                        ) + 1
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_x,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {EXPECTED_RESOLUTION}
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_min_y,

                CAST(
                    (
                        FLOOR(
                            pixel_center_y
                            / {EXPECTED_RESOLUTION}
                        ) + 1
                    )
                    * {EXPECTED_RESOLUTION}
                    AS DOUBLE
                ) AS grid_max_y,

                source_path,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM positioned_smod_pixels

            WHERE
                pixel_center_x >= {PILOT_MIN_X}
                AND pixel_center_x < {PILOT_MAX_X}

                AND pixel_center_y >= {PILOT_MIN_Y}
                AND pixel_center_y < {PILOT_MAX_Y}

                AND smod_value IS NOT NULL
                AND NOT isnan(smod_value)

                AND ABS(smod_value)
                    <= 1.7976931348623157E308

                AND smod_value >= 0.0
            """
        )


        # ====================================================
        # VALIDATE THE 2020 SMOD GRID
        # ====================================================

        smod_grid_count = (
            smod_grid_df.count()
        )

        if smod_grid_count != EXPECTED_POPULATION_CELLS:
            raise RuntimeError(
                "Unexpected 2020 SMOD pilot cell count. "
                f"Expected {EXPECTED_POPULATION_CELLS}, "
                f"found {smod_grid_count}."
            )

        duplicate_count = (
            smod_grid_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "Duplicate 2020 SMOD grid cells were "
                "produced."
            )

        invalid_class_count = (
            smod_grid_df
            .filter(
                F.col("smod_class").isNull()
                | (
                    F.col("smod_class")
                    < F.lit(0)
                )
            )
            .count()
        )

        if invalid_class_count != 0:
            raise RuntimeError(
                "One or more invalid 2020 SMOD classes "
                "were produced."
            )


        # ====================================================
        # DISPLAY 2020 SMOD CLASS DISTRIBUTION
        # ====================================================

        print()
        print("=" * 80)
        print("2020 SMOD CLASS DISTRIBUTION")
        print("=" * 80)

        (
            smod_grid_df
            .groupBy(
                "smod_class"
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                )
            )
            .orderBy(
                "smod_class"
            )
            .show(
                100,
                truncate=False,
            )
        )


        # ====================================================
        # VALIDATE COVERAGE AGAINST POPULATION 2020
        # ====================================================

        population_df = (
            spark.read
            .parquet(POPULATION_PATH)
        )

        population_count = population_df.count()

        if (
            population_count
            != EXPECTED_POPULATION_CELLS
        ):
            raise RuntimeError(
                "Unexpected 2020 population grid count. "
                f"Expected {EXPECTED_POPULATION_CELLS}, "
                f"found {population_count}."
            )

        population_duplicate_count = (
            population_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if population_duplicate_count != 0:
            raise RuntimeError(
                "The 2020 population reference contains "
                "duplicate grid cells."
            )

        overlap_count = (
            population_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                smod_grid_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="inner",
            )
            .count()
        )

        population_without_smod = (
            population_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                smod_grid_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )

        smod_without_population = (
            smod_grid_df
            .select(
                "grid_cell_id",
                "epoch",
            )
            .join(
                population_df.select(
                    "grid_cell_id",
                    "epoch",
                ),
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )

        print()
        print("=" * 80)
        print("2020 POPULATION AND SMOD COVERAGE")
        print("=" * 80)

        print(
            "Population grid cells:",
            population_count,
        )

        print(
            "SMOD pilot cells:",
            smod_grid_count,
        )

        print(
            "Population-SMOD overlapping cells:",
            overlap_count,
        )

        print(
            "Population cells without SMOD:",
            population_without_smod,
        )

        print(
            "SMOD cells without population:",
            smod_without_population,
        )

        if population_without_smod != 0:
            raise RuntimeError(
                "One or more 2020 population cells do not "
                "have a corresponding SMOD cell."
            )

        if smod_without_population != 0:
            raise RuntimeError(
                "One or more 2020 SMOD cells do not have "
                "a corresponding population cell."
            )


        # ====================================================
        # VALIDATE GRID IDS AGAINST SMOD 1975
        #
        # Classes may change, but grid identifiers must not.
        # ====================================================

        smod_1975_df = (
            spark.read
            .parquet(SMOD_1975_PATH)
            .select(
                "grid_cell_id"
            )
        )

        smod_2020_keys_df = (
            smod_grid_df
            .select(
                "grid_cell_id"
            )
        )

        missing_from_2020 = (
            smod_1975_df
            .join(
                smod_2020_keys_df,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        new_in_2020 = (
            smod_2020_keys_df
            .join(
                smod_1975_df,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        print()
        print("=" * 80)
        print("1975-2020 SMOD GRID KEY COMPARISON")
        print("=" * 80)

        print(
            "1975 SMOD cells missing from 2020:",
            missing_from_2020,
        )

        print(
            "2020 SMOD cells absent from 1975:",
            new_in_2020,
        )

        if missing_from_2020 != 0:
            raise RuntimeError(
                "The 2020 SMOD grid is missing one or more "
                "1975 grid identifiers."
            )

        if new_in_2020 != 0:
            raise RuntimeError(
                "The 2020 SMOD grid contains identifiers "
                "absent from 1975."
            )


        # ====================================================
        # WRITE SCALAR SMOD PARQUET TO HDFS
        # ====================================================

        (
            smod_grid_df
            .repartition(2)
            .write
            .mode("overwrite")
            .parquet(SMOD_OUTPUT_PATH)
        )

        print(
            "HDFS 2020 SMOD Parquet write: COMPLETE"
        )


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            SMOD_OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if (
            readback_count
            != EXPECTED_POPULATION_CELLS
        ):
            raise RuntimeError(
                "2020 SMOD read-back row count mismatch. "
                f"Expected {EXPECTED_POPULATION_CELLS}, "
                f"found {readback_count}."
            )

        readback_duplicate_count = (
            readback_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if readback_duplicate_count != 0:
            raise RuntimeError(
                "2020 SMOD read-back contains duplicate "
                "grid cells."
            )

        readback_invalid_count = (
            readback_df
            .filter(
                F.col("grid_cell_id").isNull()
                | F.col("smod_class").isNull()
                | F.col("source_path").isNull()
            )
            .count()
        )

        if readback_invalid_count != 0:
            raise RuntimeError(
                "One or more 2020 SMOD read-back rows "
                "failed required-field validation."
            )


        # ====================================================
        # DISPLAY SAMPLE RECORDS
        # ====================================================

        print()
        print("=" * 80)
        print("SAMPLE 2020 SMOD RECORDS")
        print("=" * 80)

        (
            readback_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "smod_class",
                "grid_min_x",
                "grid_min_y",
            )
            .orderBy(
                "grid_1km_row",
                "grid_1km_col",
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 11 REMOTE RESULT")
        print("=" * 80)

        print(
            "2020 SMOD source discovery from catalog: PASS"
        )

        print(
            "HDFS 2020 SMOD GeoTIFF read: PASS"
        )

        print(
            "Sedona raster decoding: PASS"
        )

        print(
            "Sedona SMOD tiling: PASS"
        )

        print(
            "Sedona band-array extraction: PASS"
        )

        print(
            "Spark SQL Bengaluru filtering: PASS"
        )

        print(
            "Spark SQL grid identifier creation: PASS"
        )

        print(
            "2020 population-SMOD coverage: PASS"
        )

        print(
            "1975-2020 SMOD grid-key agreement: PASS"
        )

        print(
            "HDFS 2020 SMOD Parquet write: PASS"
        )

        print(
            "HDFS 2020 SMOD Parquet read-back: PASS"
        )

        print(
            "SMOD output cells:",
            readback_count,
        )

        print(
            "Population cells covered by SMOD:",
            overlap_count,
        )

        print(
            "Output:",
            SMOD_OUTPUT_PATH,
        )

        print(
            "Python raster UDF used: NO"
        )

        print(
            "NumPy used: NO"
        )

        print(
            "Rasterio used: NO"
        )

        print(
            "RS_Clip used: NO"
        )

        print(
            "RS_AsGeoTiff used: NO"
        )

        print(
            "CELL 11: SUCCESS"
        )

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell11_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell11_smod_2020.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "

    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "

    "--master local[1] "

    "--driver-memory 3g "

    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 11 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Remote return code:",
    result.returncode,
)
print("CELL 11: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 11: BENGALURU 2020 SMOD 1 KM
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Population reference: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=2020/grid_1km
1975 SMOD reference: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=1975/grid_1km
2020 SMOD output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=2020/grid_1km
2020 SMOD catalog records: 1
SMOD filename: ghsl_india_smod_2020_1000m_r2023a.tif
SMOD source: hdfs://localhost:9000/data/ghsl/raw/india/smod/epoch=2020/ghsl_india_smod_2020_1000m_r2023a.tif
Catalog dimensions: 2902 x 3179
Catalog resolution: 1000.0 x 1000.0

SMOD SOURCE METADATA
-RECORD 0-----------------------------------------------------------------------

Cell 11 succeeded completely. The 2020 SMOD dataset is now persistent, and its grid aligns exactly with both 2020 population and 1975 SMOD.

Key results
SMOD source: ghsl_india_smod_2020_1000m_r2023a.tif
Source dimensions: 2,902 × 3,179
Resolution: 1,000 m
Source tiles: 156
Bengaluru tiles selected: 1
Output cells: 3,658
Population cells without SMOD: 0
SMOD cells without population: 0
1975 grid IDs missing from 2020: 0
New 2020 grid IDs: 0
2020 SMOD distribution
SMOD class	Cells10	2
11	611
12	980
13	54
21	971
22	32
23	58
30	950

The urban-centre class increased from 202 cells in 1975 to 950 cells in 2020, while class 11 decreased from 1,931 to 611 cells. This strongly indicates substantial outward urban expansion, although the final interpretation should use cell-level transition counts rather than only comparing marginal totals.

Persistent output
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=2020/grid_1km

Next step

Cell 12 should join:

Plain Text
population 2020 + SMOD 2020

into:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=2020

After that, Cell 13 can compare the two complete endpoint datasets:

Plain Text
grid_year/epoch=1975
grid_year/epoch=2020

and calculate the true cell-level population change and settlement transitions.

In [21]:
# ============================================================
# CELL 12: JOIN BENGALURU 2020 POPULATION + SMOD
#
# Inputs:
#   - Population 2020 at 1 km
#   - SMOD 2020 at 1 km
#
# Output:
#   Bengaluru 2020 grid-year analytical dataset
#
# Tools:
#   - HDFS: persistent inputs and output
#   - Spark DataFrames / Spark SQL: validation and join
#   - Sedona: 1 km cell geometry creation
#
# No source GeoTIFF is reread.
# No NumPy, Rasterio, or Python UDF is used.
# ============================================================

import base64
import textwrap


cell12_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    POPULATION_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=population/"
        "epoch=2020/"
        "grid_1km"
    )

    SMOD_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=smod/"
        "epoch=2020/"
        "grid_1km"
    )

    GRID_YEAR_1975_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "grid_year/"
        "epoch=1975"
    )

    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "grid_year/"
        "epoch=2020"
    )

    CITY = "bengaluru"
    EPOCH = 2020

    EXPECTED_GRID_CELLS = 3658
    COMPLETE_CELL_PIXELS = 100
    GRID_RESOLUTION_METERS = 1000.0


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-2020-Grid-Year-Join"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 12: BENGALURU 2020 GRID-YEAR JOIN")
        print("=" * 80)
        print("Population input:", POPULATION_PATH)
        print("SMOD input:", SMOD_PATH)
        print("1975 grid-year reference:", GRID_YEAR_1975_PATH)
        print("Output:", OUTPUT_PATH)


        # ====================================================
        # READ PERSISTENT PARQUET INPUTS
        # ====================================================

        population_raw_df = spark.read.parquet(
            POPULATION_PATH
        )

        smod_raw_df = spark.read.parquet(
            SMOD_PATH
        )

        grid_year_1975_df = spark.read.parquet(
            GRID_YEAR_1975_PATH
        )


        # ====================================================
        # SELECT AND NORMALIZE POPULATION COLUMNS
        # ====================================================

        population_df = (
            population_raw_df
            .select(
                F.col("grid_cell_id")
                .cast("string")
                .alias("grid_cell_id"),

                F.col("epoch")
                .cast("int")
                .alias("epoch"),

                F.col("grid_1km_col")
                .cast("long")
                .alias("population_grid_col"),

                F.col("grid_1km_row")
                .cast("long")
                .alias("population_grid_row"),

                F.col("grid_min_x")
                .cast("double")
                .alias("population_grid_min_x"),

                F.col("grid_max_x")
                .cast("double")
                .alias("population_grid_max_x"),

                F.col("grid_min_y")
                .cast("double")
                .alias("population_grid_min_y"),

                F.col("grid_max_y")
                .cast("double")
                .alias("population_grid_max_y"),

                F.col("valid_100m_pixels")
                .cast("long")
                .alias("valid_100m_pixels"),

                F.col("valid_coverage_fraction")
                .cast("double")
                .alias(
                    "population_coverage_fraction"
                ),

                F.col("population_total")
                .cast("double")
                .alias("population_total"),

                F.col("population_mean_100m")
                .cast("double")
                .alias("population_mean_100m"),

                F.col("population_min_100m")
                .cast("double")
                .alias("population_min_100m"),

                F.col("population_max_100m")
                .cast("double")
                .alias("population_max_100m"),

                F.col("zero_population_pixels")
                .cast("long")
                .alias("zero_population_pixels"),

                F.col("positive_population_pixels")
                .cast("long")
                .alias("positive_population_pixels"),

                F.col("source_path")
                .alias("population_source_path"),
            )
        )


        # ====================================================
        # SELECT AND NORMALIZE SMOD COLUMNS
        # ====================================================

        smod_df = (
            smod_raw_df
            .select(
                F.col("grid_cell_id")
                .cast("string")
                .alias("grid_cell_id"),

                F.col("epoch")
                .cast("int")
                .alias("epoch"),

                F.col("grid_1km_col")
                .cast("long")
                .alias("smod_grid_col"),

                F.col("grid_1km_row")
                .cast("long")
                .alias("smod_grid_row"),

                F.col("grid_min_x")
                .cast("double")
                .alias("smod_grid_min_x"),

                F.col("grid_max_x")
                .cast("double")
                .alias("smod_grid_max_x"),

                F.col("grid_min_y")
                .cast("double")
                .alias("smod_grid_min_y"),

                F.col("grid_max_y")
                .cast("double")
                .alias("smod_grid_max_y"),

                F.col("smod_class")
                .cast("int")
                .alias("smod_class"),

                F.col("source_path")
                .alias("smod_source_path"),
            )
        )


        # ====================================================
        # BASIC INPUT VALIDATION
        # ====================================================

        population_count = population_df.count()
        smod_count = smod_df.count()
        reference_1975_count = (
            grid_year_1975_df.count()
        )

        print()
        print("=" * 80)
        print("INPUT ROW COUNTS")
        print("=" * 80)
        print("Population 2020 rows:", population_count)
        print("SMOD 2020 rows:", smod_count)
        print(
            "Grid-year 1975 reference rows:",
            reference_1975_count,
        )

        if population_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected population row count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {population_count}."
            )

        if smod_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected SMOD row count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {smod_count}."
            )

        if reference_1975_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 1975 grid-year row count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {reference_1975_count}."
            )


        def duplicate_key_count(dataframe):
            return (
                dataframe
                .groupBy(
                    "grid_cell_id",
                    "epoch",
                )
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )


        if duplicate_key_count(population_df) != 0:
            raise RuntimeError(
                "Population 2020 contains duplicate keys."
            )

        if duplicate_key_count(smod_df) != 0:
            raise RuntimeError(
                "SMOD 2020 contains duplicate keys."
            )


        invalid_population_epoch_count = (
            population_df
            .filter(
                F.col("epoch") != F.lit(EPOCH)
            )
            .count()
        )

        invalid_smod_epoch_count = (
            smod_df
            .filter(
                F.col("epoch") != F.lit(EPOCH)
            )
            .count()
        )

        if (
            invalid_population_epoch_count != 0
            or invalid_smod_epoch_count != 0
        ):
            raise RuntimeError(
                "One or more inputs contain the wrong epoch."
            )


        # ====================================================
        # VERIFY TWO-WAY POPULATION-SMOD KEY COVERAGE
        # ====================================================

        population_keys_df = population_df.select(
            "grid_cell_id",
            "epoch",
        )

        smod_keys_df = smod_df.select(
            "grid_cell_id",
            "epoch",
        )

        population_without_smod = (
            population_keys_df
            .join(
                smod_keys_df,
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )

        smod_without_population = (
            smod_keys_df
            .join(
                population_keys_df,
                on=[
                    "grid_cell_id",
                    "epoch",
                ],
                how="left_anti",
            )
            .count()
        )

        print()
        print("=" * 80)
        print("2020 TWO-WAY KEY COVERAGE")
        print("=" * 80)
        print(
            "Population cells without SMOD:",
            population_without_smod,
        )
        print(
            "SMOD cells without population:",
            smod_without_population,
        )

        if population_without_smod != 0:
            raise RuntimeError(
                "Population 2020 contains cells without SMOD."
            )

        if smod_without_population != 0:
            raise RuntimeError(
                "SMOD 2020 contains cells without population."
            )


        # ====================================================
        # STRICT ONE-TO-ONE JOIN
        # ====================================================

        joined_raw_df = (
            population_df.alias("p")
            .join(
                smod_df.alias("s"),
                on=[
                    F.col("p.grid_cell_id")
                    == F.col("s.grid_cell_id"),

                    F.col("p.epoch")
                    == F.col("s.epoch"),
                ],
                how="inner",
            )
        )

        joined_count = joined_raw_df.count()

        if joined_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected joined row count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {joined_count}."
            )


        # ====================================================
        # VERIFY GRID COORDINATES AND BOUNDS AGREE
        # ====================================================

        coordinate_mismatch_count = (
            joined_raw_df
            .filter(
                (
                    F.col("p.population_grid_col")
                    != F.col("s.smod_grid_col")
                )
                | (
                    F.col("p.population_grid_row")
                    != F.col("s.smod_grid_row")
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_min_x"
                        )
                        - F.col(
                            "s.smod_grid_min_x"
                        )
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_max_x"
                        )
                        - F.col(
                            "s.smod_grid_max_x"
                        )
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_min_y"
                        )
                        - F.col(
                            "s.smod_grid_min_y"
                        )
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col(
                            "p.population_grid_max_y"
                        )
                        - F.col(
                            "s.smod_grid_max_y"
                        )
                    )
                    > F.lit(1e-9)
                )
            )
            .count()
        )

        if coordinate_mismatch_count != 0:
            raise RuntimeError(
                "Population and SMOD coordinates disagree "
                f"for {coordinate_mismatch_count} cells."
            )

        print("Grid coordinate agreement: PASS")


        # ====================================================
        # BUILD 2020 GRID-YEAR ANALYTICAL RECORDS
        # ====================================================

        grid_year_scalar_df = (
            joined_raw_df
            .select(
                F.lit(CITY).alias("city"),

                F.col("p.grid_cell_id")
                .alias("grid_cell_id"),

                F.col("p.epoch")
                .alias("epoch"),

                F.col("p.population_grid_col")
                .alias("grid_1km_col"),

                F.col("p.population_grid_row")
                .alias("grid_1km_row"),

                F.col("p.population_grid_min_x")
                .alias("grid_min_x"),

                F.col("p.population_grid_max_x")
                .alias("grid_max_x"),

                F.col("p.population_grid_min_y")
                .alias("grid_min_y"),

                F.col("p.population_grid_max_y")
                .alias("grid_max_y"),

                F.col("p.valid_100m_pixels")
                .alias("valid_100m_pixels"),

                F.col(
                    "p.population_coverage_fraction"
                )
                .alias(
                    "population_coverage_fraction"
                ),

                F.col("p.population_total")
                .alias("population_total"),

                F.col("p.population_mean_100m")
                .alias("population_mean_100m"),

                F.col("p.population_min_100m")
                .alias("population_min_100m"),

                F.col("p.population_max_100m")
                .alias("population_max_100m"),

                F.col("p.zero_population_pixels")
                .alias("zero_population_pixels"),

                F.col("p.positive_population_pixels")
                .alias("positive_population_pixels"),

                F.col("s.smod_class")
                .alias("smod_class"),

                F.when(
                    F.col("s.smod_class")
                    == F.lit(10),
                    F.lit("water"),
                )
                .when(
                    F.col("s.smod_class").between(
                        11,
                        13,
                    ),
                    F.lit("rural"),
                )
                .when(
                    F.col("s.smod_class").between(
                        21,
                        23,
                    ),
                    F.lit("urban_cluster"),
                )
                .when(
                    F.col("s.smod_class")
                    == F.lit(30),
                    F.lit("urban_centre"),
                )
                .otherwise(
                    F.lit("other_or_unknown")
                )
                .alias("settlement_group"),

                F.when(
                    F.col("s.smod_class").isin(
                        21,
                        22,
                        23,
                        30,
                    ),
                    F.lit(True),
                )
                .otherwise(
                    F.lit(False)
                )
                .alias("is_urban"),

                F.when(
                    F.col("s.smod_class")
                    == F.lit(30),
                    F.lit(True),
                )
                .otherwise(
                    F.lit(False)
                )
                .alias("is_urban_centre"),

                F.when(
                    F.col("p.valid_100m_pixels")
                    == F.lit(
                        COMPLETE_CELL_PIXELS
                    ),
                    F.lit(True),
                )
                .otherwise(
                    F.lit(False)
                )
                .alias(
                    "is_complete_population_cell"
                ),

                F.when(
                    (
                        F.col(
                            "p.valid_100m_pixels"
                        )
                        == F.lit(
                            COMPLETE_CELL_PIXELS
                        )
                    )
                    & (
                        F.abs(
                            F.col(
                                "p.population_coverage_fraction"
                            )
                            - F.lit(1.0)
                        )
                        <= F.lit(1e-12)
                    ),
                    F.lit("complete"),
                )
                .otherwise(
                    F.lit(
                        "partial_pilot_boundary"
                    )
                )
                .alias(
                    "population_quality_flag"
                ),

                F.col("p.population_source_path")
                .alias("population_source_path"),

                F.col("s.smod_source_path")
                .alias("smod_source_path"),
            )
        )


        # ====================================================
        # CREATE 1 KM GEOMETRY WITH SEDONA
        #
        # WKB is stored because binary geometry is reliable in
        # Parquet and can be reconstructed by Sedona later.
        # ====================================================

        grid_year_scalar_df.createOrReplaceTempView(
            "grid_year_2020_scalar"
        )

        grid_year_df = spark.sql(
            f"""
            SELECT
                *,

                ST_AsBinary(
                    ST_PolygonFromEnvelope(
                        grid_min_x,
                        grid_min_y,
                        grid_max_x,
                        grid_max_y
                    )
                ) AS geometry_wkb,

                'ESRI:54009' AS geometry_crs,

                CAST(
                    {GRID_RESOLUTION_METERS}
                    AS DOUBLE
                ) AS grid_resolution_m,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM grid_year_2020_scalar
            """
        )


        # ====================================================
        # VALIDATE OUTPUT FIELDS
        # ====================================================

        output_count = grid_year_df.count()

        if output_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected output count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {output_count}."
            )

        output_duplicate_count = (
            grid_year_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if output_duplicate_count != 0:
            raise RuntimeError(
                "The 2020 grid-year output contains "
                "duplicate keys."
            )

        invalid_output_count = (
            grid_year_df
            .filter(
                F.col("grid_cell_id").isNull()
                | F.col("population_total").isNull()
                | F.isnan("population_total")
                | (
                    F.col("population_total")
                    < F.lit(0.0)
                )
                | F.col("smod_class").isNull()
                | F.col("settlement_group").isNull()
                | F.col("geometry_wkb").isNull()
                | (
                    F.length("geometry_wkb")
                    <= F.lit(0)
                )
                | F.col(
                    "population_quality_flag"
                ).isNull()
            )
            .count()
        )

        if invalid_output_count != 0:
            raise RuntimeError(
                "One or more output rows failed "
                "required-field validation."
            )

        unknown_settlement_count = (
            grid_year_df
            .filter(
                F.col("settlement_group")
                == F.lit("other_or_unknown")
            )
            .count()
        )

        if unknown_settlement_count != 0:
            raise RuntimeError(
                "Unexpected SMOD classes created "
                f"{unknown_settlement_count} unknown "
                "settlement records."
            )


        # ====================================================
        # VERIFY GRID IDENTIFIERS AGAINST 1975 GRID-YEAR
        # ====================================================

        grid_1975_keys_df = (
            grid_year_1975_df
            .select("grid_cell_id")
        )

        grid_2020_keys_df = (
            grid_year_df
            .select("grid_cell_id")
        )

        missing_from_2020 = (
            grid_1975_keys_df
            .join(
                grid_2020_keys_df,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        new_in_2020 = (
            grid_2020_keys_df
            .join(
                grid_1975_keys_df,
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        print()
        print("=" * 80)
        print("1975-2020 GRID-YEAR KEY COMPARISON")
        print("=" * 80)
        print(
            "1975 cells missing from 2020:",
            missing_from_2020,
        )
        print(
            "2020 cells absent from 1975:",
            new_in_2020,
        )

        if missing_from_2020 != 0:
            raise RuntimeError(
                "The 2020 grid-year dataset is missing "
                "one or more 1975 grid cells."
            )

        if new_in_2020 != 0:
            raise RuntimeError(
                "The 2020 grid-year dataset contains "
                "grid cells absent from 1975."
            )


        # ====================================================
        # POPULATION TOTAL RECONCILIATION
        # ====================================================

        population_input_total = float(
            population_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        joined_population_total = float(
            grid_year_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        population_difference = abs(
            population_input_total
            - joined_population_total
        )

        allowed_difference = max(
            1e-6,
            abs(population_input_total)
            * 1e-12,
        )

        if population_difference > allowed_difference:
            raise RuntimeError(
                "Population total changed during the join. "
                f"Input={population_input_total}, "
                f"joined={joined_population_total}, "
                f"difference={population_difference}."
            )


        # ====================================================
        # ANALYTICAL SUMMARIES
        # ====================================================

        print()
        print("=" * 80)
        print("2020 SETTLEMENT GROUP SUMMARY")
        print("=" * 80)

        settlement_summary_df = (
            grid_year_df
            .groupBy(
                "settlement_group"
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_total"
                ).alias(
                    "population_total"
                ),

                F.avg(
                    "population_total"
                ).alias(
                    "average_population_per_cell"
                ),

                F.sum(
                    F.when(
                        F.col("population_total")
                        > F.lit(0.0),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "populated_grid_cells"
                ),
            )
            .orderBy(
                "settlement_group"
            )
        )

        settlement_summary_df.show(
            20,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("2020 QUALITY SUMMARY")
        print("=" * 80)

        quality_summary_df = (
            grid_year_df
            .groupBy(
                "population_quality_flag"
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "valid_100m_pixels"
                ).alias(
                    "valid_100m_pixel_count"
                ),

                F.sum(
                    "population_total"
                ).alias(
                    "population_total"
                ),
            )
            .orderBy(
                "population_quality_flag"
            )
        )

        quality_summary_df.show(
            20,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("TOP 25 POPULATION CELLS IN 2020")
        print("=" * 80)

        (
            grid_year_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "population_total",
                "valid_100m_pixels",
                "population_coverage_fraction",
                "smod_class",
                "settlement_group",
                "is_urban",
                "population_quality_flag",
            )
            .orderBy(
                F.col(
                    "population_total"
                ).desc()
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE 2020 GRID-YEAR DATASET TO HDFS
        # ====================================================

        (
            grid_year_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )

        print(
            "HDFS 2020 grid-year Parquet write: COMPLETE"
        )


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if readback_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "2020 grid-year read-back count mismatch. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {readback_count}."
            )

        readback_duplicate_count = (
            readback_df
            .groupBy(
                "grid_cell_id",
                "epoch",
            )
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if readback_duplicate_count != 0:
            raise RuntimeError(
                "2020 grid-year read-back contains "
                "duplicate keys."
            )

        readback_population_total = float(
            readback_df
            .agg(
                F.sum(
                    "population_total"
                ).alias("total")
            )
            .first()["total"]
        )

        readback_difference = abs(
            readback_population_total
            - joined_population_total
        )

        if readback_difference > allowed_difference:
            raise RuntimeError(
                "The HDFS read-back changed the 2020 "
                "population total."
            )


        complete_cell_count = (
            readback_df
            .filter(
                F.col(
                    "population_quality_flag"
                )
                == F.lit("complete")
            )
            .count()
        )

        partial_cell_count = (
            readback_df
            .filter(
                F.col(
                    "population_quality_flag"
                )
                == F.lit(
                    "partial_pilot_boundary"
                )
            )
            .count()
        )

        urban_cell_count = (
            readback_df
            .filter(
                F.col("is_urban")
            )
            .count()
        )

        urban_centre_cell_count = (
            readback_df
            .filter(
                F.col("is_urban_centre")
            )
            .count()
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 12 REMOTE RESULT")
        print("=" * 80)

        print("Population 2020 Parquet read: PASS")
        print("SMOD 2020 Parquet read: PASS")
        print("Two-way key coverage: PASS")
        print("Grid coordinate agreement: PASS")
        print("Spark one-to-one join: PASS")
        print("Settlement-group derivation: PASS")
        print("Sedona 1 km geometry creation: PASS")
        print("Population conservation after join: PASS")
        print("1975-2020 grid-key agreement: PASS")
        print("HDFS grid-year Parquet write: PASS")
        print("HDFS grid-year Parquet read-back: PASS")

        print(
            "Grid-year rows:",
            readback_count,
        )

        print(
            "Complete population cells:",
            complete_cell_count,
        )

        print(
            "Partial boundary cells:",
            partial_cell_count,
        )

        print(
            "Urban cells:",
            urban_cell_count,
        )

        print(
            "Urban-centre cells:",
            urban_centre_cell_count,
        )

        print(
            "Population total:",
            readback_population_total,
        )

        print(
            "Output:",
            OUTPUT_PATH,
        )

        print("Source GeoTIFF reread: NO")
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 12: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell12_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell12_grid_year_2020.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "

    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "

    "--master local[1] "
    "--driver-memory 2g "

    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1200,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 12 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Remote return code:",
    result.returncode,
)
print("CELL 12: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 12: BENGALURU 2020 GRID-YEAR JOIN
Population input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=population/epoch=2020/grid_1km
SMOD input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=smod/epoch=2020/grid_1km
1975 grid-year reference: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=2020

INPUT ROW COUNTS
Population 2020 rows: 3658
SMOD 2020 rows: 3658
Grid-year 1975 reference rows: 3658

2020 TWO-WAY KEY COVERAGE
Population cells without SMOD: 0
SMOD cells without population: 0
Grid coordinate agreement: PASS

1975-2020 GRID-YEAR KEY COMPARISON
1975 cells missing from 2020: 0
2020 cells absent from 1975: 0

2020 SETTLEMENT GROUP SUMMARY
+-

Cell 12 succeeded completely. The Bengaluru 2020 grid-year dataset is now joined, validated, and persisted in HDFS.

Confirmed 2020 result
Metric	ValueGrid-year rows	3,658
Complete population cells	3,480
Partial boundary cells	178
Urban cells	2,011
Urban-centre cells	950
Population total	14,608,885.86

The join passed:

exact two-way population-SMOD key coverage;
grid-coordinate agreement;
one-to-one Spark join;
population conservation;
exact 1975–2020 grid-key agreement;
HDFS write and read-back validation.
2020 settlement totals
Settlement group	Cells	PopulationRural	1,645	188,116.56
Urban cluster	1,061	937,602.07
Urban centre	950	13,483,167.23
Water	2	0

Approximately 92.3% of the pilot population is in urban-centre cells in 2020. The top 25 population cells are all complete 1 km urban-centre cells, with the highest containing about 76,534 people.

Persistent output
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=2020

The next step is Cell 13, which should compare the complete 1975 and 2020 grid-year datasets and produce:

cell-level population change;
percentage growth;
SMOD class transitions;
settlement-group transition matrix;
rural-to-urban conversion counts;
new urban-centre cells;
population change by transition type;
persistent transition Parquet outputs.

This will be the first direct analytical result answering the project’s research question.

In [22]:
# ============================================================
# CELL 13: BENGALURU 1975 TO 2020 TRANSITION ANALYSIS
#
# Inputs:
#   - grid_year/epoch=1975
#   - grid_year/epoch=2020
#
# Outputs:
#   - detailed cell transitions
#   - settlement-group transition matrix
#   - SMOD-class transition matrix
#   - transition-category summary
#
# Primary tools:
#   - HDFS for persistent input/output
#   - Spark DataFrames and Spark SQL for all analysis
#
# No GeoTIFF is reread.
# No Python UDF, NumPy, or Rasterio is used.
# ============================================================

import base64
import textwrap


cell13_program = textwrap.dedent(
    r'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    GRID_YEAR_1975_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "grid_year/"
        "epoch=1975"
    )

    GRID_YEAR_2020_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "grid_year/"
        "epoch=2020"
    )

    OUTPUT_ROOT = (
        "hdfs://localhost:9000"
        "/data/ghsl/analytics/pilots/bengaluru/"
        "transition_1975_2020"
    )

    DETAIL_OUTPUT_PATH = (
        OUTPUT_ROOT
        + "/cell_transitions"
    )

    GROUP_MATRIX_OUTPUT_PATH = (
        OUTPUT_ROOT
        + "/settlement_group_matrix"
    )

    SMOD_MATRIX_OUTPUT_PATH = (
        OUTPUT_ROOT
        + "/smod_class_matrix"
    )

    CATEGORY_SUMMARY_OUTPUT_PATH = (
        OUTPUT_ROOT
        + "/transition_category_summary"
    )

    CITY = "bengaluru"
    START_EPOCH = 1975
    END_EPOCH = 2020
    YEARS_ELAPSED = END_EPOCH - START_EPOCH

    EXPECTED_GRID_CELLS = 3658

    EXPECTED_POPULATION_1975 = (
        3060319.0921731493
    )

    EXPECTED_POPULATION_2020 = (
        14608885.859992506
    )


    spark = None

    try:
        # ====================================================
        # CREATE SPARK SESSION
        #
        # This stage uses compact scalar Parquet records.
        # Sedona raster functions are not required.
        # ====================================================

        spark = (
            SparkSession.builder
            .appName(
                "GHSL-Bengaluru-1975-2020-"
                "Transition-Analysis"
            )
            .getOrCreate()
        )

        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 13: BENGALURU 1975 TO 2020 TRANSITIONS")
        print("=" * 80)
        print("1975 input:", GRID_YEAR_1975_PATH)
        print("2020 input:", GRID_YEAR_2020_PATH)
        print("Output root:", OUTPUT_ROOT)


        # ====================================================
        # READ ENDPOINT GRID-YEAR DATASETS
        # ====================================================

        grid_1975_raw_df = spark.read.parquet(
            GRID_YEAR_1975_PATH
        )

        grid_2020_raw_df = spark.read.parquet(
            GRID_YEAR_2020_PATH
        )


        # ====================================================
        # NORMALIZE REQUIRED COLUMNS
        # ====================================================

        grid_1975_df = (
            grid_1975_raw_df
            .select(
                F.col("grid_cell_id").cast(
                    "string"
                ).alias("grid_cell_id"),

                F.col("grid_1km_col").cast(
                    "long"
                ).alias("grid_1km_col_1975"),

                F.col("grid_1km_row").cast(
                    "long"
                ).alias("grid_1km_row_1975"),

                F.col("grid_min_x").cast(
                    "double"
                ).alias("grid_min_x_1975"),

                F.col("grid_max_x").cast(
                    "double"
                ).alias("grid_max_x_1975"),

                F.col("grid_min_y").cast(
                    "double"
                ).alias("grid_min_y_1975"),

                F.col("grid_max_y").cast(
                    "double"
                ).alias("grid_max_y_1975"),

                F.col("population_total").cast(
                    "double"
                ).alias("population_1975"),

                F.col(
                    "population_coverage_fraction"
                ).cast(
                    "double"
                ).alias(
                    "population_coverage_1975"
                ),

                F.col("smod_class").cast(
                    "int"
                ).alias("smod_class_1975"),

                F.col("settlement_group").cast(
                    "string"
                ).alias("settlement_group_1975"),

                F.col("is_urban").cast(
                    "boolean"
                ).alias("is_urban_1975"),

                F.col("is_urban_centre").cast(
                    "boolean"
                ).alias("is_urban_centre_1975"),

                F.col(
                    "population_quality_flag"
                ).cast(
                    "string"
                ).alias(
                    "population_quality_flag_1975"
                ),

                F.col("geometry_wkb").alias(
                    "geometry_wkb"
                ),

                F.col("geometry_crs").alias(
                    "geometry_crs"
                ),
            )
        )


        grid_2020_df = (
            grid_2020_raw_df
            .select(
                F.col("grid_cell_id").cast(
                    "string"
                ).alias("grid_cell_id"),

                F.col("grid_1km_col").cast(
                    "long"
                ).alias("grid_1km_col_2020"),

                F.col("grid_1km_row").cast(
                    "long"
                ).alias("grid_1km_row_2020"),

                F.col("grid_min_x").cast(
                    "double"
                ).alias("grid_min_x_2020"),

                F.col("grid_max_x").cast(
                    "double"
                ).alias("grid_max_x_2020"),

                F.col("grid_min_y").cast(
                    "double"
                ).alias("grid_min_y_2020"),

                F.col("grid_max_y").cast(
                    "double"
                ).alias("grid_max_y_2020"),

                F.col("population_total").cast(
                    "double"
                ).alias("population_2020"),

                F.col(
                    "population_coverage_fraction"
                ).cast(
                    "double"
                ).alias(
                    "population_coverage_2020"
                ),

                F.col("smod_class").cast(
                    "int"
                ).alias("smod_class_2020"),

                F.col("settlement_group").cast(
                    "string"
                ).alias("settlement_group_2020"),

                F.col("is_urban").cast(
                    "boolean"
                ).alias("is_urban_2020"),

                F.col("is_urban_centre").cast(
                    "boolean"
                ).alias("is_urban_centre_2020"),

                F.col(
                    "population_quality_flag"
                ).cast(
                    "string"
                ).alias(
                    "population_quality_flag_2020"
                ),
            )
        )


        # ====================================================
        # INPUT VALIDATION
        # ====================================================

        count_1975 = grid_1975_df.count()
        count_2020 = grid_2020_df.count()

        print()
        print("=" * 80)
        print("INPUT COUNTS")
        print("=" * 80)
        print("1975 grid cells:", count_1975)
        print("2020 grid cells:", count_2020)

        if count_1975 != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 1975 grid count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {count_1975}."
            )

        if count_2020 != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 2020 grid count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {count_2020}."
            )


        def duplicate_count(dataframe):
            return (
                dataframe
                .groupBy("grid_cell_id")
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )


        if duplicate_count(grid_1975_df) != 0:
            raise RuntimeError(
                "1975 input contains duplicate grid IDs."
            )

        if duplicate_count(grid_2020_df) != 0:
            raise RuntimeError(
                "2020 input contains duplicate grid IDs."
            )


        missing_from_2020 = (
            grid_1975_df
            .select("grid_cell_id")
            .join(
                grid_2020_df.select(
                    "grid_cell_id"
                ),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        new_in_2020 = (
            grid_2020_df
            .select("grid_cell_id")
            .join(
                grid_1975_df.select(
                    "grid_cell_id"
                ),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        if missing_from_2020 != 0:
            raise RuntimeError(
                "One or more 1975 cells are missing "
                "from 2020."
            )

        if new_in_2020 != 0:
            raise RuntimeError(
                "One or more 2020 cells are absent "
                "from 1975."
            )

        print("Endpoint grid-key agreement: PASS")


        # ====================================================
        # JOIN ENDPOINTS
        # ====================================================

        joined_df = (
            grid_1975_df.alias("a")
            .join(
                grid_2020_df.alias("b"),
                on="grid_cell_id",
                how="inner",
            )
        )

        joined_count = joined_df.count()

        if joined_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected endpoint join count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {joined_count}."
            )


        # ====================================================
        # VERIFY SPATIAL GRID AGREEMENT
        # ====================================================

        spatial_mismatch_count = (
            joined_df
            .filter(
                (
                    F.col("a.grid_1km_col_1975")
                    != F.col("b.grid_1km_col_2020")
                )
                | (
                    F.col("a.grid_1km_row_1975")
                    != F.col("b.grid_1km_row_2020")
                )
                | (
                    F.abs(
                        F.col("a.grid_min_x_1975")
                        - F.col("b.grid_min_x_2020")
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col("a.grid_max_x_1975")
                        - F.col("b.grid_max_x_2020")
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col("a.grid_min_y_1975")
                        - F.col("b.grid_min_y_2020")
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col("a.grid_max_y_1975")
                        - F.col("b.grid_max_y_2020")
                    )
                    > F.lit(1e-9)
                )
            )
            .count()
        )

        if spatial_mismatch_count != 0:
            raise RuntimeError(
                "Endpoint spatial coordinates differ for "
                f"{spatial_mismatch_count} cells."
            )

        print("Endpoint spatial grid agreement: PASS")


        # ====================================================
        # BUILD CELL-LEVEL TRANSITION RECORDS
        # ====================================================

        transitions_df = (
            joined_df
            .select(
                F.lit(CITY).alias("city"),
                F.col("grid_cell_id"),

                F.lit(START_EPOCH).alias(
                    "start_epoch"
                ),

                F.lit(END_EPOCH).alias(
                    "end_epoch"
                ),

                F.lit(YEARS_ELAPSED).alias(
                    "years_elapsed"
                ),

                F.col(
                    "a.grid_1km_col_1975"
                ).alias("grid_1km_col"),

                F.col(
                    "a.grid_1km_row_1975"
                ).alias("grid_1km_row"),

                F.col(
                    "a.grid_min_x_1975"
                ).alias("grid_min_x"),

                F.col(
                    "a.grid_max_x_1975"
                ).alias("grid_max_x"),

                F.col(
                    "a.grid_min_y_1975"
                ).alias("grid_min_y"),

                F.col(
                    "a.grid_max_y_1975"
                ).alias("grid_max_y"),

                F.col(
                    "a.population_1975"
                ).alias("population_1975"),

                F.col(
                    "b.population_2020"
                ).alias("population_2020"),

                (
                    F.col("b.population_2020")
                    - F.col("a.population_1975")
                ).alias(
                    "population_change_absolute"
                ),

                F.when(
                    F.col("a.population_1975")
                    > F.lit(0.0),

                    (
                        (
                            F.col("b.population_2020")
                            - F.col(
                                "a.population_1975"
                            )
                        )
                        / F.col(
                            "a.population_1975"
                        )
                    )
                    * F.lit(100.0),
                )
                .otherwise(
                    F.lit(None).cast("double")
                )
                .alias(
                    "population_change_percent"
                ),

                F.when(
                    F.col("a.population_1975")
                    > F.lit(0.0),

                    F.pow(
                        (
                            F.col("b.population_2020")
                            / F.col(
                                "a.population_1975"
                            )
                        ),
                        F.lit(
                            1.0 / YEARS_ELAPSED
                        ),
                    )
                    - F.lit(1.0),
                )
                .otherwise(
                    F.lit(None).cast("double")
                )
                .alias(
                    "population_cagr_fraction"
                ),

                F.col(
                    "a.population_coverage_1975"
                ).alias(
                    "population_coverage_1975"
                ),

                F.col(
                    "b.population_coverage_2020"
                ).alias(
                    "population_coverage_2020"
                ),

                F.col(
                    "a.smod_class_1975"
                ).alias("smod_class_1975"),

                F.col(
                    "b.smod_class_2020"
                ).alias("smod_class_2020"),

                F.col(
                    "a.settlement_group_1975"
                ).alias("settlement_group_1975"),

                F.col(
                    "b.settlement_group_2020"
                ).alias("settlement_group_2020"),

                F.col(
                    "a.is_urban_1975"
                ).alias("is_urban_1975"),

                F.col(
                    "b.is_urban_2020"
                ).alias("is_urban_2020"),

                F.col(
                    "a.is_urban_centre_1975"
                ).alias(
                    "is_urban_centre_1975"
                ),

                F.col(
                    "b.is_urban_centre_2020"
                ).alias(
                    "is_urban_centre_2020"
                ),

                F.col(
                    "a.population_quality_flag_1975"
                ).alias(
                    "population_quality_flag_1975"
                ),

                F.col(
                    "b.population_quality_flag_2020"
                ).alias(
                    "population_quality_flag_2020"
                ),

                F.col("a.geometry_wkb"),
                F.col("a.geometry_crs"),
            )
            .withColumn(
                "settlement_transition",
                F.concat_ws(
                    " -> ",
                    F.col("settlement_group_1975"),
                    F.col("settlement_group_2020"),
                ),
            )
            .withColumn(
                "smod_class_transition",
                F.concat_ws(
                    " -> ",
                    F.col(
                        "smod_class_1975"
                    ).cast("string"),
                    F.col(
                        "smod_class_2020"
                    ).cast("string"),
                ),
            )
            .withColumn(
                "transition_category",

                F.when(
                    (
                        F.col(
                            "settlement_group_1975"
                        )
                        == F.lit("rural")
                    )
                    & (
                        F.col(
                            "settlement_group_2020"
                        )
                        == F.lit(
                            "urban_centre"
                        )
                    ),
                    F.lit(
                        "rural_to_urban_centre"
                    ),
                )

                .when(
                    (
                        F.col(
                            "settlement_group_1975"
                        )
                        == F.lit("rural")
                    )
                    & (
                        F.col(
                            "settlement_group_2020"
                        )
                        == F.lit(
                            "urban_cluster"
                        )
                    ),
                    F.lit(
                        "rural_to_urban_cluster"
                    ),
                )

                .when(
                    (
                        F.col(
                            "settlement_group_1975"
                        )
                        == F.lit(
                            "urban_cluster"
                        )
                    )
                    & (
                        F.col(
                            "settlement_group_2020"
                        )
                        == F.lit(
                            "urban_centre"
                        )
                    ),
                    F.lit(
                        "urban_cluster_to_centre"
                    ),
                )

                .when(
                    (
                        ~F.col(
                            "is_urban_centre_1975"
                        )
                    )
                    & F.col(
                        "is_urban_centre_2020"
                    ),
                    F.lit(
                        "new_urban_centre_other_origin"
                    ),
                )

                .when(
                    (
                        ~F.col("is_urban_1975")
                    )
                    & F.col("is_urban_2020"),
                    F.lit(
                        "new_urban_other"
                    ),
                )

                .when(
                    F.col(
                        "settlement_group_1975"
                    )
                    == F.col(
                        "settlement_group_2020"
                    ),
                    F.lit(
                        "stable_settlement_group"
                    ),
                )

                .when(
                    F.col("is_urban_1975")
                    & (
                        ~F.col("is_urban_2020")
                    ),
                    F.lit(
                        "urban_to_nonurban"
                    ),
                )

                .otherwise(
                    F.lit(
                        "other_transition"
                    )
                ),
            )
            .withColumn(
                "became_urban",
                (
                    ~F.col("is_urban_1975")
                )
                & F.col("is_urban_2020"),
            )
            .withColumn(
                "became_urban_centre",
                (
                    ~F.col(
                        "is_urban_centre_1975"
                    )
                )
                & F.col(
                    "is_urban_centre_2020"
                ),
            )
            .withColumn(
                "remained_urban",
                F.col("is_urban_1975")
                & F.col("is_urban_2020"),
            )
            .withColumn(
                "remained_rural",
                (
                    F.col(
                        "settlement_group_1975"
                    )
                    == F.lit("rural")
                )
                & (
                    F.col(
                        "settlement_group_2020"
                    )
                    == F.lit("rural")
                ),
            )
            .withColumn(
                "population_growth_positive",
                F.col(
                    "population_change_absolute"
                )
                > F.lit(0.0),
            )
            .withColumn(
                "is_complete_both_epochs",
                (
                    F.col(
                        "population_quality_flag_1975"
                    )
                    == F.lit("complete")
                )
                & (
                    F.col(
                        "population_quality_flag_2020"
                    )
                    == F.lit("complete")
                ),
            )
            .withColumn(
                "processed_at_utc",
                F.current_timestamp(),
            )
        )


        # ====================================================
        # TRANSITION VALIDATION
        # ====================================================

        transition_count = transitions_df.count()

        if transition_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected transition row count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {transition_count}."
            )


        transition_duplicate_count = (
            transitions_df
            .groupBy("grid_cell_id")
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if transition_duplicate_count != 0:
            raise RuntimeError(
                "Transition output contains duplicate cells."
            )


        invalid_transition_count = (
            transitions_df
            .filter(
                F.col("grid_cell_id").isNull()
                | F.col("population_1975").isNull()
                | F.col("population_2020").isNull()
                | F.isnan("population_1975")
                | F.isnan("population_2020")
                | F.col("smod_class_1975").isNull()
                | F.col("smod_class_2020").isNull()
                | F.col(
                    "settlement_group_1975"
                ).isNull()
                | F.col(
                    "settlement_group_2020"
                ).isNull()
                | F.col(
                    "transition_category"
                ).isNull()
                | F.col("geometry_wkb").isNull()
            )
            .count()
        )

        if invalid_transition_count != 0:
            raise RuntimeError(
                "One or more transition records failed "
                "required-field validation."
            )


        # ====================================================
        # ENDPOINT TOTAL VALIDATION
        # ====================================================

        totals = (
            transitions_df
            .agg(
                F.sum(
                    "population_1975"
                ).alias(
                    "population_1975"
                ),

                F.sum(
                    "population_2020"
                ).alias(
                    "population_2020"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),
            )
            .first()
        )

        total_1975 = float(
            totals["population_1975"]
        )

        total_2020 = float(
            totals["population_2020"]
        )

        total_change = float(
            totals[
                "population_change_absolute"
            ]
        )

        tolerance_1975 = max(
            1e-6,
            EXPECTED_POPULATION_1975 * 1e-12,
        )

        tolerance_2020 = max(
            1e-6,
            EXPECTED_POPULATION_2020 * 1e-12,
        )

        if (
            abs(
                total_1975
                - EXPECTED_POPULATION_1975
            )
            > tolerance_1975
        ):
            raise RuntimeError(
                "1975 population total does not match "
                "the validated endpoint."
            )

        if (
            abs(
                total_2020
                - EXPECTED_POPULATION_2020
            )
            > tolerance_2020
        ):
            raise RuntimeError(
                "2020 population total does not match "
                "the validated endpoint."
            )

        expected_change = (
            total_2020 - total_1975
        )

        if (
            abs(total_change - expected_change)
            > max(
                1e-6,
                abs(expected_change) * 1e-12,
            )
        ):
            raise RuntimeError(
                "Population-change conservation failed."
            )


        # ====================================================
        # SETTLEMENT-GROUP TRANSITION MATRIX
        # ====================================================

        group_matrix_df = (
            transitions_df
            .groupBy(
                "settlement_group_1975",
                "settlement_group_2020",
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_1975"
                ).alias(
                    "population_1975"
                ),

                F.sum(
                    "population_2020"
                ).alias(
                    "population_2020"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),

                F.avg(
                    "population_change_absolute"
                ).alias(
                    "average_change_per_cell"
                ),
            )
            .withColumn(
                "population_change_percent",

                F.when(
                    F.col("population_1975")
                    > F.lit(0.0),

                    (
                        F.col(
                            "population_change_absolute"
                        )
                        / F.col(
                            "population_1975"
                        )
                    )
                    * F.lit(100.0),
                )
                .otherwise(
                    F.lit(None).cast("double")
                ),
            )
            .orderBy(
                "settlement_group_1975",
                "settlement_group_2020",
            )
        )


        # ====================================================
        # SMOD-CLASS TRANSITION MATRIX
        # ====================================================

        smod_matrix_df = (
            transitions_df
            .groupBy(
                "smod_class_1975",
                "smod_class_2020",
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_1975"
                ).alias(
                    "population_1975"
                ),

                F.sum(
                    "population_2020"
                ).alias(
                    "population_2020"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),
            )
            .orderBy(
                "smod_class_1975",
                "smod_class_2020",
            )
        )


        # ====================================================
        # TRANSITION-CATEGORY SUMMARY
        # ====================================================

        category_summary_df = (
            transitions_df
            .groupBy(
                "transition_category"
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_1975"
                ).alias(
                    "population_1975"
                ),

                F.sum(
                    "population_2020"
                ).alias(
                    "population_2020"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),

                F.avg(
                    "population_change_absolute"
                ).alias(
                    "average_change_per_cell"
                ),

                F.sum(
                    F.when(
                        F.col(
                            "population_growth_positive"
                        ),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "cells_with_population_growth"
                ),
            )
            .orderBy(
                F.col(
                    "grid_cell_count"
                ).desc(),
            )
        )


        # ====================================================
        # SUMMARY METRICS
        # ====================================================

        became_urban_count = (
            transitions_df
            .filter(
                F.col("became_urban")
            )
            .count()
        )

        became_urban_centre_count = (
            transitions_df
            .filter(
                F.col("became_urban_centre")
            )
            .count()
        )

        remained_urban_count = (
            transitions_df
            .filter(
                F.col("remained_urban")
            )
            .count()
        )

        remained_rural_count = (
            transitions_df
            .filter(
                F.col("remained_rural")
            )
            .count()
        )

        positive_growth_cell_count = (
            transitions_df
            .filter(
                F.col(
                    "population_growth_positive"
                )
            )
            .count()
        )

        complete_both_epochs_count = (
            transitions_df
            .filter(
                F.col(
                    "is_complete_both_epochs"
                )
            )
            .count()
        )

        total_growth_percent = (
            total_change
            / total_1975
            * 100.0
        )

        total_growth_multiple = (
            total_2020
            / total_1975
        )

        total_cagr_percent = (
            (
                total_growth_multiple
                ** (
                    1.0 / YEARS_ELAPSED
                )
            )
            - 1.0
        ) * 100.0


        # ====================================================
        # DISPLAY ANALYTICAL RESULTS
        # ====================================================

        print()
        print("=" * 80)
        print("ENDPOINT POPULATION CHANGE")
        print("=" * 80)
        print("Population 1975:", total_1975)
        print("Population 2020:", total_2020)
        print("Absolute increase:", total_change)
        print(
            "Percentage increase:",
            total_growth_percent,
        )
        print(
            "Growth multiple:",
            total_growth_multiple,
        )
        print(
            "Endpoint CAGR percent:",
            total_cagr_percent,
        )


        print()
        print("=" * 80)
        print("SETTLEMENT-GROUP TRANSITION MATRIX")
        print("=" * 80)

        group_matrix_df.show(
            100,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("SMOD-CLASS TRANSITION MATRIX")
        print("=" * 80)

        smod_matrix_df.show(
            200,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("TRANSITION-CATEGORY SUMMARY")
        print("=" * 80)

        category_summary_df.show(
            100,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("TOP 30 POPULATION-GROWTH CELLS")
        print("=" * 80)

        (
            transitions_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "population_1975",
                "population_2020",
                "population_change_absolute",
                "population_change_percent",
                "smod_class_1975",
                "smod_class_2020",
                "settlement_group_1975",
                "settlement_group_2020",
                "transition_category",
            )
            .orderBy(
                F.col(
                    "population_change_absolute"
                ).desc(),
            )
            .show(
                30,
                truncate=False,
            )
        )


        print()
        print("=" * 80)
        print("TOP 30 POPULATION-DECLINE CELLS")
        print("=" * 80)

        (
            transitions_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "population_1975",
                "population_2020",
                "population_change_absolute",
                "population_change_percent",
                "settlement_group_1975",
                "settlement_group_2020",
                "transition_category",
            )
            .orderBy(
                F.col(
                    "population_change_absolute"
                ).asc(),
            )
            .show(
                30,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE ALL ANALYTICAL OUTPUTS TO HDFS
        # ====================================================

        (
            transitions_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(DETAIL_OUTPUT_PATH)
        )

        (
            group_matrix_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(
                GROUP_MATRIX_OUTPUT_PATH
            )
        )

        (
            smod_matrix_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(
                SMOD_MATRIX_OUTPUT_PATH
            )
        )

        (
            category_summary_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(
                CATEGORY_SUMMARY_OUTPUT_PATH
            )
        )

        print("HDFS transition outputs: WRITTEN")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        detail_readback_df = spark.read.parquet(
            DETAIL_OUTPUT_PATH
        )

        group_readback_df = spark.read.parquet(
            GROUP_MATRIX_OUTPUT_PATH
        )

        smod_readback_df = spark.read.parquet(
            SMOD_MATRIX_OUTPUT_PATH
        )

        category_readback_df = spark.read.parquet(
            CATEGORY_SUMMARY_OUTPUT_PATH
        )

        detail_readback_count = (
            detail_readback_df.count()
        )

        if (
            detail_readback_count
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "Transition-detail read-back count "
                "does not match the expected grid."
            )

        group_matrix_cell_total = (
            group_readback_df
            .agg(
                F.sum(
                    "grid_cell_count"
                ).alias("total")
            )
            .first()["total"]
        )

        smod_matrix_cell_total = (
            smod_readback_df
            .agg(
                F.sum(
                    "grid_cell_count"
                ).alias("total")
            )
            .first()["total"]
        )

        category_cell_total = (
            category_readback_df
            .agg(
                F.sum(
                    "grid_cell_count"
                ).alias("total")
            )
            .first()["total"]
        )

        if (
            int(group_matrix_cell_total)
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "Settlement transition matrix does not "
                "reconcile to the full grid."
            )

        if (
            int(smod_matrix_cell_total)
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "SMOD transition matrix does not "
                "reconcile to the full grid."
            )

        if (
            int(category_cell_total)
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "Transition-category summary does not "
                "reconcile to the full grid."
            )


        readback_totals = (
            detail_readback_df
            .agg(
                F.sum(
                    "population_1975"
                ).alias(
                    "population_1975"
                ),

                F.sum(
                    "population_2020"
                ).alias(
                    "population_2020"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),
            )
            .first()
        )

        if (
            abs(
                float(
                    readback_totals[
                        "population_1975"
                    ]
                )
                - total_1975
            )
            > tolerance_1975
        ):
            raise RuntimeError(
                "Transition read-back changed the "
                "1975 population total."
            )

        if (
            abs(
                float(
                    readback_totals[
                        "population_2020"
                    ]
                )
                - total_2020
            )
            > tolerance_2020
        ):
            raise RuntimeError(
                "Transition read-back changed the "
                "2020 population total."
            )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 13 REMOTE RESULT")
        print("=" * 80)
        print("1975 grid-year read: PASS")
        print("2020 grid-year read: PASS")
        print("Endpoint key agreement: PASS")
        print("Endpoint spatial-grid agreement: PASS")
        print("Spark cell-level transition join: PASS")
        print("Population-change conservation: PASS")
        print("Settlement transition matrix: PASS")
        print("SMOD transition matrix: PASS")
        print("Transition-category summary: PASS")
        print("HDFS output writes: PASS")
        print("HDFS output read-back: PASS")
        print("Transition rows:", detail_readback_count)
        print(
            "Complete cells in both epochs:",
            complete_both_epochs_count,
        )
        print(
            "Cells that became urban:",
            became_urban_count,
        )
        print(
            "Cells that became urban centres:",
            became_urban_centre_count,
        )
        print(
            "Cells remaining urban:",
            remained_urban_count,
        )
        print(
            "Cells remaining rural:",
            remained_rural_count,
        )
        print(
            "Cells with positive population growth:",
            positive_growth_cell_count,
        )
        print("Population 1975:", total_1975)
        print("Population 2020:", total_2020)
        print("Absolute population increase:", total_change)
        print(
            "Percentage population increase:",
            total_growth_percent,
        )
        print(
            "Population growth multiple:",
            total_growth_multiple,
        )
        print(
            "Endpoint CAGR percent:",
            total_cagr_percent,
        )
        print("Detailed output:", DETAIL_OUTPUT_PATH)
        print(
            "Settlement matrix output:",
            GROUP_MATRIX_OUTPUT_PATH,
        )
        print(
            "SMOD matrix output:",
            SMOD_MATRIX_OUTPUT_PATH,
        )
        print(
            "Category summary output:",
            CATEGORY_SUMMARY_OUTPUT_PATH,
        )
        print("Source GeoTIFF reread: NO")
        print("Python UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 13: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell13_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell13_transitions_1975_2020.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "

    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 2g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.driver.maxResultSize=256m "
    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1200,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 13 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Remote return code:",
    result.returncode,
)
print("CELL 13: SUCCESS")

Remote Python syntax: PASS
CELL 13: BENGALURU 1975 TO 2020 TRANSITIONS
1975 input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975
2020 input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=2020
Output root: hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/transition_1975_2020

INPUT COUNTS
1975 grid cells: 3658
2020 grid cells: 3658
Endpoint grid-key agreement: PASS
Endpoint spatial grid agreement: PASS

ENDPOINT POPULATION CHANGE
Population 1975: 3060319.092173149
Population 2020: 14608885.859992506
Absolute increase: 11548566.767819364
Percentage increase: 377.36479170930716
Growth multiple: 4.7736479170930695
Endpoint CAGR percent: 3.534612903919254

SETTLEMENT-GROUP TRANSITION MATRIX
+---------------------+---------------------+---------------+------------------+------------------+--------------------------+-----------------------+-------------------------+
|settlement_group_1975|settlement_group_2020|grid_cell_coun

Cell 13 succeeded completely. You now have the first direct analytical result for the project, not merely setup or preprocessing.

Bengaluru, 1975 to 2020

Metric	ResultComparable 1 km cells	3,658

Complete cells in both epochs	3,480

Population in 1975	3,060,319

Population in 2020	14,608,886

Absolute increase	11,548,567

Percentage increase	377.36%

Growth multiple	4.77×

Endpoint CAGR	3.53% per year

Cells with positive population growth	3,033

Cells that became urban	1,630

Cells that became urban centres	748

Cells that remained urban	381

Cells that remained rural	1,627

Main settlement transitions
1,029 cells changed from rural to urban cluster, adding about 752,428 people.
601 cells changed directly from rural to urban centre, adding approximately 4.04 million people.
147 cells changed from urban cluster to urban centre, adding approximately 1.41 million people.
The 202 cells that were already urban centres in 1975 remained urban centres and added approximately 5.30 million people.
17 cells changed from urban cluster to rural and lost about 6,038 people in total. These should eventually be treated as possible reclassification or boundary effects rather than immediately interpreted as deurbanization.
Strongest interpretation so far

The results indicate two simultaneous processes:

Core intensification: existing urban-centre cells accounted for approximately 5.30 million of the population increase.
Outward spatial expansion: rural-to-urban-centre and rural-to-urban-cluster conversions accounted for another approximately 4.79 million of growth.

Together, those three main transition paths explain roughly 87% of the total population increase. This is consistent with Bengaluru growing through both concentration in the established core and conversion of surrounding rural cells.

Persistent outputs
Cell-level transitions:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/transition_1975_2020/cell_transitions
Settlement-group matrix:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/transition_1975_2020/settlement_group_matrix
SMOD-class matrix:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/transition_1975_2020/smod_class_matrix
Transition-category summary:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/transition_1975_2020/transition_category_summary


All outputs passed HDFS write and read-back validation. The analysis used the compact scalar Parquet datasets and did not reread the source GeoTIFFs or use a Python UDF, NumPy, or Rasterio.

Next step

We should now add built-up surface for 1975 and 2020 to the same grid.

Recommended order:

Built surface 1975, 100 m to 1 km.
Built surface 2020, 100 m to 1 km.
Join built surface into both existing grid-year datasets.
Extend Cell 13 with:
built-up area change;
population density per built-up area;
physical expansion versus population intensification;
rural-to-urban transitions with built-up evidence.

This directly advances the proposal’s goal of analyzing population and physical urban expansion together.

We have completed an end-to-end Bengaluru endpoint prototype using population and SMOD for 1975 and 2020, including a validated cell-level transition analysis.

1. Datasets used so far
A. GHSL population

We processed the 100 m population rasters for:

1975
2020

For each year, the pipeline:

Plain Text
HDFS GeoTIFF
→ Spark binaryFile
→ Sedona raster decoding
→ Sedona 256 × 256 tiling
→ Bengaluru tile selection
→ Sedona band extraction
→ Spark SQL 100 m to 1 km aggregation
→ HDFS Parquet


Results:

Year	Population	1 km cells1975	3,060,319	3,658
2020	14,608,886	3,658

The grid identifiers match exactly between the two years, and population conservation passed for both aggregations.

B. GHSL SMOD settlement classification

We processed the native 1 km SMOD rasters for:

1975
2020

Each SMOD record was aligned with the same 3,658 Mollweide 1 km cells used by population.

The outputs contain:

numeric SMOD class;
broad settlement group;
urban flag;
urban-centre flag;
grid coordinates;
source lineage.

SMOD coverage matched population with zero missing grid cells in both years.

C. GHSL metadata catalog

We used the existing Spark/HDFS metadata catalog to discover source rasters rather than hard-coding every filename.

The catalog contains all 325 raw GHSL GeoTIFFs, including:

Product	FilesPopulation	150
Built-up surface	150
SMOD	10
Building height	15
Total	325

The catalog itself was created and benchmarked in 03_ghsl_apache_sedona_setup_and_benchmark (1).py.

However, cataloging a dataset does not mean we have analytically processed it. So far, only population and SMOD have entered the grid-year analysis.

2. What analytical outputs we created
Bengaluru 1975 grid-year dataset

We joined:

Plain Text
Population 1975 at 1 km
+
SMOD 1975 at 1 km

Output:

Plain Text
/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975

It contains 3,658 records with population, settlement class, quality information, lineage, and 1 km geometry.

Bengaluru 2020 grid-year dataset

We joined:

Plain Text
Population 2020 at 1 km
+
SMOD 2020 at 1 km

Output:

Plain Text
/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=2020

It contains the same 3,658 grid cells, allowing a direct cell-by-cell historical comparison.

Bengaluru 1975-to-2020 transition dataset

We compared the two complete grid-year datasets and calculated:

absolute population change;
percentage population growth;
endpoint CAGR;
SMOD class transition;
settlement-group transition;
rural-to-urban conversion;
new urban-centre cells;
stable urban and rural cells;
population change by transition type.

Main results:

Metric	ResultComparable cells	3,658
Population increase	11,548,567
Percentage increase	377.36%
Growth multiple	4.77×
Cells that became urban	1,630
Cells that became urban centres	748
Cells remaining urban	381
Cells remaining rural	1,627

The detailed transition output and three summary matrices were written to HDFS and passed read-back validation.

3. Datasets still left out
A. Built-up surface

This is the main missing core dataset.

We still need to process built-up surface for:

1975
2020
eventually every historical epoch from 1975 through 2020

The 100 m built-up raster must be aggregated onto the same 1 km grid.

Likely output fields:

Plain Text
built_surface_total_m2
built_surface_fraction
built_valid_100m_pixels
built_coverage_fraction
nonresidential_built_surface_m2

Built-up surface is needed to distinguish:

population growth without proportional physical expansion;
outward land expansion;
densification of established built-up areas;
newly developed urban cells;
population growth relative to built-up-area growth.

This is explicitly part of the project’s intended population and physical urban-development analysis.

B. Intermediate historical epochs

We have processed only the endpoints:

Plain Text
1975
2020

Still unprocessed:

Plain Text
1980
1985
1990
1995
2000
2005
2010
2015

These intermediate years are essential because endpoint comparison alone cannot tell us when a transition happened.

For example, a cell classified as rural in 1975 and urban centre in 2020 could have transitioned in:

1985;
2000;
2015;
or any other intermediate epoch.

Processing all historical epochs will allow us to calculate the first year of urbanization and the sequence:

Plain Text
rural
→ urban cluster
→ urban centre

C. Building height

The inventory contains 15 building-height files, but building height has not entered the analytical pipeline.

Building height is a contextual layer, not a full historical time series comparable with population, built surface, and SMOD. It can later help characterize:

vertical versus horizontal development;
high-rise cores;
dense built-up corridors;
areas with high population but limited horizontal expansion.

It should not be added before built-up surface and the historical grid-year pipeline are stable.

D. 2025 and 2030 held-out epochs

The project planning discussed using historical epochs through 2020 for development and reserving 2025 and 2030 as later evaluation/test epochs.

Those years have not yet been processed. They should remain untouched until:

the historical pipeline is complete;
transition rules are defined;
growth or classification logic is finalized.

This avoids using held-out periods while designing the model or detection rules.

E. District or administrative-boundary dataset

No district boundary dataset has yet been joined.

Therefore, we do not yet have:

district names;
district-level population changes;
district-level built-up changes;
transition counts by district;
comparisons between Bengaluru Urban and surrounding districts.

The original proposal includes district-level summaries, so an authoritative administrative boundary layer still needs to be introduced and spatially joined with Sedona.

F. Other pilot cities

Only the Bengaluru pilot has been processed.

Still left:

Delhi NCR
Mumbai
Chennai

These should be robustness pilots after the complete population, SMOD, and built-up pipeline works for Bengaluru.

The final intended scope remains broader than these four cities. The city pilots validate the method before scaling across India.

G. India-wide production processing

We have not yet run the new production pipeline across all India shards.

The India-wide stage still requires:

Plain Text
one product
→ one epoch
→ one source shard
→ selected tiles
→ 1 km scalar Parquet
→ merge shard-level records

The current Bengaluru pipeline proves the method, but it is not yet the final India-wide dataset.

4. Current status by product
Dataset/product	1975	2020	Intermediate years	India-widePopulation	Complete for Bengaluru	Complete for Bengaluru	Not processed	Not processed
SMOD	Complete for Bengaluru	Complete for Bengaluru	Not processed	Not processed
Population + SMOD join	Complete	Complete	Not processed	Not processed
Endpoint transition analysis	Complete	Complete	Not applicable	Not processed
Built-up surface	Not processed	Not processed	Not processed	Not processed
Building height	Not used	Contextual only	No historical series	Not processed
District boundaries	Not joined	Not joined	Not joined	Not joined
2025/2030 evaluation	Held out	Held out	Not started	Not started
5. What the project currently proves

At this point, we have proven that the architecture works:

Plain Text
HDFS
→ Spark
→ Sedona raster processing
→ compact 1 km Parquet
→ Spark SQL joins
→ temporal transition analysis

We have also proven that:

raster processing can be restricted to city-intersecting tiles;
100 m population can be aggregated accurately to 1 km;
identical grid IDs persist across epochs and products;
population and SMOD can be joined one-to-one;
cell-level urban transitions can be calculated in Spark;
outputs can be persisted and recovered from HDFS.
6. Correct next step

The next product should be:

Plain Text
Bengaluru built-up surface, 1975

Then:

Plain Text
Bengaluru built-up surface, 2020

After both are validated, we join built-up surface into the existing endpoint grid-year datasets and calculate:

built-up area increase;
built-up percentage per cell;
population change per unit of built-up expansion;
physical expansion versus densification;
rural-to-urban transitions supported by built-up growth.

That completes the three essential GHSL components for the endpoint pilot:

Plain Text
Population
+ SMOD
+ Built-up surface

In [24]:
# ============================================================
# CELL 14: DISCOVER AND INSPECT BENGALURU BUILT SURFACE 1975
#
# Corrected catalog schema:
#   numBands -> band_count
#
# Purpose:
#   - Discover the built-surface product in the HDFS catalog
#   - Find the 1975 shard intersecting Bengaluru
#   - Read that GeoTIFF from HDFS
#   - Validate dimensions, resolution, bands and NoData
#   - Verify Sedona tiling and Bengaluru tile selection
#
# This cell is read-only.
# No HDFS output is written.
# No pixel arrays are extracted.
# No Python raster UDF, NumPy or Rasterio is used.
# ============================================================

import base64
import textwrap


cell14_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )

    EPOCH = 1975

    TILE_WIDTH = 256
    TILE_HEIGHT = 256

    EXPECTED_RESOLUTION = 100.0
    EXPECTED_PRODUCT = "built_surface"

    # Verified Bengaluru pilot extent in World Mollweide.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Built-Surface-Inspection"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 14: BENGALURU BUILT SURFACE 1975")
        print("=" * 80)
        print("Catalog:", HDFS_CATALOG)
        print("Epoch:", EPOCH)


        # ====================================================
        # READ EXISTING METADATA CATALOG FROM HDFS
        # ====================================================

        catalog_df = spark.read.parquet(
            HDFS_CATALOG
        )

        catalog_columns = set(
            catalog_df.columns
        )

        required_catalog_columns = {
            "temporal_value",
            "filename",
            "path",
            "fileSize",
            "width",
            "height",
            "numBands",
            "band_data_types",
            "band_nodata_values",
            "bands",
            "crs",
            "resolution_x",
            "resolution_y",
            "min_x",
            "min_y",
            "max_x",
            "max_y",
            "geoTransform",
            "product",
        }

        missing_catalog_columns = sorted(
            required_catalog_columns
            - catalog_columns
        )

        if missing_catalog_columns:
            raise RuntimeError(
                "Metadata catalog is missing required columns: "
                + ", ".join(missing_catalog_columns)
            )

        print("Catalog schema validation: PASS")


        # ====================================================
        # DISCOVER PRODUCTS CONTAINING "BUILT"
        # ====================================================

        built_product_inventory_df = (
            catalog_df
            .filter(
                F.lower(
                    F.col("product")
                ).contains("built")
            )
            .groupBy("product")
            .agg(
                F.count("*").alias(
                    "file_count"
                ),
                F.min(
                    F.col(
                        "temporal_value"
                    ).cast("int")
                ).alias(
                    "minimum_epoch"
                ),
                F.max(
                    F.col(
                        "temporal_value"
                    ).cast("int")
                ).alias(
                    "maximum_epoch"
                ),
            )
            .orderBy("product")
        )

        print()
        print("=" * 80)
        print("CATALOG PRODUCTS CONTAINING 'BUILT'")
        print("=" * 80)

        built_product_inventory_df.show(
            100,
            truncate=False,
        )

        built_product_rows = (
            built_product_inventory_df.collect()
        )

        if not built_product_rows:
            raise RuntimeError(
                "No catalog product contains 'built'."
            )

        discovered_products = {
            row["product"]
            for row in built_product_rows
        }

        if EXPECTED_PRODUCT not in discovered_products:
            raise RuntimeError(
                "Expected built-surface product was not found. "
                f"Expected '{EXPECTED_PRODUCT}', found "
                f"{sorted(discovered_products)}."
            )

        print(
            "Built-surface product discovered:",
            EXPECTED_PRODUCT,
        )


        # ====================================================
        # SELECT 1975 BUILT-SURFACE CATALOG RECORDS
        #
        # Correct catalog column:
        #   numBands
        #
        # It is aliased to:
        #   band_count
        # ====================================================

        built_1975_candidates_df = (
            catalog_df
            .filter(
                F.lower(
                    F.col("product")
                )
                == F.lit(EXPECTED_PRODUCT)
            )
            .filter(
                F.col(
                    "temporal_value"
                ).cast("int")
                == F.lit(EPOCH)
            )
            .select(
                F.col("product"),
                F.col("filename"),
                F.col("path"),
                F.col("fileSize"),
                F.col("width"),
                F.col("height"),

                F.col("numBands").alias(
                    "band_count"
                ),

                F.col("band_data_types"),
                F.col("band_nodata_values"),
                F.col("bands"),

                F.col("resolution_x"),
                F.col("resolution_y"),
                F.col("min_x"),
                F.col("min_y"),
                F.col("max_x"),
                F.col("max_y"),
                F.col("geoTransform"),
                F.col("crs"),
            )
        )

        candidate_count = (
            built_1975_candidates_df.count()
        )

        print()
        print("=" * 80)
        print("1975 BUILT-SURFACE CANDIDATES")
        print("=" * 80)
        print("Candidate records:", candidate_count)

        (
            built_1975_candidates_df
            .orderBy("filename")
            .show(
                100,
                truncate=False,
                vertical=True,
            )
        )

        if candidate_count != 15:
            raise RuntimeError(
                "Expected 15 built-surface shards for 1975, "
                f"found {candidate_count}."
            )

        built_1975_candidates_df.createOrReplaceTempView(
            "built_1975_catalog"
        )


        # ====================================================
        # SELECT THE SHARD INTERSECTING BENGALURU
        # ====================================================

        selected_source_df = spark.sql(
            f"""
            SELECT
                product,
                filename,
                path,
                fileSize,
                width,
                height,
                band_count,
                band_data_types,
                band_nodata_values,
                bands,
                resolution_x,
                resolution_y,
                min_x,
                min_y,
                max_x,
                max_y,
                geoTransform,
                crs

            FROM built_1975_catalog

            WHERE ST_Intersects(
                ST_PolygonFromEnvelope(
                    min_x,
                    min_y,
                    max_x,
                    max_y
                ),
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                )
            )

            ORDER BY filename
            """
        )

        selected_source_count = (
            selected_source_df.count()
        )

        print()
        print("=" * 80)
        print("BENGALURU-INTERSECTING BUILT-SURFACE SOURCES")
        print("=" * 80)
        print(
            "Selected source count:",
            selected_source_count,
        )

        selected_source_df.show(
            20,
            truncate=False,
            vertical=True,
        )

        if selected_source_count != 1:
            raise RuntimeError(
                "Expected exactly one Bengaluru built-surface "
                "source for 1975, found "
                f"{selected_source_count}."
            )

        selected_source = (
            selected_source_df.first()
        )

        source_path = selected_source["path"]

        print(
            "Selected product:",
            selected_source["product"],
        )

        print(
            "Selected filename:",
            selected_source["filename"],
        )

        print(
            "Selected HDFS path:",
            source_path,
        )


        # ====================================================
        # VALIDATE CATALOG METADATA BEFORE DECODING
        # ====================================================

        catalog_width = int(
            selected_source["width"]
        )

        catalog_height = int(
            selected_source["height"]
        )

        catalog_band_count = int(
            selected_source["band_count"]
        )

        catalog_resolution_x = float(
            selected_source["resolution_x"]
        )

        catalog_resolution_y = float(
            selected_source["resolution_y"]
        )

        if catalog_width <= 0:
            raise RuntimeError(
                "Catalog width is not positive."
            )

        if catalog_height <= 0:
            raise RuntimeError(
                "Catalog height is not positive."
            )

        if catalog_band_count < 1:
            raise RuntimeError(
                "Catalog reports no raster bands."
            )

        if (
            abs(
                catalog_resolution_x
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Catalog X resolution is not 100 m."
            )

        if (
            abs(
                abs(catalog_resolution_y)
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Catalog Y resolution is not 100 m."
            )

        print("Selected catalog metadata: PASS")


        # ====================================================
        # READ AND DECODE SELECTED GEOTIFF FROM HDFS
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(source_path)
            .select(
                F.col("path"),

                F.col("length").alias(
                    "source_file_bytes"
                ),

                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias(
                    "source_raster"
                ),
            )
        )

        source_count = source_df.count()

        if source_count != 1:
            raise RuntimeError(
                "Expected exactly one built-surface GeoTIFF, "
                f"found {source_count}."
            )

        source_df.createOrReplaceTempView(
            "built_surface_source"
        )


        # ====================================================
        # INSPECT DECODED RASTER METADATA
        # ====================================================

        metadata_df = spark.sql(
            """
            SELECT
                path,
                source_file_bytes,

                RS_Width(
                    source_raster
                ) AS source_width,

                RS_Height(
                    source_raster
                ) AS source_height,

                RS_NumBands(
                    source_raster
                ) AS source_band_count,

                RS_ScaleX(
                    source_raster
                ) AS source_scale_x,

                RS_ScaleY(
                    source_raster
                ) AS source_scale_y,

                RS_UpperLeftX(
                    source_raster
                ) AS source_upper_left_x,

                RS_UpperLeftY(
                    source_raster
                ) AS source_upper_left_y,

                ST_AsText(
                    RS_Envelope(
                        source_raster
                    )
                ) AS source_envelope_wkt

            FROM built_surface_source
            """
        )

        metadata = metadata_df.first()

        print()
        print("=" * 80)
        print("DECODED BUILT-SURFACE METADATA")
        print("=" * 80)

        metadata_df.show(
            truncate=False,
            vertical=True,
        )

        source_width = int(
            metadata["source_width"]
        )

        source_height = int(
            metadata["source_height"]
        )

        source_band_count = int(
            metadata["source_band_count"]
        )

        source_scale_x = float(
            metadata["source_scale_x"]
        )

        source_scale_y = float(
            metadata["source_scale_y"]
        )


        # ====================================================
        # CROSS-CHECK CATALOG AND DECODED METADATA
        # ====================================================

        if source_width != catalog_width:
            raise RuntimeError(
                "Catalog and decoded widths differ. "
                f"Catalog={catalog_width}, "
                f"decoded={source_width}."
            )

        if source_height != catalog_height:
            raise RuntimeError(
                "Catalog and decoded heights differ. "
                f"Catalog={catalog_height}, "
                f"decoded={source_height}."
            )

        if source_band_count != catalog_band_count:
            raise RuntimeError(
                "Catalog and decoded band counts differ. "
                f"Catalog={catalog_band_count}, "
                f"decoded={source_band_count}."
            )

        if (
            abs(
                source_scale_x
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Decoded X resolution is not 100 m."
            )

        if (
            abs(
                abs(source_scale_y)
                - EXPECTED_RESOLUTION
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Decoded Y resolution is not 100 m."
            )

        print(
            "Catalog-to-decoded metadata agreement: PASS"
        )


        # ====================================================
        # DISPLAY CATALOG BAND METADATA
        # ====================================================

        print()
        print("=" * 80)
        print("CATALOG BAND METADATA")
        print("=" * 80)

        print(
            "Catalog band count:",
            catalog_band_count,
        )

        print(
            "Catalog band data types:",
            selected_source["band_data_types"],
        )

        print(
            "Catalog band NoData values:",
            selected_source["band_nodata_values"],
        )

        print(
            "Catalog bands metadata:",
            selected_source["bands"],
        )


        # ====================================================
        # INSPECT SEDONA BAND NODATA METADATA
        # ====================================================

        print()
        print("=" * 80)
        print("SEDONA BAND NODATA METADATA")
        print("=" * 80)

        for band_index in range(
            1,
            source_band_count + 1,
        ):
            band_metadata_df = spark.sql(
                f"""
                SELECT
                    CAST(
                        {band_index}
                        AS INT
                    ) AS band_index,

                    RS_BandNoDataValue(
                        source_raster,
                        {band_index}
                    ) AS band_nodata_value

                FROM built_surface_source
                """
            )

            band_metadata_df.show(
                truncate=False,
                vertical=True,
            )


        # ====================================================
        # VERIFY SEDONA TILE LAYOUT
        #
        # Only band 1 is tiled for this structural inspection.
        # No pixel arrays are extracted.
        # ====================================================

        tile_columns = (
            source_width + TILE_WIDTH - 1
        ) // TILE_WIDTH

        tile_rows = (
            source_height + TILE_HEIGHT - 1
        ) // TILE_HEIGHT

        expected_tile_count = (
            tile_columns
            * tile_rows
        )

        tiles_df = spark.sql(
            f"""
            SELECT
                tile_index,
                tile

            FROM built_surface_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded
            AS tile_index, tile
            """
        )

        tile_count = tiles_df.count()

        if tile_count != expected_tile_count:
            raise RuntimeError(
                "Unexpected built-surface tile count. "
                f"Expected {expected_tile_count}, "
                f"found {tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "built_surface_tiles"
        )


        # ====================================================
        # COUNT BENGALURU-INTERSECTING TILES
        # ====================================================

        selected_tiles_df = spark.sql(
            f"""
            SELECT
                tile_index,

                RS_Width(
                    tile
                ) AS tile_width,

                RS_Height(
                    tile
                ) AS tile_height,

                RS_UpperLeftX(
                    tile
                ) AS tile_upper_left_x,

                RS_UpperLeftY(
                    tile
                ) AS tile_upper_left_y,

                RS_ScaleX(
                    tile
                ) AS tile_scale_x,

                RS_ScaleY(
                    tile
                ) AS tile_scale_y,

                ST_AsText(
                    RS_Envelope(tile)
                ) AS tile_envelope_wkt

            FROM built_surface_tiles

            WHERE ST_Intersects(
                RS_Envelope(tile),
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                )
            )

            ORDER BY tile_index
            """
        )

        selected_tile_count = (
            selected_tiles_df.count()
        )

        if selected_tile_count < 1:
            raise RuntimeError(
                "No built-surface tile intersects Bengaluru."
            )

        if selected_tile_count > 20:
            raise RuntimeError(
                "Unexpectedly many Bengaluru built-surface "
                f"tiles were selected: {selected_tile_count}."
            )


        print()
        print("=" * 80)
        print("BENGALURU-INTERSECTING TILE METADATA")
        print("=" * 80)

        selected_tiles_df.show(
            30,
            truncate=False,
            vertical=True,
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 14 REMOTE RESULT")
        print("=" * 80)

        print("Catalog schema validation: PASS")
        print("Built product discovery: PASS")
        print("1975 source discovery: PASS")
        print("Bengaluru shard selection: PASS")
        print("HDFS GeoTIFF read: PASS")
        print("Sedona raster decoding: PASS")
        print("Raster metadata validation: PASS")
        print("Band metadata inspection: PASS")
        print("Sedona tile-layout validation: PASS")

        print(
            "Selected product:",
            selected_source["product"],
        )

        print(
            "Selected file:",
            selected_source["filename"],
        )

        print(
            "Selected HDFS path:",
            source_path,
        )

        print(
            "Source width:",
            source_width,
        )

        print(
            "Source height:",
            source_height,
        )

        print(
            "Source bands:",
            source_band_count,
        )

        print(
            "Resolution X:",
            source_scale_x,
        )

        print(
            "Resolution Y:",
            source_scale_y,
        )

        print(
            "Tile layout:",
            f"{tile_columns} x {tile_rows}",
        )

        print(
            "Total source tiles:",
            tile_count,
        )

        print(
            "Bengaluru-intersecting tiles:",
            selected_tile_count,
        )

        print("HDFS output written: NO")
        print("Pixel values processed: NO")
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 14: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER REMOTE PROGRAM
# ============================================================

encoded_program = base64.b64encode(
    cell14_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell14_built_surface_1975_inspection.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "

    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "

    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=2 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


# ============================================================
# EXECUTE THROUGH remote_exec.sh
# ============================================================

result = run_remote(
    remote_command,
    timeout=1800,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 14 NOTEBOOK RESULT")
print("=" * 80)
print(
    "Remote return code:",
    result.returncode,
)
print("CELL 14: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 14: BENGALURU BUILT SURFACE 1975
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Epoch: 1975
Catalog schema validation: PASS

CATALOG PRODUCTS CONTAINING 'BUILT'
+-------------+----------+-------------+-------------+
|product      |file_count|minimum_epoch|maximum_epoch|
+-------------+----------+-------------+-------------+
|built_surface|150       |1975         |2020         |
+-------------+----------+-------------+-------------+

Built-surface product discovered: built_surface

1975 BUILT-SURFACE CANDIDATES
Candidate records: 15
-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Confirmed built-surface structure
Product: built_surface
Epoch: 1975
Bengaluru source: ghsl_india_built_surface_1975_100m_r2023a-0000016384-0000008192.tif
Dimensions: 8192 × 8192
Resolution: 100 m
Bands: 2
Data type: unsigned 16-bit for both bands
Declared NoData: none for both bands
Source tiles: 1,024
Bengaluru-intersecting tiles: 12
No data was written or pixel values processed.
Meaning of the two bands

The official GHS-BUILT-S R2023A product contains:

built_surface: total built-up surface in square metres per 100 m grid cell
built_surface_nres: built-up surface allocated to predominantly non-residential use, also in square metres per 100 m grid cell.

Therefore, the next aggregation should process both bands, not only band 1.

Next operation

For each 1 km grid cell, Apache Spark will aggregate the underlying 100 m values into:

Plain Text
built_surface_total_m2
built_surface_nres_m2
built_surface_residential_m2
built_surface_fraction
built_surface_percent
nres_share_of_built_surface
valid_100m_pixels
coverage_fraction

The derived residential surface will be:

Plain Text
built_surface_residential_m2
= built_surface_total_m2 - built_surface_nres_m2

We will validate that:

Plain Text
0 <= built_surface_nres_m2 <= built_surface_total_m2

and that a complete 1 km cell cannot exceed:

Plain Text
1,000,000 m²

The first processing cell should now be Cell 15: Bengaluru built-up surface 1975, 100 m to 1 km, using the same 12-tile Spark/Sedona pattern that already worked for population.

In [25]:
# ============================================================
# CELL 15: BENGALURU 1975 BUILT SURFACE, 100 M TO 1 KM
#
# Input:
#   GHS-BUILT-S 1975, 100 m, two bands
#
# Band 1:
#   Total built-up surface, square metres per 100 m cell
#
# Band 2:
#   Predominantly non-residential built-up surface,
#   square metres per 100 m cell
#
# Pipeline:
#   HDFS catalog
#   -> Spark source discovery
#   -> HDFS GeoTIFF
#   -> Sedona decoding
#   -> Sedona 256 x 256 tiling
#   -> Bengaluru tile selection
#   -> Sedona two-band array extraction
#   -> Spark SQL 100 m to 1 km aggregation
#   -> grid-key validation
#   -> HDFS Parquet write and read-back
#
# No Python raster UDF, NumPy, Rasterio, RS_Clip,
# or RS_AsGeoTiff.
# ============================================================

import base64
import textwrap


cell15_program = textwrap.dedent(
    r'''
    from sedona.spark import SedonaContext
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_CATALOG = (
        "hdfs://localhost:9000"
        "/data/ghsl/metadata/india/geotiff_catalog"
    )

    GRID_YEAR_REFERENCE_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "grid_year/epoch=1975"
    )

    OUTPUT_PATH = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru/"
        "product=built_surface/"
        "epoch=1975/"
        "grid_1km"
    )

    CITY = "bengaluru"
    PRODUCT = "built_surface"
    EPOCH = 1975

    SOURCE_WIDTH = 8192
    SOURCE_HEIGHT = 8192
    SOURCE_BANDS = 2

    SOURCE_RESOLUTION_M = 100.0
    TARGET_RESOLUTION_M = 1000.0

    SOURCE_PIXEL_AREA_M2 = 10000.0
    COMPLETE_GRID_AREA_M2 = 1000000.0

    TILE_WIDTH = 256
    TILE_HEIGHT = 256
    TILE_COLUMNS = SOURCE_WIDTH // TILE_WIDTH

    EXPECTED_SOURCE_TILES = 1024
    EXPECTED_SELECTED_TILES = 12
    EXPECTED_GRID_CELLS = 3658
    EXPECTED_COMPLETE_PIXELS = 100

    NUMERICAL_TOLERANCE = 1e-6

    # Verified Bengaluru pilot envelope, World Mollweide.
    PILOT_MIN_X = 7624177.350136818
    PILOT_MIN_Y = 1571479.656526347
    PILOT_MAX_X = 7683017.20414696
    PILOT_MAX_Y = 1632690.7573197454


    spark = None

    try:
        # ====================================================
        # CREATE SPARK + SEDONA CONTEXT
        # ====================================================

        config = (
            SedonaContext.builder()
            .appName(
                "GHSL-Bengaluru-1975-"
                "Built-Surface-100m-to-1km"
            )
            .getOrCreate()
        )

        spark = SedonaContext.create(config)
        spark.sparkContext.setLogLevel("WARN")

        print("=" * 80)
        print("CELL 15: BENGALURU 1975 BUILT SURFACE")
        print("=" * 80)
        print("Catalog:", HDFS_CATALOG)
        print("Grid reference:", GRID_YEAR_REFERENCE_PATH)
        print("Output:", OUTPUT_PATH)


        # ====================================================
        # DISCOVER THE ONE SOURCE SHARD
        # ====================================================

        catalog_df = (
            spark.read
            .parquet(HDFS_CATALOG)
            .filter(
                F.lower(F.col("product"))
                == F.lit(PRODUCT)
            )
            .filter(
                F.col("temporal_value").cast("int")
                == F.lit(EPOCH)
            )
        )

        catalog_count = catalog_df.count()

        if catalog_count != 15:
            raise RuntimeError(
                "Expected 15 built-surface records for 1975, "
                f"found {catalog_count}."
            )

        catalog_df.createOrReplaceTempView(
            "built_surface_catalog"
        )

        selected_sources_df = spark.sql(
            f"""
            SELECT
                filename,
                path,
                fileSize,
                width,
                height,
                numBands,
                resolution_x,
                resolution_y,
                min_x,
                min_y,
                max_x,
                max_y

            FROM built_surface_catalog

            WHERE ST_Intersects(
                ST_PolygonFromEnvelope(
                    min_x,
                    min_y,
                    max_x,
                    max_y
                ),
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                )
            )

            ORDER BY filename
            """
        )

        selected_source_count = selected_sources_df.count()

        if selected_source_count != 1:
            raise RuntimeError(
                "Expected exactly one Bengaluru source shard, "
                f"found {selected_source_count}."
            )

        selected_source = selected_sources_df.first()
        source_path = selected_source["path"]

        if (
            int(selected_source["width"]) != SOURCE_WIDTH
            or int(selected_source["height"]) != SOURCE_HEIGHT
            or int(selected_source["numBands"]) != SOURCE_BANDS
        ):
            raise RuntimeError(
                "Unexpected built-surface source structure."
            )

        print("Selected source:", source_path)


        # ====================================================
        # READ AND DECODE GEOTIFF FROM HDFS
        # ====================================================

        source_df = (
            spark.read
            .format("binaryFile")
            .load(source_path)
            .select(
                "path",
                F.col("length").alias("source_file_bytes"),
                F.expr(
                    "RS_FromGeoTiff(content)"
                ).alias("source_raster"),
            )
        )

        if source_df.count() != 1:
            raise RuntimeError(
                "Expected exactly one source GeoTIFF."
            )

        source_df.createOrReplaceTempView(
            "built_surface_source"
        )


        # ====================================================
        # VALIDATE DECODED METADATA
        # ====================================================

        metadata = spark.sql(
            """
            SELECT
                RS_Width(source_raster)
                    AS source_width,
                RS_Height(source_raster)
                    AS source_height,
                RS_NumBands(source_raster)
                    AS source_band_count,
                RS_ScaleX(source_raster)
                    AS source_scale_x,
                RS_ScaleY(source_raster)
                    AS source_scale_y
            FROM built_surface_source
            """
        ).first()

        if (
            int(metadata["source_width"]) != SOURCE_WIDTH
            or int(metadata["source_height"]) != SOURCE_HEIGHT
            or int(metadata["source_band_count"])
            != SOURCE_BANDS
        ):
            raise RuntimeError(
                "Decoded metadata validation failed."
            )

        if (
            abs(
                float(metadata["source_scale_x"])
                - SOURCE_RESOLUTION_M
            )
            > 1e-9
            or abs(
                abs(float(metadata["source_scale_y"]))
                - SOURCE_RESOLUTION_M
            )
            > 1e-9
        ):
            raise RuntimeError(
                "Built-surface resolution is not 100 m."
            )

        print("Decoded source metadata: PASS")


        # ====================================================
        # TILE BOTH BANDS TOGETHER
        #
        # Both bands remain aligned inside each tile.
        # ====================================================

        tiles_df = spark.sql(
            f"""
            SELECT
                path AS source_path,

                tile_index,

                CAST(
                    FLOOR(tile_index / {TILE_COLUMNS})
                    AS INT
                ) AS tile_row,

                CAST(
                    tile_index % {TILE_COLUMNS}
                    AS INT
                ) AS tile_col,

                tile

            FROM built_surface_source

            LATERAL VIEW posexplode(
                RS_Tile(
                    source_raster,
                    ARRAY(1, 2),
                    {TILE_WIDTH},
                    {TILE_HEIGHT},
                    false
                )
            ) exploded
            AS tile_index, tile
            """
        )

        source_tile_count = tiles_df.count()

        if source_tile_count != EXPECTED_SOURCE_TILES:
            raise RuntimeError(
                "Expected 1024 source tiles, found "
                f"{source_tile_count}."
            )

        tiles_df.createOrReplaceTempView(
            "built_surface_tiles"
        )


        # ====================================================
        # SELECT BENGALURU TILES
        # ====================================================

        selected_tiles_df = spark.sql(
            f"""
            SELECT
                source_path,
                tile_index,
                tile_row,
                tile_col,

                RS_Width(tile)
                    AS tile_width,

                RS_Height(tile)
                    AS tile_height,

                RS_NumBands(tile)
                    AS tile_band_count,

                RS_UpperLeftX(tile)
                    AS tile_upper_left_x,

                RS_UpperLeftY(tile)
                    AS tile_upper_left_y,

                RS_ScaleX(tile)
                    AS tile_scale_x,

                RS_ScaleY(tile)
                    AS tile_scale_y,

                RS_BandAsArray(
                    tile,
                    1
                ) AS total_built_array,

                RS_BandAsArray(
                    tile,
                    2
                ) AS nres_built_array

            FROM built_surface_tiles

            WHERE ST_Intersects(
                RS_Envelope(tile),
                ST_PolygonFromEnvelope(
                    {PILOT_MIN_X},
                    {PILOT_MIN_Y},
                    {PILOT_MAX_X},
                    {PILOT_MAX_Y}
                )
            )
            """
        )

        selected_tile_count = selected_tiles_df.count()

        if selected_tile_count != EXPECTED_SELECTED_TILES:
            raise RuntimeError(
                "Expected 12 Bengaluru tiles, found "
                f"{selected_tile_count}."
            )

        invalid_tile_count = (
            selected_tiles_df
            .filter(
                (F.col("tile_width") != TILE_WIDTH)
                | (F.col("tile_height") != TILE_HEIGHT)
                | (F.col("tile_band_count") != SOURCE_BANDS)
                | F.col("total_built_array").isNull()
                | F.col("nres_built_array").isNull()
                | (
                    F.size("total_built_array")
                    != TILE_WIDTH * TILE_HEIGHT
                )
                | (
                    F.size("nres_built_array")
                    != TILE_WIDTH * TILE_HEIGHT
                )
            )
            .count()
        )

        if invalid_tile_count != 0:
            raise RuntimeError(
                "Selected built-surface tiles failed validation."
            )

        print("Selected Bengaluru tiles:", selected_tile_count)

        selected_tiles_df.createOrReplaceTempView(
            "selected_built_tiles"
        )


        # ====================================================
        # EXPLODE BAND 1 AND RETRIEVE ALIGNED BAND 2 VALUE
        #
        # element_at uses one-based array indexing.
        # pixel_index from posexplode is zero-based.
        # ====================================================

        pixels_df = spark.sql(
            """
            SELECT
                source_path,
                tile_index,
                tile_row,
                tile_col,

                tile_width,
                tile_height,

                tile_upper_left_x,
                tile_upper_left_y,

                tile_scale_x,
                tile_scale_y,

                pixel_index,

                CAST(
                    FLOOR(pixel_index / tile_width)
                    AS INT
                ) AS local_pixel_row,

                CAST(
                    pixel_index % tile_width
                    AS INT
                ) AS local_pixel_col,

                CAST(
                    total_built_value
                    AS DOUBLE
                ) AS built_surface_total_m2_100m,

                CAST(
                    element_at(
                        nres_built_array,
                        pixel_index + 1
                    )
                    AS DOUBLE
                ) AS built_surface_nres_m2_100m

            FROM selected_built_tiles

            LATERAL VIEW posexplode(
                total_built_array
            ) exploded
            AS pixel_index, total_built_value
            """
        )

        pixels_df.createOrReplaceTempView(
            "built_surface_pixels"
        )


        # ====================================================
        # POSITION AND CLASSIFY PIXELS
        #
        # Valid range for each 100 m source pixel:
        #   0 to 10,000 square metres
        #
        # NRES must not exceed total built surface.
        # ====================================================

        classified_pixels_df = spark.sql(
            f"""
            SELECT
                source_path,

                (
                    tile_upper_left_x
                    + (
                        local_pixel_col + 0.5
                    ) * tile_scale_x
                ) AS pixel_center_x,

                (
                    tile_upper_left_y
                    + (
                        local_pixel_row + 0.5
                    ) * tile_scale_y
                ) AS pixel_center_y,

                built_surface_total_m2_100m,
                built_surface_nres_m2_100m,

                CASE
                    WHEN
                        built_surface_total_m2_100m
                            IS NOT NULL
                        AND NOT isnan(
                            built_surface_total_m2_100m
                        )
                        AND ABS(
                            built_surface_total_m2_100m
                        ) <= 1.7976931348623157E308
                        AND built_surface_total_m2_100m
                            >= 0.0
                        AND built_surface_total_m2_100m
                            <= {SOURCE_PIXEL_AREA_M2}
                    THEN true
                    ELSE false
                END AS is_valid_total,

                CASE
                    WHEN
                        built_surface_nres_m2_100m
                            IS NOT NULL
                        AND NOT isnan(
                            built_surface_nres_m2_100m
                        )
                        AND ABS(
                            built_surface_nres_m2_100m
                        ) <= 1.7976931348623157E308
                        AND built_surface_nres_m2_100m
                            >= 0.0
                        AND built_surface_nres_m2_100m
                            <= {SOURCE_PIXEL_AREA_M2}
                    THEN true
                    ELSE false
                END AS is_valid_nres

            FROM built_surface_pixels
            """
        )

        classified_pixels_df.createOrReplaceTempView(
            "classified_built_pixels"
        )


        # ====================================================
        # FILTER TO VALID BENGALURU PIXELS
        # ====================================================

        valid_pixels_df = spark.sql(
            f"""
            SELECT
                source_path,

                pixel_center_x,
                pixel_center_y,

                CAST(
                    FLOOR(
                        pixel_center_x
                        / {TARGET_RESOLUTION_M}
                    )
                    AS BIGINT
                ) AS grid_1km_col,

                CAST(
                    FLOOR(
                        pixel_center_y
                        / {TARGET_RESOLUTION_M}
                    )
                    AS BIGINT
                ) AS grid_1km_row,

                built_surface_total_m2_100m,
                built_surface_nres_m2_100m,

                (
                    built_surface_total_m2_100m
                    - built_surface_nres_m2_100m
                ) AS built_surface_residential_m2_100m

            FROM classified_built_pixels

            WHERE
                pixel_center_x >= {PILOT_MIN_X}
                AND pixel_center_x < {PILOT_MAX_X}
                AND pixel_center_y >= {PILOT_MIN_Y}
                AND pixel_center_y < {PILOT_MAX_Y}

                AND is_valid_total
                AND is_valid_nres

                AND built_surface_nres_m2_100m
                    <= (
                        built_surface_total_m2_100m
                        + {NUMERICAL_TOLERANCE}
                    )
            """
        )

        valid_pixels_df.createOrReplaceTempView(
            "valid_built_pixels"
        )


        # ====================================================
        # PIXEL-LEVEL QUALITY CHECKS
        # ====================================================

        pilot_pixel_count = valid_pixels_df.count()

        if pilot_pixel_count <= 0:
            raise RuntimeError(
                "No valid built-surface pixels were found."
            )

        invalid_relationship_count = (
            valid_pixels_df
            .filter(
                F.col(
                    "built_surface_nres_m2_100m"
                )
                > (
                    F.col(
                        "built_surface_total_m2_100m"
                    )
                    + F.lit(NUMERICAL_TOLERANCE)
                )
            )
            .count()
        )

        if invalid_relationship_count != 0:
            raise RuntimeError(
                "One or more NRES values exceed total built "
                "surface."
            )

        print("Valid Bengaluru pixels:", pilot_pixel_count)


        # ====================================================
        # AGGREGATE TO CANONICAL 1 KM GRID
        # ====================================================

        built_1km_df = spark.sql(
            f"""
            SELECT
                '{CITY}' AS city,
                '{PRODUCT}' AS product,
                CAST({EPOCH} AS INT) AS epoch,

                grid_1km_col,
                grid_1km_row,

                CONCAT(
                    'MOLL_1KM_',
                    CAST(grid_1km_col AS STRING),
                    '_',
                    CAST(grid_1km_row AS STRING)
                ) AS grid_cell_id,

                CAST(
                    grid_1km_col
                    * {TARGET_RESOLUTION_M}
                    AS DOUBLE
                ) AS grid_min_x,

                CAST(
                    (grid_1km_col + 1)
                    * {TARGET_RESOLUTION_M}
                    AS DOUBLE
                ) AS grid_max_x,

                CAST(
                    grid_1km_row
                    * {TARGET_RESOLUTION_M}
                    AS DOUBLE
                ) AS grid_min_y,

                CAST(
                    (grid_1km_row + 1)
                    * {TARGET_RESOLUTION_M}
                    AS DOUBLE
                ) AS grid_max_y,

                COUNT(*) AS valid_100m_pixels,

                COUNT(*) * {SOURCE_PIXEL_AREA_M2}
                    AS observed_grid_area_m2,

                COUNT(*) / 100.0
                    AS built_coverage_fraction,

                SUM(
                    built_surface_total_m2_100m
                ) AS built_surface_total_m2,

                SUM(
                    built_surface_nres_m2_100m
                ) AS built_surface_nres_m2,

                SUM(
                    built_surface_residential_m2_100m
                ) AS built_surface_residential_m2,

                SUM(
                    CASE
                        WHEN built_surface_total_m2_100m > 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS built_100m_pixels,

                SUM(
                    CASE
                        WHEN built_surface_nres_m2_100m > 0.0
                        THEN 1
                        ELSE 0
                    END
                ) AS nres_100m_pixels,

                MIN(source_path) AS source_path,

                CURRENT_TIMESTAMP()
                    AS processed_at_utc

            FROM valid_built_pixels

            GROUP BY
                grid_1km_col,
                grid_1km_row
            """
        )


        # ====================================================
        # DERIVED FRACTIONS
        #
        # observed_grid_area_m2 handles partial pilot-boundary
        # cells correctly.
        # ====================================================

        built_1km_df = (
            built_1km_df
            .withColumn(
                "built_surface_fraction",
                F.col("built_surface_total_m2")
                / F.col("observed_grid_area_m2"),
            )
            .withColumn(
                "built_surface_percent",
                F.col("built_surface_fraction")
                * F.lit(100.0),
            )
            .withColumn(
                "built_surface_nres_fraction",
                F.col("built_surface_nres_m2")
                / F.col("observed_grid_area_m2"),
            )
            .withColumn(
                "built_surface_residential_fraction",
                F.col("built_surface_residential_m2")
                / F.col("observed_grid_area_m2"),
            )
            .withColumn(
                "nres_share_of_built_surface",
                F.when(
                    F.col("built_surface_total_m2")
                    > F.lit(0.0),
                    F.col("built_surface_nres_m2")
                    / F.col("built_surface_total_m2"),
                ).otherwise(F.lit(0.0)),
            )
            .withColumn(
                "is_complete_built_cell",
                F.col("valid_100m_pixels")
                == F.lit(EXPECTED_COMPLETE_PIXELS),
            )
            .withColumn(
                "built_quality_flag",
                F.when(
                    F.col("valid_100m_pixels")
                    == F.lit(EXPECTED_COMPLETE_PIXELS),
                    F.lit("complete"),
                ).otherwise(
                    F.lit("partial_pilot_boundary")
                ),
            )
        )


        # ====================================================
        # OUTPUT VALIDATION
        # ====================================================

        output_count = built_1km_df.count()

        if output_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected built-surface grid count. "
                f"Expected {EXPECTED_GRID_CELLS}, "
                f"found {output_count}."
            )

        duplicate_count = (
            built_1km_df
            .groupBy("grid_cell_id", "epoch")
            .count()
            .filter(F.col("count") != 1)
            .count()
        )

        if duplicate_count != 0:
            raise RuntimeError(
                "Duplicate built-surface grid cells produced."
            )

        invalid_output_count = (
            built_1km_df
            .filter(
                F.col("built_surface_total_m2").isNull()
                | F.col("built_surface_nres_m2").isNull()
                | F.col(
                    "built_surface_residential_m2"
                ).isNull()
                | F.isnan("built_surface_total_m2")
                | F.isnan("built_surface_nres_m2")
                | (
                    F.col("built_surface_total_m2")
                    < F.lit(0.0)
                )
                | (
                    F.col("built_surface_nres_m2")
                    < F.lit(0.0)
                )
                | (
                    F.col("built_surface_residential_m2")
                    < F.lit(-NUMERICAL_TOLERANCE)
                )
                | (
                    F.col("built_surface_nres_m2")
                    > (
                        F.col("built_surface_total_m2")
                        + F.lit(NUMERICAL_TOLERANCE)
                    )
                )
                | (
                    F.col("built_surface_total_m2")
                    > (
                        F.col("observed_grid_area_m2")
                        + F.lit(NUMERICAL_TOLERANCE)
                    )
                )
                | (
                    F.col("built_surface_fraction")
                    < F.lit(0.0)
                )
                | (
                    F.col("built_surface_fraction")
                    > F.lit(1.0 + 1e-9)
                )
                | (
                    F.col("nres_share_of_built_surface")
                    < F.lit(0.0)
                )
                | (
                    F.col("nres_share_of_built_surface")
                    > F.lit(1.0 + 1e-9)
                )
            )
            .count()
        )

        if invalid_output_count != 0:
            raise RuntimeError(
                "One or more aggregated cells failed "
                "built-surface validation."
            )


        # ====================================================
        # CONSERVATION CHECKS
        # ====================================================

        pixel_totals = valid_pixels_df.agg(
            F.sum(
                "built_surface_total_m2_100m"
            ).alias("total_built"),
            F.sum(
                "built_surface_nres_m2_100m"
            ).alias("nres_built"),
            F.sum(
                "built_surface_residential_m2_100m"
            ).alias("residential_built"),
        ).first()

        grid_totals = built_1km_df.agg(
            F.sum(
                "built_surface_total_m2"
            ).alias("total_built"),
            F.sum(
                "built_surface_nres_m2"
            ).alias("nres_built"),
            F.sum(
                "built_surface_residential_m2"
            ).alias("residential_built"),
        ).first()

        for metric in [
            "total_built",
            "nres_built",
            "residential_built",
        ]:
            source_total = float(pixel_totals[metric])
            aggregated_total = float(grid_totals[metric])

            allowed_difference = max(
                1e-6,
                abs(source_total) * 1e-12,
            )

            if (
                abs(source_total - aggregated_total)
                > allowed_difference
            ):
                raise RuntimeError(
                    f"Conservation failed for {metric}. "
                    f"Source={source_total}, "
                    f"aggregated={aggregated_total}."
                )

        print("Built-surface conservation: PASS")


        # ====================================================
        # VERIFY EXACT GRID ALIGNMENT WITH 1975 GRID-YEAR
        # ====================================================

        reference_df = (
            spark.read
            .parquet(GRID_YEAR_REFERENCE_PATH)
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "grid_min_x",
                "grid_max_x",
                "grid_min_y",
                "grid_max_y",
            )
        )

        reference_count = reference_df.count()

        if reference_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 1975 grid-year reference count."
            )

        missing_built_keys = (
            reference_df
            .select("grid_cell_id")
            .join(
                built_1km_df.select("grid_cell_id"),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        extra_built_keys = (
            built_1km_df
            .select("grid_cell_id")
            .join(
                reference_df.select("grid_cell_id"),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        if missing_built_keys != 0 or extra_built_keys != 0:
            raise RuntimeError(
                "Built-surface grid keys do not match the "
                "1975 analytical grid."
            )

        coordinate_mismatch_count = (
            built_1km_df.alias("b")
            .join(
                reference_df.alias("r"),
                on="grid_cell_id",
                how="inner",
            )
            .filter(
                (
                    F.col("b.grid_1km_col")
                    != F.col("r.grid_1km_col")
                )
                | (
                    F.col("b.grid_1km_row")
                    != F.col("r.grid_1km_row")
                )
                | (
                    F.abs(
                        F.col("b.grid_min_x")
                        - F.col("r.grid_min_x")
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col("b.grid_min_y")
                        - F.col("r.grid_min_y")
                    )
                    > F.lit(1e-9)
                )
            )
            .count()
        )

        if coordinate_mismatch_count != 0:
            raise RuntimeError(
                "Built-surface grid coordinates differ from "
                "the 1975 analytical grid."
            )

        print("1975 analytical-grid alignment: PASS")


        # ====================================================
        # DISPLAY SUMMARY
        # ====================================================

        summary_df = built_1km_df.agg(
            F.count("*").alias("grid_cell_count"),

            F.sum(
                F.when(
                    F.col("is_complete_built_cell"),
                    F.lit(1),
                ).otherwise(F.lit(0))
            ).alias("complete_cells"),

            F.sum(
                F.when(
                    ~F.col("is_complete_built_cell"),
                    F.lit(1),
                ).otherwise(F.lit(0))
            ).alias("partial_cells"),

            F.sum(
                "built_surface_total_m2"
            ).alias("built_surface_total_m2"),

            F.sum(
                "built_surface_nres_m2"
            ).alias("built_surface_nres_m2"),

            F.sum(
                "built_surface_residential_m2"
            ).alias(
                "built_surface_residential_m2"
            ),

            F.avg(
                "built_surface_fraction"
            ).alias("average_built_fraction"),

            F.max(
                "built_surface_fraction"
            ).alias("maximum_built_fraction"),
        )

        print()
        print("=" * 80)
        print("1975 BUILT-SURFACE SUMMARY")
        print("=" * 80)

        summary_df.show(
            truncate=False,
            vertical=True,
        )

        print()
        print("=" * 80)
        print("TOP 25 BUILT-UP CELLS")
        print("=" * 80)

        (
            built_1km_df
            .select(
                "grid_cell_id",
                "grid_1km_col",
                "grid_1km_row",
                "valid_100m_pixels",
                "built_surface_total_m2",
                "built_surface_nres_m2",
                "built_surface_residential_m2",
                "built_surface_fraction",
                "built_surface_percent",
                "nres_share_of_built_surface",
                "built_quality_flag",
            )
            .orderBy(
                F.col(
                    "built_surface_total_m2"
                ).desc()
            )
            .show(
                25,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE HDFS PARQUET
        # ====================================================

        (
            built_1km_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(OUTPUT_PATH)
        )

        print("HDFS 1975 built-surface write: COMPLETE")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        readback_df = spark.read.parquet(
            OUTPUT_PATH
        )

        readback_count = readback_df.count()

        if readback_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Built-surface read-back count mismatch."
            )

        readback_totals = readback_df.agg(
            F.sum(
                "built_surface_total_m2"
            ).alias("total_built"),
            F.sum(
                "built_surface_nres_m2"
            ).alias("nres_built"),
            F.sum(
                "built_surface_residential_m2"
            ).alias("residential_built"),
        ).first()

        for metric in [
            "total_built",
            "nres_built",
            "residential_built",
        ]:
            expected_value = float(grid_totals[metric])
            readback_value = float(readback_totals[metric])

            allowed_difference = max(
                1e-6,
                abs(expected_value) * 1e-12,
            )

            if (
                abs(expected_value - readback_value)
                > allowed_difference
            ):
                raise RuntimeError(
                    f"HDFS read-back changed {metric}."
                )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        final_summary = summary_df.first()

        print()
        print("=" * 80)
        print("CELL 15 REMOTE RESULT")
        print("=" * 80)

        print("1975 built-surface discovery: PASS")
        print("HDFS GeoTIFF read: PASS")
        print("Sedona two-band decoding: PASS")
        print("Sedona two-band tiling: PASS")
        print("Sedona band-array extraction: PASS")
        print("Spark SQL pixel validation: PASS")
        print("Spark SQL 100 m to 1 km aggregation: PASS")
        print("Built-surface conservation: PASS")
        print("1975 analytical-grid alignment: PASS")
        print("HDFS Parquet write: PASS")
        print("HDFS Parquet read-back: PASS")

        print(
            "Output grid cells:",
            readback_count,
        )

        print(
            "Complete built cells:",
            final_summary["complete_cells"],
        )

        print(
            "Partial boundary cells:",
            final_summary["partial_cells"],
        )

        print(
            "Total built surface m2:",
            final_summary[
                "built_surface_total_m2"
            ],
        )

        print(
            "Non-residential built surface m2:",
            final_summary[
                "built_surface_nres_m2"
            ],
        )

        print(
            "Residential built surface m2:",
            final_summary[
                "built_surface_residential_m2"
            ],
        )

        print("Output:", OUTPUT_PATH)
        print("Python raster UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 15: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

encoded_program = base64.b64encode(
    cell15_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell15_built_surface_1975.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=3600,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 15 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 15: SUCCESS")

Remote Python syntax: PASS
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
CELL 15: BENGALURU 1975 BUILT SURFACE
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Grid reference: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1975/grid_1km
Selected source: hdfs://localhost:9000/data/ghsl/raw/india/built_surface/epoch=1975/ghsl_india_built_surface_1975_100m_r2023a-0000016384-0000008192.tif
Decoded source metadata: PASS
Selected Bengaluru tiles: 12
Valid Bengaluru pixels: 359856
Built-surface conservation: PASS
1975 analytical-grid alignment: PASS

1975 BUILT-SURFACE SUMMARY
-RECORD 0-------------------------------------------
 grid_cell_count              | 3658                
 complete_cells               | 3480           

The 1975 built-surface dataset is now aggregated to the same 3,658-cell 1 km grid and persisted in HDFS.

1975 built-surface result
Metric	ValueGrid cells	3,658
Complete cells	3,480
Partial boundary cells	178
Valid 100 m pixels	359,856
Total built surface	67.09 km²
Residential built surface	60.54 km²
Non-residential built surface	6.54 km²
Average built fraction per cell	1.84%
Maximum built fraction in one cell	61.59%

The stored values were 67,085,621 m² total, 60,540,667 m² residential, and 6,544,954 m² non-residential.

Approximately 90.2% of the 1975 built surface was residential and 9.8% was predominantly non-residential. This split is derived directly from the two-band totals.

Validation status

All important checks passed:

two-band raster decoding and tiling;
12-tile Bengaluru selection;
two-band array extraction;
pixel validation;
100 m to 1 km aggregation;
total, residential, and non-residential conservation;
exact alignment with the existing 1975 analytical grid;
HDFS Parquet write and read-back.

The process used Apache Spark, Apache Sedona, and HDFS. It did not use a Python raster UDF, NumPy, or Rasterio.

Persistent output
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1975/grid_1km

Next step

Now process built surface 2020 with the same validated logic. The only conceptual changes are:

Plain Text
EPOCH = 2020
Grid reference = grid_year/epoch=2020
Output = product=built_surface/epoch=2020/grid_1km

After that succeeds, we will join built surface into both grid-year datasets and calculate physical expansion versus population intensification.

In [26]:
# ============================================================
# CELL 16: BENGALURU 2020 BUILT SURFACE, 100 M TO 1 KM
#
# Reuses the validated Cell 15 Spark/Sedona implementation.
#
# Changes:
#   - epoch: 1975 -> 2020
#   - analytical grid reference: 2020
#   - output path: built_surface/epoch=2020/grid_1km
#
# No Python raster UDF, NumPy, Rasterio, RS_Clip,
# or RS_AsGeoTiff.
# ============================================================

import base64
import subprocess
import textwrap


# ------------------------------------------------------------
# Existing validated Cell 15 program on AWS
# ------------------------------------------------------------

REMOTE_CELL15_SCRIPT = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell15_built_surface_1975.py"
)

REMOTE_CELL16_SCRIPT = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell16_built_surface_2020.py"
)

REMOTE_LOG = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell16_built_surface_2020.log"
)


# ------------------------------------------------------------
# Python patch program
#
# It reads the validated Cell 15 source and makes only the
# controlled epoch-specific replacements required for 2020.
# ------------------------------------------------------------

patch_program = textwrap.dedent(
    f'''
    from pathlib import Path


    source_path = Path(
        "{REMOTE_CELL15_SCRIPT}"
    )

    target_path = Path(
        "{REMOTE_CELL16_SCRIPT}"
    )


    if not source_path.is_file():
        raise FileNotFoundError(
            f"Validated Cell 15 remote script not found: "
            f"{{source_path}}"
        )


    program = source_path.read_text(
        encoding="utf-8"
    )


    # --------------------------------------------------------
    # Pre-patch validation
    # --------------------------------------------------------

    required_1975_fragments = [
        'EPOCH = 1975',
        'grid_year/epoch=1975',
        '"epoch=1975/"',
        '"GHSL-Bengaluru-1975-"',
        '"CELL 15: BENGALURU 1975 BUILT SURFACE"',
        '"CELL 15 REMOTE RESULT"',
        '"CELL 15: SUCCESS"',
    ]

    missing_fragments = [
        fragment
        for fragment in required_1975_fragments
        if fragment not in program
    ]

    if missing_fragments:
        raise RuntimeError(
            "The validated Cell 15 source does not contain "
            "the expected markers:\\n"
            + "\\n".join(missing_fragments)
        )


    # --------------------------------------------------------
    # Controlled replacements
    # --------------------------------------------------------

    replacements = {{
        'EPOCH = 1975':
            'EPOCH = 2020',

        'grid_year/epoch=1975':
            'grid_year/epoch=2020',

        '"epoch=1975/"':
            '"epoch=2020/"',

        '"GHSL-Bengaluru-1975-"':
            '"GHSL-Bengaluru-2020-"',

        '"CELL 15: BENGALURU 1975 BUILT SURFACE"':
            '"CELL 16: BENGALURU 2020 BUILT SURFACE"',

        '"1975 built-surface discovery: PASS"':
            '"2020 built-surface discovery: PASS"',

        '"1975 analytical-grid alignment: PASS"':
            '"2020 analytical-grid alignment: PASS"',

        '"HDFS 1975 built-surface write: COMPLETE"':
            '"HDFS 2020 built-surface write: COMPLETE"',

        '"1975 BUILT-SURFACE SUMMARY"':
            '"2020 BUILT-SURFACE SUMMARY"',

        '"CELL 15 REMOTE RESULT"':
            '"CELL 16 REMOTE RESULT"',

        '"CELL 15: SUCCESS"':
            '"CELL 16: SUCCESS"',
    }}


    replacement_counts = {{}}

    for old_text, new_text in replacements.items():
        replacement_counts[old_text] = (
            program.count(old_text)
        )

        if replacement_counts[old_text] < 1:
            raise RuntimeError(
                "Expected replacement marker was absent: "
                f"{{old_text}}"
            )

        program = program.replace(
            old_text,
            new_text,
        )


    # --------------------------------------------------------
    # Ensure no active 1975 configuration remains
    #
    # Comments mentioning Cell 15 are acceptable, but active
    # paths and constants must now refer to 2020.
    # --------------------------------------------------------

    forbidden_fragments = [
        'EPOCH = 1975',
        'grid_year/epoch=1975',
        '"epoch=1975/"',
        '"GHSL-Bengaluru-1975-"',
    ]

    remaining_forbidden = [
        fragment
        for fragment in forbidden_fragments
        if fragment in program
    ]

    if remaining_forbidden:
        raise RuntimeError(
            "Active 1975 configuration remains after patch:\\n"
            + "\\n".join(remaining_forbidden)
        )


    required_2020_fragments = [
        'EPOCH = 2020',
        'grid_year/epoch=2020',
        '"epoch=2020/"',
        '"GHSL-Bengaluru-2020-"',
        '"CELL 16: BENGALURU 2020 BUILT SURFACE"',
        '"CELL 16 REMOTE RESULT"',
        '"CELL 16: SUCCESS"',
    ]

    missing_2020_fragments = [
        fragment
        for fragment in required_2020_fragments
        if fragment not in program
    ]

    if missing_2020_fragments:
        raise RuntimeError(
            "Generated Cell 16 source is missing required "
            "2020 markers:\\n"
            + "\\n".join(missing_2020_fragments)
        )


    # --------------------------------------------------------
    # Compile before saving
    # --------------------------------------------------------

    compile(
        program,
        str(target_path),
        "exec",
    )


    target_path.write_text(
        program,
        encoding="utf-8",
    )


    print("Source script:", source_path)
    print("Generated script:", target_path)
    print(
        "Generated size:",
        target_path.stat().st_size,
        "bytes",
    )
    print("2020 configuration validation: PASS")
    print("Generated Python syntax: PASS")
    '''
).strip() + "\n"


encoded_patch = base64.b64encode(
    patch_program.encode("utf-8")
).decode("ascii")


# ------------------------------------------------------------
# Generate the Cell 16 script and validate it remotely
# ------------------------------------------------------------

generate_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_patch}' "
    "| base64 -d "
    "> /home/ubuntu/"
    "generate_ghsl_cell16_built_surface_2020.py; "

    "python3 "
    "/home/ubuntu/"
    "generate_ghsl_cell16_built_surface_2020.py; "

    f"python3 -m py_compile {REMOTE_CELL16_SCRIPT}; "

    "echo 'Remote Cell 16 syntax: PASS'"
)


generate_result = run_remote(
    generate_command,
    timeout=180,
    show_output=True,
)


# ------------------------------------------------------------
# Execute the generated Spark/Sedona program
# ------------------------------------------------------------

run_command = (
    "set -o pipefail; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 3g "

    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.sql.codegen.wholeStage=false "
    "--conf spark.driver.maxResultSize=256m "

    "--packages "
    "org.apache.sedona:"
    "sedona-spark-3.5_2.12:1.9.1,"
    "org.datasyslab:"
    "geotools-wrapper:1.9.1-33.5 "

    "--repositories "
    "https://artifacts.unidata.ucar.edu/"
    "repository/unidata-all "

    f"{REMOTE_CELL16_SCRIPT} "

    "2>&1 "
    f"| tee {REMOTE_LOG}; "

    "status=${PIPESTATUS[0]}; "
    "echo SPARK_EXIT_CODE=${status}; "
    "exit ${status}"
)


result = run_remote(
    run_command,
    timeout=3600,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 16 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("Remote log:", REMOTE_LOG)
print("CELL 16: SUCCESS")

Source script: /home/ubuntu/ghsl_notebook04_cell15_built_surface_1975.py
Generated script: /home/ubuntu/ghsl_notebook04_cell16_built_surface_2020.py
Generated size: 31136 bytes
2020 configuration validation: PASS
Generated Python syntax: PASS
Remote Cell 16 syntax: PASS

REMOTE STDERR:

https://artifacts.unidata.ucar.edu/repository/unidata-all added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/home/ubuntu/ghsl-sedona-env/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c7ab253-3917-4f16-8291-4a1edd108c2f;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.9.1 in central
	found org.apache

Cell 16 succeeded completely. The 2020 built-surface output is validated, aligned to the same 3,658-cell analytical grid, and persisted in HDFS.

2020 built-surface result
Metric	ValueGrid cells	3,658
Complete cells	3,480
Partial boundary cells	178
Valid 100 m pixels	359,856
Total built surface	342.43 km²
Residential built surface	314.11 km²
Non-residential built surface	28.31 km²
Average built fraction per cell	9.42%
Maximum built fraction in one cell	63.06%

The exact totals are 342,426,224 m² total, 314,112,965 m² residential, and 28,313,259 m² non-residential.

Preliminary 1975 to 2020 built-up change

Using the validated endpoint totals:

Metric	1975	2020	ChangeTotal built surface	67.09 km²	342.43 km²	+275.34 km²
Residential	60.54 km²	314.11 km²	+253.57 km²
Non-residential	6.54 km²	28.31 km²	+21.77 km²

Total built surface increased by approximately 410.4%, or 5.10×. Population increased by approximately 377.4%, or 4.77×. This suggests that physical built-up expansion slightly outpaced population growth over the full pilot area, but the correct conclusion must come from cell-level analysis because some cells densified while others expanded spatially.

The non-residential share decreased slightly:

1975: approximately 9.76%
2020: approximately 8.27%

This means residential built surface expanded somewhat faster than non-residential built surface across the pilot extent.

Persistent output
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=2020/grid_1km


Next step

Now join built surface into both endpoint grid-year datasets:

Plain Text
1975 grid_year + built_surface 1975
2020 grid_year + built_surface 2020

The enriched records will contain:

Plain Text
population
SMOD class
settlement group
total built surface
residential built surface
non-residential built surface
built fraction
people per built hectare
quality flags

Then the following cell can classify each 1 km cell as:

physical expansion;
population densification;
combined growth;
stable;
decline or anomalous transition.

The thresholds should be calculated from the cell-level change distributions rather than arbitrarily assigned.

In [27]:
# ============================================================
# CELL 17: ENRICH 1975 AND 2020 GRID-YEAR DATASETS
#          WITH BUILT SURFACE
#
# Inputs for each epoch:
#   - population + SMOD grid-year Parquet
#   - built-surface 1 km Parquet
#
# Outputs:
#   - enriched_grid_year/epoch=1975
#   - enriched_grid_year/epoch=2020
#
# Derived metrics:
#   - total/residential/non-residential built surface
#   - built fraction and percentage
#   - population per built hectare
#   - population per residential built hectare
#   - people per built 100 m²
#
# Primary tools:
#   - HDFS
#   - Apache Spark DataFrames and Spark SQL
#
# No GeoTIFF is reread.
# No Python UDF, NumPy, Rasterio, or Sedona raster operation.
# ============================================================

import base64
import textwrap


cell17_program = textwrap.dedent(
    r'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    EPOCHS = [1975, 2020]

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_COMPLETE_CELLS = 3480
    EXPECTED_PARTIAL_CELLS = 178

    EXPECTED_TOTALS = {
        1975: {
            "population_total": 3060319.0921731493,
            "built_surface_total_m2": 67085621.0,
            "built_surface_nres_m2": 6544954.0,
            "built_surface_residential_m2": 60540667.0,
        },
        2020: {
            "population_total": 14608885.859992506,
            "built_surface_total_m2": 342426224.0,
            "built_surface_nres_m2": 28313259.0,
            "built_surface_residential_m2": 314112965.0,
        },
    }

    NUMERICAL_TOLERANCE = 1e-6


    spark = None

    try:
        # ====================================================
        # CREATE SPARK SESSION
        #
        # All inputs are compact scalar Parquet datasets.
        # ====================================================

        spark = (
            SparkSession.builder
            .appName(
                "GHSL-Bengaluru-Endpoint-"
                "Built-Surface-Enrichment"
            )
            .getOrCreate()
        )

        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 17: ENRICH ENDPOINT GRID-YEAR DATASETS")
        print("=" * 80)
        print("Epochs:", EPOCHS)


        # ====================================================
        # PROCESS ONE EPOCH AT A TIME
        # ====================================================

        epoch_results = []

        for epoch in EPOCHS:
            grid_year_path = (
                f"{HDFS_BASE}/grid_year/epoch={epoch}"
            )

            built_surface_path = (
                f"{HDFS_BASE}/product=built_surface/"
                f"epoch={epoch}/grid_1km"
            )

            output_path = (
                f"{HDFS_BASE}/enriched_grid_year/"
                f"epoch={epoch}"
            )

            print()
            print("=" * 80)
            print(f"PROCESSING EPOCH {epoch}")
            print("=" * 80)
            print("Grid-year input:", grid_year_path)
            print("Built-surface input:", built_surface_path)
            print("Output:", output_path)


            # ================================================
            # READ INPUTS FROM HDFS
            # ================================================

            grid_year_raw_df = spark.read.parquet(
                grid_year_path
            )

            built_raw_df = spark.read.parquet(
                built_surface_path
            )


            # ================================================
            # NORMALIZE BUILT-SURFACE COLUMNS
            #
            # Grid-year columns are retained unchanged.
            # ================================================

            built_df = (
                built_raw_df
                .select(
                    F.col("grid_cell_id")
                    .cast("string")
                    .alias("grid_cell_id"),

                    F.col("epoch")
                    .cast("int")
                    .alias("epoch"),

                    F.col("grid_1km_col")
                    .cast("long")
                    .alias("built_grid_col"),

                    F.col("grid_1km_row")
                    .cast("long")
                    .alias("built_grid_row"),

                    F.col("grid_min_x")
                    .cast("double")
                    .alias("built_grid_min_x"),

                    F.col("grid_max_x")
                    .cast("double")
                    .alias("built_grid_max_x"),

                    F.col("grid_min_y")
                    .cast("double")
                    .alias("built_grid_min_y"),

                    F.col("grid_max_y")
                    .cast("double")
                    .alias("built_grid_max_y"),

                    F.col("valid_100m_pixels")
                    .cast("long")
                    .alias(
                        "built_valid_100m_pixels"
                    ),

                    F.col("observed_grid_area_m2")
                    .cast("double")
                    .alias(
                        "built_observed_grid_area_m2"
                    ),

                    F.col("built_coverage_fraction")
                    .cast("double")
                    .alias(
                        "built_coverage_fraction"
                    ),

                    F.col("built_surface_total_m2")
                    .cast("double")
                    .alias(
                        "built_surface_total_m2"
                    ),

                    F.col("built_surface_nres_m2")
                    .cast("double")
                    .alias(
                        "built_surface_nres_m2"
                    ),

                    F.col(
                        "built_surface_residential_m2"
                    )
                    .cast("double")
                    .alias(
                        "built_surface_residential_m2"
                    ),

                    F.col("built_surface_fraction")
                    .cast("double")
                    .alias(
                        "built_surface_fraction"
                    ),

                    F.col("built_surface_percent")
                    .cast("double")
                    .alias(
                        "built_surface_percent"
                    ),

                    F.col(
                        "built_surface_nres_fraction"
                    )
                    .cast("double")
                    .alias(
                        "built_surface_nres_fraction"
                    ),

                    F.col(
                        "built_surface_residential_fraction"
                    )
                    .cast("double")
                    .alias(
                        "built_surface_residential_fraction"
                    ),

                    F.col(
                        "nres_share_of_built_surface"
                    )
                    .cast("double")
                    .alias(
                        "nres_share_of_built_surface"
                    ),

                    F.col("built_100m_pixels")
                    .cast("long")
                    .alias("built_100m_pixels"),

                    F.col("nres_100m_pixels")
                    .cast("long")
                    .alias("nres_100m_pixels"),

                    F.col("is_complete_built_cell")
                    .cast("boolean")
                    .alias(
                        "is_complete_built_cell"
                    ),

                    F.col("built_quality_flag")
                    .cast("string")
                    .alias("built_quality_flag"),

                    F.col("source_path")
                    .alias(
                        "built_surface_source_path"
                    ),
                )
            )


            # ================================================
            # INPUT VALIDATION
            # ================================================

            grid_count = grid_year_raw_df.count()
            built_count = built_df.count()

            if grid_count != EXPECTED_GRID_CELLS:
                raise RuntimeError(
                    f"Epoch {epoch}: expected "
                    f"{EXPECTED_GRID_CELLS} grid-year rows, "
                    f"found {grid_count}."
                )

            if built_count != EXPECTED_GRID_CELLS:
                raise RuntimeError(
                    f"Epoch {epoch}: expected "
                    f"{EXPECTED_GRID_CELLS} built rows, "
                    f"found {built_count}."
                )


            grid_duplicate_count = (
                grid_year_raw_df
                .groupBy(
                    "grid_cell_id",
                    "epoch",
                )
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )

            built_duplicate_count = (
                built_df
                .groupBy(
                    "grid_cell_id",
                    "epoch",
                )
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )

            if grid_duplicate_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: grid-year input "
                    "contains duplicate keys."
                )

            if built_duplicate_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: built-surface input "
                    "contains duplicate keys."
                )


            # ================================================
            # TWO-WAY KEY COVERAGE
            # ================================================

            grid_keys_df = (
                grid_year_raw_df
                .select(
                    "grid_cell_id",
                    "epoch",
                )
            )

            built_keys_df = (
                built_df
                .select(
                    "grid_cell_id",
                    "epoch",
                )
            )

            grid_without_built = (
                grid_keys_df
                .join(
                    built_keys_df,
                    on=[
                        "grid_cell_id",
                        "epoch",
                    ],
                    how="left_anti",
                )
                .count()
            )

            built_without_grid = (
                built_keys_df
                .join(
                    grid_keys_df,
                    on=[
                        "grid_cell_id",
                        "epoch",
                    ],
                    how="left_anti",
                )
                .count()
            )

            if grid_without_built != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: "
                    f"{grid_without_built} grid cells "
                    "have no built-surface record."
                )

            if built_without_grid != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: "
                    f"{built_without_grid} built cells "
                    "have no grid-year record."
                )

            print("Two-way key coverage: PASS")


            # ================================================
            # STRICT ONE-TO-ONE JOIN
            # ================================================

            joined_raw_df = (
                grid_year_raw_df.alias("g")
                .join(
                    built_df.alias("b"),
                    on=[
                        F.col("g.grid_cell_id")
                        == F.col("b.grid_cell_id"),

                        F.col("g.epoch")
                        == F.col("b.epoch"),
                    ],
                    how="inner",
                )
            )

            joined_count = joined_raw_df.count()

            if joined_count != EXPECTED_GRID_CELLS:
                raise RuntimeError(
                    f"Epoch {epoch}: unexpected joined "
                    f"count {joined_count}."
                )


            # ================================================
            # COORDINATE AND COVERAGE AGREEMENT
            # ================================================

            coordinate_mismatch_count = (
                joined_raw_df
                .filter(
                    (
                        F.col("g.grid_1km_col")
                        != F.col("b.built_grid_col")
                    )
                    | (
                        F.col("g.grid_1km_row")
                        != F.col("b.built_grid_row")
                    )
                    | (
                        F.abs(
                            F.col("g.grid_min_x")
                            - F.col(
                                "b.built_grid_min_x"
                            )
                        )
                        > F.lit(1e-9)
                    )
                    | (
                        F.abs(
                            F.col("g.grid_max_x")
                            - F.col(
                                "b.built_grid_max_x"
                            )
                        )
                        > F.lit(1e-9)
                    )
                    | (
                        F.abs(
                            F.col("g.grid_min_y")
                            - F.col(
                                "b.built_grid_min_y"
                            )
                        )
                        > F.lit(1e-9)
                    )
                    | (
                        F.abs(
                            F.col("g.grid_max_y")
                            - F.col(
                                "b.built_grid_max_y"
                            )
                        )
                        > F.lit(1e-9)
                    )
                )
                .count()
            )

            if coordinate_mismatch_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: coordinate mismatch "
                    f"for {coordinate_mismatch_count} cells."
                )


            coverage_mismatch_count = (
                joined_raw_df
                .filter(
                    (
                        F.col("g.valid_100m_pixels")
                        != F.col(
                            "b.built_valid_100m_pixels"
                        )
                    )
                    | (
                        F.abs(
                            F.col(
                                "g.population_coverage_fraction"
                            )
                            - F.col(
                                "b.built_coverage_fraction"
                            )
                        )
                        > F.lit(1e-12)
                    )
                )
                .count()
            )

            if coverage_mismatch_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: population and built "
                    "coverage differ for "
                    f"{coverage_mismatch_count} cells."
                )

            print("Grid coordinate agreement: PASS")
            print("Population-built coverage agreement: PASS")


            # ================================================
            # CREATE ENRICHED RECORDS
            #
            # Population per built hectare:
            #   population / (built_m2 / 10,000)
            #
            # People per built 100 m2:
            #   population / (built_m2 / 100)
            #
            # Metrics are NULL when built surface is zero.
            # ================================================

            enriched_df = (
                joined_raw_df
                .select(
                    F.col("g.*"),

                    F.col(
                        "b.built_valid_100m_pixels"
                    ),

                    F.col(
                        "b.built_observed_grid_area_m2"
                    ),

                    F.col(
                        "b.built_coverage_fraction"
                    ),

                    F.col(
                        "b.built_surface_total_m2"
                    ),

                    F.col(
                        "b.built_surface_nres_m2"
                    ),

                    F.col(
                        "b.built_surface_residential_m2"
                    ),

                    F.col(
                        "b.built_surface_fraction"
                    ),

                    F.col(
                        "b.built_surface_percent"
                    ),

                    F.col(
                        "b.built_surface_nres_fraction"
                    ),

                    F.col(
                        "b.built_surface_residential_fraction"
                    ),

                    F.col(
                        "b.nres_share_of_built_surface"
                    ),

                    F.col(
                        "b.built_100m_pixels"
                    ),

                    F.col(
                        "b.nres_100m_pixels"
                    ),

                    F.col(
                        "b.is_complete_built_cell"
                    ),

                    F.col(
                        "b.built_quality_flag"
                    ),

                    F.col(
                        "b.built_surface_source_path"
                    ),
                )
                .withColumn(
                    "built_surface_total_km2",
                    F.col(
                        "built_surface_total_m2"
                    )
                    / F.lit(1000000.0),
                )
                .withColumn(
                    "built_surface_residential_km2",
                    F.col(
                        "built_surface_residential_m2"
                    )
                    / F.lit(1000000.0),
                )
                .withColumn(
                    "built_surface_nres_km2",
                    F.col(
                        "built_surface_nres_m2"
                    )
                    / F.lit(1000000.0),
                )
                .withColumn(
                    "population_per_built_hectare",
                    F.when(
                        F.col(
                            "built_surface_total_m2"
                        )
                        > F.lit(0.0),

                        F.col("population_total")
                        / (
                            F.col(
                                "built_surface_total_m2"
                            )
                            / F.lit(10000.0)
                        ),
                    ).otherwise(
                        F.lit(None).cast("double")
                    ),
                )
                .withColumn(
                    "population_per_residential_built_hectare",
                    F.when(
                        F.col(
                            "built_surface_residential_m2"
                        )
                        > F.lit(0.0),

                        F.col("population_total")
                        / (
                            F.col(
                                "built_surface_residential_m2"
                            )
                            / F.lit(10000.0)
                        ),
                    ).otherwise(
                        F.lit(None).cast("double")
                    ),
                )
                .withColumn(
                    "people_per_built_100m2",
                    F.when(
                        F.col(
                            "built_surface_total_m2"
                        )
                        > F.lit(0.0),

                        F.col("population_total")
                        / (
                            F.col(
                                "built_surface_total_m2"
                            )
                            / F.lit(100.0)
                        ),
                    ).otherwise(
                        F.lit(None).cast("double")
                    ),
                )
                .withColumn(
                    "has_built_surface",
                    F.col(
                        "built_surface_total_m2"
                    )
                    > F.lit(0.0),
                )
                .withColumn(
                    "is_complete_all_products",
                    (
                        F.col(
                            "population_quality_flag"
                        )
                        == F.lit("complete")
                    )
                    & (
                        F.col(
                            "built_quality_flag"
                        )
                        == F.lit("complete")
                    ),
                )
                .withColumn(
                    "enriched_processed_at_utc",
                    F.current_timestamp(),
                )
            )


            # ================================================
            # OUTPUT VALIDATION
            # ================================================

            output_count = enriched_df.count()

            if output_count != EXPECTED_GRID_CELLS:
                raise RuntimeError(
                    f"Epoch {epoch}: expected "
                    f"{EXPECTED_GRID_CELLS} enriched rows, "
                    f"found {output_count}."
                )


            duplicate_output_count = (
                enriched_df
                .groupBy(
                    "grid_cell_id",
                    "epoch",
                )
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )

            if duplicate_output_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: enriched output "
                    "contains duplicate keys."
                )


            invalid_output_count = (
                enriched_df
                .filter(
                    F.col(
                        "built_surface_total_m2"
                    ).isNull()
                    | F.col(
                        "built_surface_nres_m2"
                    ).isNull()
                    | F.col(
                        "built_surface_residential_m2"
                    ).isNull()
                    | F.isnan(
                        "built_surface_total_m2"
                    )
                    | F.isnan(
                        "built_surface_nres_m2"
                    )
                    | (
                        F.col(
                            "built_surface_total_m2"
                        )
                        < F.lit(0.0)
                    )
                    | (
                        F.col(
                            "built_surface_nres_m2"
                        )
                        < F.lit(0.0)
                    )
                    | (
                        F.col(
                            "built_surface_residential_m2"
                        )
                        < F.lit(
                            -NUMERICAL_TOLERANCE
                        )
                    )
                    | (
                        F.col(
                            "built_surface_nres_m2"
                        )
                        > (
                            F.col(
                                "built_surface_total_m2"
                            )
                            + F.lit(
                                NUMERICAL_TOLERANCE
                            )
                        )
                    )
                    | (
                        F.col(
                            "built_surface_fraction"
                        )
                        < F.lit(0.0)
                    )
                    | (
                        F.col(
                            "built_surface_fraction"
                        )
                        > F.lit(1.0 + 1e-9)
                    )
                )
                .count()
            )

            if invalid_output_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: "
                    f"{invalid_output_count} enriched rows "
                    "failed validation."
                )


            zero_built_density_error_count = (
                enriched_df
                .filter(
                    (
                        F.col(
                            "built_surface_total_m2"
                        )
                        == F.lit(0.0)
                    )
                    & F.col(
                        "population_per_built_hectare"
                    ).isNotNull()
                )
                .count()
            )

            if zero_built_density_error_count != 0:
                raise RuntimeError(
                    f"Epoch {epoch}: density is present "
                    "for zero-built-surface cells."
                )


            # ================================================
            # TOTAL RECONCILIATION
            # ================================================

            totals = (
                enriched_df
                .agg(
                    F.sum(
                        "population_total"
                    ).alias("population_total"),

                    F.sum(
                        "built_surface_total_m2"
                    ).alias(
                        "built_surface_total_m2"
                    ),

                    F.sum(
                        "built_surface_nres_m2"
                    ).alias(
                        "built_surface_nres_m2"
                    ),

                    F.sum(
                        "built_surface_residential_m2"
                    ).alias(
                        "built_surface_residential_m2"
                    ),
                )
                .first()
            )

            for metric, expected_value in (
                EXPECTED_TOTALS[epoch].items()
            ):
                actual_value = float(
                    totals[metric]
                )

                allowed_difference = max(
                    1e-6,
                    abs(expected_value) * 1e-12,
                )

                if (
                    abs(actual_value - expected_value)
                    > allowed_difference
                ):
                    raise RuntimeError(
                        f"Epoch {epoch}: total mismatch "
                        f"for {metric}. "
                        f"Expected={expected_value}, "
                        f"actual={actual_value}."
                    )


            complete_count = (
                enriched_df
                .filter(
                    F.col(
                        "is_complete_all_products"
                    )
                )
                .count()
            )

            partial_count = (
                output_count - complete_count
            )

            if (
                complete_count
                != EXPECTED_COMPLETE_CELLS
            ):
                raise RuntimeError(
                    f"Epoch {epoch}: expected "
                    f"{EXPECTED_COMPLETE_CELLS} complete "
                    f"cells, found {complete_count}."
                )

            if (
                partial_count
                != EXPECTED_PARTIAL_CELLS
            ):
                raise RuntimeError(
                    f"Epoch {epoch}: expected "
                    f"{EXPECTED_PARTIAL_CELLS} partial "
                    f"cells, found {partial_count}."
                )


            # ================================================
            # ANALYTICAL SUMMARY
            # ================================================

            summary_df = (
                enriched_df
                .agg(
                    F.count("*").alias(
                        "grid_cell_count"
                    ),

                    F.sum(
                        "population_total"
                    ).alias(
                        "population_total"
                    ),

                    F.sum(
                        "built_surface_total_m2"
                    ).alias(
                        "built_surface_total_m2"
                    ),

                    F.sum(
                        "built_surface_residential_m2"
                    ).alias(
                        "built_surface_residential_m2"
                    ),

                    F.sum(
                        "built_surface_nres_m2"
                    ).alias(
                        "built_surface_nres_m2"
                    ),

                    F.avg(
                        "built_surface_fraction"
                    ).alias(
                        "average_built_fraction"
                    ),

                    F.sum(
                        F.when(
                            F.col(
                                "has_built_surface"
                            ),
                            F.lit(1),
                        ).otherwise(
                            F.lit(0)
                        )
                    ).alias(
                        "cells_with_built_surface"
                    ),

                    F.avg(
                        "population_per_built_hectare"
                    ).alias(
                        "mean_cell_population_per_built_hectare"
                    ),
                )
            )

            print()
            print(
                "=" * 80
            )
            print(
                f"ENRICHED {epoch} SUMMARY"
            )
            print(
                "=" * 80
            )

            summary_df.show(
                truncate=False,
                vertical=True,
            )


            print()
            print(
                "=" * 80
            )
            print(
                f"TOP 20 {epoch} POPULATION PER BUILT HECTARE"
            )
            print(
                "=" * 80
            )

            (
                enriched_df
                .filter(
                    F.col(
                        "population_per_built_hectare"
                    ).isNotNull()
                )
                .select(
                    "grid_cell_id",
                    "population_total",
                    "built_surface_total_m2",
                    "built_surface_fraction",
                    "population_per_built_hectare",
                    "smod_class",
                    "settlement_group",
                    "population_quality_flag",
                    "built_quality_flag",
                )
                .orderBy(
                    F.col(
                        "population_per_built_hectare"
                    ).desc()
                )
                .show(
                    20,
                    truncate=False,
                )
            )


            # ================================================
            # WRITE ENRICHED DATASET
            # ================================================

            (
                enriched_df
                .repartition(4)
                .write
                .mode("overwrite")
                .parquet(output_path)
            )

            print(
                f"HDFS enriched {epoch} write: COMPLETE"
            )


            # ================================================
            # READ-BACK VALIDATION
            # ================================================

            readback_df = spark.read.parquet(
                output_path
            )

            readback_count = readback_df.count()

            if readback_count != EXPECTED_GRID_CELLS:
                raise RuntimeError(
                    f"Epoch {epoch}: read-back row "
                    "count mismatch."
                )


            readback_totals = (
                readback_df
                .agg(
                    F.sum(
                        "population_total"
                    ).alias("population_total"),

                    F.sum(
                        "built_surface_total_m2"
                    ).alias(
                        "built_surface_total_m2"
                    ),

                    F.sum(
                        "built_surface_nres_m2"
                    ).alias(
                        "built_surface_nres_m2"
                    ),

                    F.sum(
                        "built_surface_residential_m2"
                    ).alias(
                        "built_surface_residential_m2"
                    ),
                )
                .first()
            )

            for metric, expected_value in (
                EXPECTED_TOTALS[epoch].items()
            ):
                readback_value = float(
                    readback_totals[metric]
                )

                allowed_difference = max(
                    1e-6,
                    abs(expected_value) * 1e-12,
                )

                if (
                    abs(
                        readback_value
                        - expected_value
                    )
                    > allowed_difference
                ):
                    raise RuntimeError(
                        f"Epoch {epoch}: read-back "
                        f"changed {metric}."
                    )


            result_record = (
                summary_df
                .first()
                .asDict()
            )

            result_record["epoch"] = epoch
            result_record["output_path"] = (
                output_path
            )
            result_record["complete_cells"] = (
                complete_count
            )
            result_record["partial_cells"] = (
                partial_count
            )

            epoch_results.append(
                result_record
            )

            print(
                f"Epoch {epoch} enrichment: PASS"
            )


        # ====================================================
        # ENDPOINT COMPARISON
        # ====================================================

        results_by_epoch = {
            row["epoch"]: row
            for row in epoch_results
        }

        result_1975 = results_by_epoch[1975]
        result_2020 = results_by_epoch[2020]

        population_change = (
            result_2020["population_total"]
            - result_1975["population_total"]
        )

        built_change_m2 = (
            result_2020[
                "built_surface_total_m2"
            ]
            - result_1975[
                "built_surface_total_m2"
            ]
        )

        population_growth_percent = (
            population_change
            / result_1975["population_total"]
            * 100.0
        )

        built_growth_percent = (
            built_change_m2
            / result_1975[
                "built_surface_total_m2"
            ]
            * 100.0
        )

        population_growth_multiple = (
            result_2020["population_total"]
            / result_1975["population_total"]
        )

        built_growth_multiple = (
            result_2020[
                "built_surface_total_m2"
            ]
            / result_1975[
                "built_surface_total_m2"
            ]
        )


        print()
        print("=" * 80)
        print("1975 TO 2020 ENDPOINT COMPARISON")
        print("=" * 80)

        print(
            "Population 1975:",
            result_1975["population_total"],
        )

        print(
            "Population 2020:",
            result_2020["population_total"],
        )

        print(
            "Population increase:",
            population_change,
        )

        print(
            "Population growth percent:",
            population_growth_percent,
        )

        print(
            "Population growth multiple:",
            population_growth_multiple,
        )

        print(
            "Built surface 1975 m2:",
            result_1975[
                "built_surface_total_m2"
            ],
        )

        print(
            "Built surface 2020 m2:",
            result_2020[
                "built_surface_total_m2"
            ],
        )

        print(
            "Built-surface increase m2:",
            built_change_m2,
        )

        print(
            "Built-surface growth percent:",
            built_growth_percent,
        )

        print(
            "Built-surface growth multiple:",
            built_growth_multiple,
        )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        print()
        print("=" * 80)
        print("CELL 17 REMOTE RESULT")
        print("=" * 80)

        print("1975 grid-year read: PASS")
        print("1975 built-surface read: PASS")
        print("2020 grid-year read: PASS")
        print("2020 built-surface read: PASS")
        print("Two-way key coverage: PASS")
        print("Grid coordinate agreement: PASS")
        print("Coverage agreement: PASS")
        print("Spark one-to-one joins: PASS")
        print("Density metric derivation: PASS")
        print("Endpoint total reconciliation: PASS")
        print("HDFS enriched writes: PASS")
        print("HDFS enriched read-back: PASS")

        print(
            "1975 output:",
            result_1975["output_path"],
        )

        print(
            "2020 output:",
            result_2020["output_path"],
        )

        print(
            "Grid cells per epoch:",
            EXPECTED_GRID_CELLS,
        )

        print(
            "Complete cells per epoch:",
            EXPECTED_COMPLETE_CELLS,
        )

        print(
            "Partial cells per epoch:",
            EXPECTED_PARTIAL_CELLS,
        )

        print(
            "Population growth percent:",
            population_growth_percent,
        )

        print(
            "Built-surface growth percent:",
            built_growth_percent,
        )

        print("Source GeoTIFF reread: NO")
        print("Python UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 17: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

encoded_program = base64.b64encode(
    cell17_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell17_enrich_endpoints.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 2g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.driver.maxResultSize=256m "
    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=1200,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 17 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 17: SUCCESS")

Remote Python syntax: PASS
CELL 17: ENRICH ENDPOINT GRID-YEAR DATASETS
Epochs: [1975, 2020]

PROCESSING EPOCH 1975
Grid-year input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/grid_year/epoch=1975
Built-surface input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/product=built_surface/epoch=1975/grid_1km
Output: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=1975
Two-way key coverage: PASS
Grid coordinate agreement: PASS
Population-built coverage agreement: PASS

ENRICHED 1975 SUMMARY
-RECORD 0------------------------------------------------------
 grid_cell_count                        | 3658                 
 population_total                       | 3060319.0921731493   
 built_surface_total_m2                 | 6.7085621E7          
 built_surface_residential_m2           | 6.0540667E7          
 built_surface_nres_m2                  | 6544954.0            
 average_built_fraction                 | 0.018379621944075625 
 ce

Both enriched endpoint datasets now combine population, SMOD, and built surface on the same 3,658-cell grid, with exact key, coordinate, coverage, and total reconciliation.

Enriched endpoint results
Metric	1975	2020	ChangePopulation	3.06M	14.61M	+377.36%
Built surface	67.09 km²	342.43 km²	+410.43%
Cells with built surface	3,249	3,601	+352 cells
Mean cell population per built hectare	776.73	228.69	−70.6%

Important interpretation

Built surface expanded 5.10×, while population grew 4.77×. At the pilot-wide level, physical urban expansion slightly outpaced population growth.

However, the reported mean population-per-built-hectare fell sharply partly because the 1975 mean is distorted by cells with extremely small built-surface denominators. For example, a cell with only 97 m² of estimated built surface produces a density above 4,300 people per built hectare. We should not classify densification from these raw ratios alone.

Persistent enriched outputs
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=1975
Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=2020

Next step

Cell 18 should perform the integrated endpoint change analysis and derive data-driven thresholds from complete cells:

Join enriched 1975 and 2020 records.
Calculate per-cell population and built-surface change.
Exclude partial boundary cells from threshold estimation.
Treat very small 1975 denominators carefully.
Compute quantile thresholds with Spark approxQuantile, such as:
25th percentile
median
75th percentile
Classify cells as:
combined high growth
population-led densification
built-surface-led expansion
moderate growth
stable
decline or anomaly
Summarize growth types by SMOD transition.
Write detailed and summary outputs to HDFS.

The thresholds should be learned from the actual cell-change distribution, not chosen arbitrarily.

In [28]:
# ============================================================
# CELL 18: INTEGRATED POPULATION + BUILT-SURFACE CHANGE
#          CLASSIFICATION, BENGALURU 1975 TO 2020
#
# Inputs:
#   enriched_grid_year/epoch=1975
#   enriched_grid_year/epoch=2020
#
# Outputs:
#   integrated_change/cell_change
#   integrated_change/growth_class_summary
#   integrated_change/growth_by_settlement_transition
#   integrated_change/thresholds
#
# Threshold method:
#   Spark approxQuantile on complete cells
#
# Main classifications:
#   - combined_high_growth
#   - population_led_densification
#   - built_surface_led_expansion
#   - moderate_combined_growth
#   - stable_low_change
#   - population_decline
#   - built_surface_decline_or_revision
#   - mixed_or_other
#
# Primary tools:
#   - HDFS
#   - Spark DataFrames
#   - Spark SQL
#
# No GeoTIFF is reread.
# No Python UDF, NumPy, Rasterio, or local pixel processing.
# ============================================================

import base64
import textwrap


cell18_program = textwrap.dedent(
    r'''
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.types import (
        StructType,
        StructField,
        StringType,
        DoubleType,
        LongType,
    )


    # ========================================================
    # CONFIGURATION
    # ========================================================

    HDFS_BASE = (
        "hdfs://localhost:9000"
        "/data/ghsl/curated/pilots/bengaluru"
    )

    INPUT_1975_PATH = (
        f"{HDFS_BASE}/enriched_grid_year/epoch=1975"
    )

    INPUT_2020_PATH = (
        f"{HDFS_BASE}/enriched_grid_year/epoch=2020"
    )

    OUTPUT_ROOT = (
        "hdfs://localhost:9000"
        "/data/ghsl/analytics/pilots/bengaluru/"
        "integrated_change_1975_2020"
    )

    CELL_OUTPUT_PATH = (
        f"{OUTPUT_ROOT}/cell_change"
    )

    CLASS_SUMMARY_PATH = (
        f"{OUTPUT_ROOT}/growth_class_summary"
    )

    SETTLEMENT_SUMMARY_PATH = (
        f"{OUTPUT_ROOT}/growth_by_settlement_transition"
    )

    THRESHOLD_OUTPUT_PATH = (
        f"{OUTPUT_ROOT}/thresholds"
    )

    CITY = "bengaluru"
    START_EPOCH = 1975
    END_EPOCH = 2020
    YEARS_ELAPSED = 45

    EXPECTED_GRID_CELLS = 3658
    EXPECTED_COMPLETE_CELLS = 3480
    EXPECTED_PARTIAL_CELLS = 178

    EXPECTED_POPULATION_1975 = (
        3060319.0921731493
    )

    EXPECTED_POPULATION_2020 = (
        14608885.859992508
    )

    EXPECTED_BUILT_1975_M2 = 67085621.0
    EXPECTED_BUILT_2020_M2 = 342426224.0

    QUANTILE_PROBABILITIES = [
        0.25,
        0.50,
        0.75,
    ]

    QUANTILE_RELATIVE_ERROR = 0.001

    MIN_BASE_POPULATION_FOR_RATIO = 10.0
    MIN_BASE_BUILT_M2_FOR_RATIO = 1000.0

    NUMERICAL_TOLERANCE = 1e-6


    spark = None

    try:
        # ====================================================
        # CREATE SPARK SESSION
        # ====================================================

        spark = (
            SparkSession.builder
            .appName(
                "GHSL-Bengaluru-Integrated-"
                "Change-1975-2020"
            )
            .getOrCreate()
        )

        spark.sparkContext.setLogLevel("WARN")


        print("=" * 80)
        print("CELL 18: INTEGRATED ENDPOINT CHANGE ANALYSIS")
        print("=" * 80)
        print("1975 input:", INPUT_1975_PATH)
        print("2020 input:", INPUT_2020_PATH)
        print("Output root:", OUTPUT_ROOT)


        # ====================================================
        # READ ENRICHED ENDPOINT DATASETS
        # ====================================================

        endpoint_1975_raw_df = spark.read.parquet(
            INPUT_1975_PATH
        )

        endpoint_2020_raw_df = spark.read.parquet(
            INPUT_2020_PATH
        )


        # ====================================================
        # NORMALIZE REQUIRED COLUMNS
        # ====================================================

        endpoint_1975_df = (
            endpoint_1975_raw_df
            .select(
                F.col("grid_cell_id"),

                F.col("grid_1km_col").alias(
                    "grid_1km_col_1975"
                ),

                F.col("grid_1km_row").alias(
                    "grid_1km_row_1975"
                ),

                F.col("grid_min_x").alias(
                    "grid_min_x_1975"
                ),

                F.col("grid_max_x").alias(
                    "grid_max_x_1975"
                ),

                F.col("grid_min_y").alias(
                    "grid_min_y_1975"
                ),

                F.col("grid_max_y").alias(
                    "grid_max_y_1975"
                ),

                F.col("population_total").alias(
                    "population_1975"
                ),

                F.col("built_surface_total_m2").alias(
                    "built_surface_total_m2_1975"
                ),

                F.col(
                    "built_surface_residential_m2"
                ).alias(
                    "built_surface_residential_m2_1975"
                ),

                F.col("built_surface_nres_m2").alias(
                    "built_surface_nres_m2_1975"
                ),

                F.col("built_surface_fraction").alias(
                    "built_surface_fraction_1975"
                ),

                F.col(
                    "population_per_built_hectare"
                ).alias(
                    "population_per_built_hectare_1975"
                ),

                F.col("smod_class").alias(
                    "smod_class_1975"
                ),

                F.col("settlement_group").alias(
                    "settlement_group_1975"
                ),

                F.col("is_urban").alias(
                    "is_urban_1975"
                ),

                F.col("is_urban_centre").alias(
                    "is_urban_centre_1975"
                ),

                F.col(
                    "is_complete_all_products"
                ).alias(
                    "is_complete_all_products_1975"
                ),

                F.col("geometry_wkb"),
                F.col("geometry_crs"),
            )
        )


        endpoint_2020_df = (
            endpoint_2020_raw_df
            .select(
                F.col("grid_cell_id"),

                F.col("grid_1km_col").alias(
                    "grid_1km_col_2020"
                ),

                F.col("grid_1km_row").alias(
                    "grid_1km_row_2020"
                ),

                F.col("grid_min_x").alias(
                    "grid_min_x_2020"
                ),

                F.col("grid_max_x").alias(
                    "grid_max_x_2020"
                ),

                F.col("grid_min_y").alias(
                    "grid_min_y_2020"
                ),

                F.col("grid_max_y").alias(
                    "grid_max_y_2020"
                ),

                F.col("population_total").alias(
                    "population_2020"
                ),

                F.col("built_surface_total_m2").alias(
                    "built_surface_total_m2_2020"
                ),

                F.col(
                    "built_surface_residential_m2"
                ).alias(
                    "built_surface_residential_m2_2020"
                ),

                F.col("built_surface_nres_m2").alias(
                    "built_surface_nres_m2_2020"
                ),

                F.col("built_surface_fraction").alias(
                    "built_surface_fraction_2020"
                ),

                F.col(
                    "population_per_built_hectare"
                ).alias(
                    "population_per_built_hectare_2020"
                ),

                F.col("smod_class").alias(
                    "smod_class_2020"
                ),

                F.col("settlement_group").alias(
                    "settlement_group_2020"
                ),

                F.col("is_urban").alias(
                    "is_urban_2020"
                ),

                F.col("is_urban_centre").alias(
                    "is_urban_centre_2020"
                ),

                F.col(
                    "is_complete_all_products"
                ).alias(
                    "is_complete_all_products_2020"
                ),
            )
        )


        # ====================================================
        # INPUT VALIDATION
        # ====================================================

        count_1975 = endpoint_1975_df.count()
        count_2020 = endpoint_2020_df.count()

        if count_1975 != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 1975 enriched row count."
            )

        if count_2020 != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected 2020 enriched row count."
            )


        def duplicate_count(dataframe):
            return (
                dataframe
                .groupBy("grid_cell_id")
                .count()
                .filter(
                    F.col("count") != 1
                )
                .count()
            )


        if duplicate_count(endpoint_1975_df) != 0:
            raise RuntimeError(
                "1975 input contains duplicate grid IDs."
            )

        if duplicate_count(endpoint_2020_df) != 0:
            raise RuntimeError(
                "2020 input contains duplicate grid IDs."
            )


        missing_from_2020 = (
            endpoint_1975_df
            .select("grid_cell_id")
            .join(
                endpoint_2020_df.select(
                    "grid_cell_id"
                ),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        new_in_2020 = (
            endpoint_2020_df
            .select("grid_cell_id")
            .join(
                endpoint_1975_df.select(
                    "grid_cell_id"
                ),
                on="grid_cell_id",
                how="left_anti",
            )
            .count()
        )

        if missing_from_2020 != 0:
            raise RuntimeError(
                "One or more 1975 cells are missing "
                "from 2020."
            )

        if new_in_2020 != 0:
            raise RuntimeError(
                "One or more 2020 cells are absent "
                "from 1975."
            )

        print("Endpoint key agreement: PASS")


        # ====================================================
        # JOIN ENDPOINTS
        # ====================================================

        joined_df = (
            endpoint_1975_df.alias("a")
            .join(
                endpoint_2020_df.alias("b"),
                on="grid_cell_id",
                how="inner",
            )
        )

        if joined_df.count() != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected endpoint join count."
            )


        spatial_mismatch_count = (
            joined_df
            .filter(
                (
                    F.col("a.grid_1km_col_1975")
                    != F.col("b.grid_1km_col_2020")
                )
                | (
                    F.col("a.grid_1km_row_1975")
                    != F.col("b.grid_1km_row_2020")
                )
                | (
                    F.abs(
                        F.col("a.grid_min_x_1975")
                        - F.col("b.grid_min_x_2020")
                    )
                    > F.lit(1e-9)
                )
                | (
                    F.abs(
                        F.col("a.grid_min_y_1975")
                        - F.col("b.grid_min_y_2020")
                    )
                    > F.lit(1e-9)
                )
            )
            .count()
        )

        if spatial_mismatch_count != 0:
            raise RuntimeError(
                "Endpoint grid coordinates do not agree."
            )

        print("Endpoint spatial agreement: PASS")


        # ====================================================
        # CALCULATE CONTINUOUS CHANGE METRICS
        # ====================================================

        change_df = (
            joined_df
            .select(
                F.lit(CITY).alias("city"),
                F.col("grid_cell_id"),

                F.lit(START_EPOCH).alias(
                    "start_epoch"
                ),

                F.lit(END_EPOCH).alias(
                    "end_epoch"
                ),

                F.lit(YEARS_ELAPSED).alias(
                    "years_elapsed"
                ),

                F.col(
                    "a.grid_1km_col_1975"
                ).alias("grid_1km_col"),

                F.col(
                    "a.grid_1km_row_1975"
                ).alias("grid_1km_row"),

                F.col(
                    "a.grid_min_x_1975"
                ).alias("grid_min_x"),

                F.col(
                    "a.grid_max_x_1975"
                ).alias("grid_max_x"),

                F.col(
                    "a.grid_min_y_1975"
                ).alias("grid_min_y"),

                F.col(
                    "a.grid_max_y_1975"
                ).alias("grid_max_y"),

                F.col("a.population_1975"),
                F.col("b.population_2020"),

                F.col(
                    "a.built_surface_total_m2_1975"
                ),

                F.col(
                    "b.built_surface_total_m2_2020"
                ),

                F.col(
                    "a.built_surface_residential_m2_1975"
                ),

                F.col(
                    "b.built_surface_residential_m2_2020"
                ),

                F.col(
                    "a.built_surface_nres_m2_1975"
                ),

                F.col(
                    "b.built_surface_nres_m2_2020"
                ),

                F.col(
                    "a.built_surface_fraction_1975"
                ),

                F.col(
                    "b.built_surface_fraction_2020"
                ),

                F.col(
                    "a.population_per_built_hectare_1975"
                ),

                F.col(
                    "b.population_per_built_hectare_2020"
                ),

                F.col("a.smod_class_1975"),
                F.col("b.smod_class_2020"),

                F.col(
                    "a.settlement_group_1975"
                ),

                F.col(
                    "b.settlement_group_2020"
                ),

                F.col("a.is_urban_1975"),
                F.col("b.is_urban_2020"),

                F.col(
                    "a.is_urban_centre_1975"
                ),

                F.col(
                    "b.is_urban_centre_2020"
                ),

                F.col(
                    "a.is_complete_all_products_1975"
                ),

                F.col(
                    "b.is_complete_all_products_2020"
                ),

                F.col("a.geometry_wkb"),
                F.col("a.geometry_crs"),
            )
            .withColumn(
                "population_change_absolute",
                F.col("population_2020")
                - F.col("population_1975"),
            )
            .withColumn(
                "built_surface_change_m2",
                F.col(
                    "built_surface_total_m2_2020"
                )
                - F.col(
                    "built_surface_total_m2_1975"
                ),
            )
            .withColumn(
                "built_surface_residential_change_m2",
                F.col(
                    "built_surface_residential_m2_2020"
                )
                - F.col(
                    "built_surface_residential_m2_1975"
                ),
            )
            .withColumn(
                "built_surface_nres_change_m2",
                F.col(
                    "built_surface_nres_m2_2020"
                )
                - F.col(
                    "built_surface_nres_m2_1975"
                ),
            )
            .withColumn(
                "built_fraction_change",
                F.col(
                    "built_surface_fraction_2020"
                )
                - F.col(
                    "built_surface_fraction_1975"
                ),
            )
            .withColumn(
                "population_change_percent",
                F.when(
                    F.col("population_1975")
                    >= F.lit(
                        MIN_BASE_POPULATION_FOR_RATIO
                    ),

                    (
                        F.col(
                            "population_change_absolute"
                        )
                        / F.col("population_1975")
                    )
                    * F.lit(100.0),
                ).otherwise(
                    F.lit(None).cast("double")
                ),
            )
            .withColumn(
                "built_surface_change_percent",
                F.when(
                    F.col(
                        "built_surface_total_m2_1975"
                    )
                    >= F.lit(
                        MIN_BASE_BUILT_M2_FOR_RATIO
                    ),

                    (
                        F.col(
                            "built_surface_change_m2"
                        )
                        / F.col(
                            "built_surface_total_m2_1975"
                        )
                    )
                    * F.lit(100.0),
                ).otherwise(
                    F.lit(None).cast("double")
                ),
            )
            .withColumn(
                "density_change_people_per_built_hectare",
                F.when(
                    F.col(
                        "population_per_built_hectare_1975"
                    ).isNotNull()
                    & F.col(
                        "population_per_built_hectare_2020"
                    ).isNotNull(),

                    F.col(
                        "population_per_built_hectare_2020"
                    )
                    - F.col(
                        "population_per_built_hectare_1975"
                    ),
                ).otherwise(
                    F.lit(None).cast("double")
                ),
            )
            .withColumn(
                "population_change_per_new_built_hectare",
                F.when(
                    F.col("built_surface_change_m2")
                    > F.lit(0.0),

                    F.col(
                        "population_change_absolute"
                    )
                    / (
                        F.col(
                            "built_surface_change_m2"
                        )
                        / F.lit(10000.0)
                    ),
                ).otherwise(
                    F.lit(None).cast("double")
                ),
            )
            .withColumn(
                "settlement_transition",
                F.concat_ws(
                    " -> ",
                    F.col(
                        "settlement_group_1975"
                    ),
                    F.col(
                        "settlement_group_2020"
                    ),
                ),
            )
            .withColumn(
                "became_urban",
                (
                    ~F.col("is_urban_1975")
                )
                & F.col("is_urban_2020"),
            )
            .withColumn(
                "became_urban_centre",
                (
                    ~F.col(
                        "is_urban_centre_1975"
                    )
                )
                & F.col(
                    "is_urban_centre_2020"
                ),
            )
            .withColumn(
                "is_complete_both_epochs",
                F.col(
                    "is_complete_all_products_1975"
                )
                & F.col(
                    "is_complete_all_products_2020"
                ),
            )
        )


        # ====================================================
        # COMPLETE-CELL THRESHOLD ESTIMATION DATASET
        #
        # Boundary cells are excluded only from threshold
        # estimation, not from classification or output.
        # ====================================================

        threshold_base_df = (
            change_df
            .filter(
                F.col(
                    "is_complete_both_epochs"
                )
            )
            .select(
                "population_change_absolute",
                "built_surface_change_m2",
                "built_fraction_change",
            )
        )

        threshold_base_count = (
            threshold_base_df.count()
        )

        if (
            threshold_base_count
            != EXPECTED_COMPLETE_CELLS
        ):
            raise RuntimeError(
                "Unexpected complete-cell threshold base. "
                f"Expected {EXPECTED_COMPLETE_CELLS}, "
                f"found {threshold_base_count}."
            )


        # ====================================================
        # DATA-DRIVEN QUANTILES
        # ====================================================

        population_quantiles = (
            threshold_base_df.approxQuantile(
                "population_change_absolute",
                QUANTILE_PROBABILITIES,
                QUANTILE_RELATIVE_ERROR,
            )
        )

        built_quantiles = (
            threshold_base_df.approxQuantile(
                "built_surface_change_m2",
                QUANTILE_PROBABILITIES,
                QUANTILE_RELATIVE_ERROR,
            )
        )

        built_fraction_quantiles = (
            threshold_base_df.approxQuantile(
                "built_fraction_change",
                QUANTILE_PROBABILITIES,
                QUANTILE_RELATIVE_ERROR,
            )
        )


        if (
            len(population_quantiles) != 3
            or len(built_quantiles) != 3
            or len(built_fraction_quantiles) != 3
        ):
            raise RuntimeError(
                "Spark did not return all required quantiles."
            )


        pop_q25, pop_q50, pop_q75 = [
            float(value)
            for value in population_quantiles
        ]

        built_q25, built_q50, built_q75 = [
            float(value)
            for value in built_quantiles
        ]

        (
            built_fraction_q25,
            built_fraction_q50,
            built_fraction_q75,
        ) = [
            float(value)
            for value in built_fraction_quantiles
        ]


        print()
        print("=" * 80)
        print("DATA-DRIVEN COMPLETE-CELL THRESHOLDS")
        print("=" * 80)

        print(
            "Population absolute-change quantiles:",
            {
                "q25": pop_q25,
                "median": pop_q50,
                "q75": pop_q75,
            },
        )

        print(
            "Built-surface change m2 quantiles:",
            {
                "q25": built_q25,
                "median": built_q50,
                "q75": built_q75,
            },
        )

        print(
            "Built-fraction change quantiles:",
            {
                "q25": built_fraction_q25,
                "median": built_fraction_q50,
                "q75": built_fraction_q75,
            },
        )


        # ====================================================
        # CLASSIFY GROWTH TYPE
        #
        # Classification priority:
        #   1. material decline/anomaly
        #   2. high-high combined growth
        #   3. population-led growth
        #   4. built-surface-led expansion
        #   5. moderate combined growth
        #   6. stable low change
        #   7. mixed or other
        #
        # q75 identifies relatively high change.
        # q25 identifies relatively low change.
        # ====================================================

        classified_df = (
            change_df
            .withColumn(
                "population_growth_level",

                F.when(
                    F.col(
                        "population_change_absolute"
                    )
                    >= F.lit(pop_q75),
                    F.lit("high"),
                )
                .when(
                    F.col(
                        "population_change_absolute"
                    )
                    >= F.lit(pop_q25),
                    F.lit("moderate"),
                )
                .otherwise(
                    F.lit("low_or_negative")
                ),
            )
            .withColumn(
                "built_growth_level",

                F.when(
                    F.col(
                        "built_surface_change_m2"
                    )
                    >= F.lit(built_q75),
                    F.lit("high"),
                )
                .when(
                    F.col(
                        "built_surface_change_m2"
                    )
                    >= F.lit(built_q25),
                    F.lit("moderate"),
                )
                .otherwise(
                    F.lit("low_or_negative")
                ),
            )
            .withColumn(
                "growth_class",

                F.when(
                    F.col("population_change_absolute")
                    < F.lit(0.0),
                    F.lit("population_decline"),
                )

                .when(
                    F.col("built_surface_change_m2")
                    < F.lit(
                        -NUMERICAL_TOLERANCE
                    ),
                    F.lit(
                        "built_surface_decline_or_revision"
                    ),
                )

                .when(
                    (
                        F.col(
                            "population_change_absolute"
                        )
                        >= F.lit(pop_q75)
                    )
                    & (
                        F.col(
                            "built_surface_change_m2"
                        )
                        >= F.lit(built_q75)
                    ),
                    F.lit("combined_high_growth"),
                )

                .when(
                    (
                        F.col(
                            "population_change_absolute"
                        )
                        >= F.lit(pop_q75)
                    )
                    & (
                        F.col(
                            "built_surface_change_m2"
                        )
                        < F.lit(built_q50)
                    ),
                    F.lit(
                        "population_led_densification"
                    ),
                )

                .when(
                    (
                        F.col(
                            "built_surface_change_m2"
                        )
                        >= F.lit(built_q75)
                    )
                    & (
                        F.col(
                            "population_change_absolute"
                        )
                        < F.lit(pop_q50)
                    ),
                    F.lit(
                        "built_surface_led_expansion"
                    ),
                )

                .when(
                    (
                        F.col(
                            "population_change_absolute"
                        )
                        >= F.lit(pop_q25)
                    )
                    & (
                        F.col(
                            "built_surface_change_m2"
                        )
                        >= F.lit(built_q25)
                    ),
                    F.lit(
                        "moderate_combined_growth"
                    ),
                )

                .when(
                    (
                        F.abs(
                            F.col(
                                "population_change_absolute"
                            )
                        )
                        <= F.lit(
                            max(abs(pop_q25), 1e-9)
                        )
                    )
                    & (
                        F.abs(
                            F.col(
                                "built_surface_change_m2"
                            )
                        )
                        <= F.lit(
                            max(abs(built_q25), 1e-9)
                        )
                    ),
                    F.lit("stable_low_change"),
                )

                .otherwise(
                    F.lit("mixed_or_other")
                ),
            )
            .withColumn(
                "threshold_population_q25",
                F.lit(pop_q25),
            )
            .withColumn(
                "threshold_population_median",
                F.lit(pop_q50),
            )
            .withColumn(
                "threshold_population_q75",
                F.lit(pop_q75),
            )
            .withColumn(
                "threshold_built_m2_q25",
                F.lit(built_q25),
            )
            .withColumn(
                "threshold_built_m2_median",
                F.lit(built_q50),
            )
            .withColumn(
                "threshold_built_m2_q75",
                F.lit(built_q75),
            )
            .withColumn(
                "threshold_method",
                F.lit(
                    "complete-cell Spark approxQuantile"
                ),
            )
            .withColumn(
                "processed_at_utc",
                F.current_timestamp(),
            )
        )


        # ====================================================
        # OUTPUT VALIDATION
        # ====================================================

        classified_count = classified_df.count()

        if classified_count != EXPECTED_GRID_CELLS:
            raise RuntimeError(
                "Unexpected classified row count."
            )


        duplicate_output_count = (
            classified_df
            .groupBy("grid_cell_id")
            .count()
            .filter(
                F.col("count") != 1
            )
            .count()
        )

        if duplicate_output_count != 0:
            raise RuntimeError(
                "Classified output contains duplicate cells."
            )


        invalid_output_count = (
            classified_df
            .filter(
                F.col("growth_class").isNull()
                | F.col(
                    "population_change_absolute"
                ).isNull()
                | F.col(
                    "built_surface_change_m2"
                ).isNull()
                | F.isnan(
                    "population_change_absolute"
                )
                | F.isnan(
                    "built_surface_change_m2"
                )
                | F.col("geometry_wkb").isNull()
            )
            .count()
        )

        if invalid_output_count != 0:
            raise RuntimeError(
                "One or more classified rows failed "
                "required-field validation."
            )


        # ====================================================
        # CONSERVATION VALIDATION
        # ====================================================

        totals = (
            classified_df
            .agg(
                F.sum("population_1975").alias(
                    "population_1975"
                ),

                F.sum("population_2020").alias(
                    "population_2020"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),

                F.sum(
                    "built_surface_total_m2_1975"
                ).alias(
                    "built_surface_total_m2_1975"
                ),

                F.sum(
                    "built_surface_total_m2_2020"
                ).alias(
                    "built_surface_total_m2_2020"
                ),

                F.sum(
                    "built_surface_change_m2"
                ).alias(
                    "built_surface_change_m2"
                ),
            )
            .first()
        )


        expected_totals = {
            "population_1975":
                EXPECTED_POPULATION_1975,

            "population_2020":
                EXPECTED_POPULATION_2020,

            "built_surface_total_m2_1975":
                EXPECTED_BUILT_1975_M2,

            "built_surface_total_m2_2020":
                EXPECTED_BUILT_2020_M2,
        }


        for metric, expected_value in (
            expected_totals.items()
        ):
            actual_value = float(
                totals[metric]
            )

            allowed_difference = max(
                1e-6,
                abs(expected_value) * 1e-12,
            )

            if (
                abs(actual_value - expected_value)
                > allowed_difference
            ):
                raise RuntimeError(
                    f"Total reconciliation failed for "
                    f"{metric}. Expected={expected_value}, "
                    f"actual={actual_value}."
                )


        expected_population_change = (
            EXPECTED_POPULATION_2020
            - EXPECTED_POPULATION_1975
        )

        expected_built_change = (
            EXPECTED_BUILT_2020_M2
            - EXPECTED_BUILT_1975_M2
        )


        if (
            abs(
                float(
                    totals[
                        "population_change_absolute"
                    ]
                )
                - expected_population_change
            )
            > max(
                1e-6,
                abs(expected_population_change)
                * 1e-12,
            )
        ):
            raise RuntimeError(
                "Population-change conservation failed."
            )


        if (
            abs(
                float(
                    totals[
                        "built_surface_change_m2"
                    ]
                )
                - expected_built_change
            )
            > max(
                1e-6,
                abs(expected_built_change)
                * 1e-12,
            )
        ):
            raise RuntimeError(
                "Built-surface-change conservation failed."
            )

        print("Endpoint total conservation: PASS")


        # ====================================================
        # GROWTH-CLASS SUMMARY
        # ====================================================

        growth_class_summary_df = (
            classified_df
            .groupBy("growth_class")
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_1975"
                ).alias("population_1975"),

                F.sum(
                    "population_2020"
                ).alias("population_2020"),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),

                F.sum(
                    "built_surface_total_m2_1975"
                ).alias(
                    "built_surface_total_m2_1975"
                ),

                F.sum(
                    "built_surface_total_m2_2020"
                ).alias(
                    "built_surface_total_m2_2020"
                ),

                F.sum(
                    "built_surface_change_m2"
                ).alias(
                    "built_surface_change_m2"
                ),

                F.avg(
                    "population_change_absolute"
                ).alias(
                    "average_population_change"
                ),

                F.avg(
                    "built_surface_change_m2"
                ).alias(
                    "average_built_change_m2"
                ),

                F.sum(
                    F.when(
                        F.col("became_urban"),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "cells_became_urban"
                ),

                F.sum(
                    F.when(
                        F.col(
                            "became_urban_centre"
                        ),
                        F.lit(1),
                    ).otherwise(
                        F.lit(0)
                    )
                ).alias(
                    "cells_became_urban_centre"
                ),
            )
            .orderBy(
                F.col(
                    "grid_cell_count"
                ).desc(),
            )
        )


        # ====================================================
        # GROWTH CLASS BY SETTLEMENT TRANSITION
        # ====================================================

        settlement_summary_df = (
            classified_df
            .groupBy(
                "settlement_transition",
                "growth_class",
            )
            .agg(
                F.count("*").alias(
                    "grid_cell_count"
                ),

                F.sum(
                    "population_change_absolute"
                ).alias(
                    "population_change_absolute"
                ),

                F.sum(
                    "built_surface_change_m2"
                ).alias(
                    "built_surface_change_m2"
                ),

                F.avg(
                    "population_change_absolute"
                ).alias(
                    "average_population_change"
                ),

                F.avg(
                    "built_surface_change_m2"
                ).alias(
                    "average_built_change_m2"
                ),
            )
            .orderBy(
                "settlement_transition",
                F.col(
                    "grid_cell_count"
                ).desc(),
            )
        )


        # ====================================================
        # THRESHOLD DATASET
        # ====================================================

        threshold_schema = StructType(
            [
                StructField(
                    "metric",
                    StringType(),
                    False,
                ),
                StructField(
                    "q25",
                    DoubleType(),
                    False,
                ),
                StructField(
                    "median",
                    DoubleType(),
                    False,
                ),
                StructField(
                    "q75",
                    DoubleType(),
                    False,
                ),
                StructField(
                    "complete_cell_count",
                    LongType(),
                    False,
                ),
                StructField(
                    "method",
                    StringType(),
                    False,
                ),
            ]
        )


        threshold_rows = [
            (
                "population_change_absolute",
                pop_q25,
                pop_q50,
                pop_q75,
                threshold_base_count,
                "Spark approxQuantile",
            ),
            (
                "built_surface_change_m2",
                built_q25,
                built_q50,
                built_q75,
                threshold_base_count,
                "Spark approxQuantile",
            ),
            (
                "built_fraction_change",
                built_fraction_q25,
                built_fraction_q50,
                built_fraction_q75,
                threshold_base_count,
                "Spark approxQuantile",
            ),
        ]


        threshold_df = spark.createDataFrame(
            threshold_rows,
            schema=threshold_schema,
        )


        # ====================================================
        # DISPLAY RESULTS
        # ====================================================

        print()
        print("=" * 80)
        print("GROWTH-CLASS SUMMARY")
        print("=" * 80)

        growth_class_summary_df.show(
            100,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("GROWTH CLASS BY SETTLEMENT TRANSITION")
        print("=" * 80)

        settlement_summary_df.show(
            200,
            truncate=False,
        )


        print()
        print("=" * 80)
        print("TOP 30 COMBINED-GROWTH CELLS")
        print("=" * 80)

        (
            classified_df
            .filter(
                F.col("growth_class")
                == F.lit(
                    "combined_high_growth"
                )
            )
            .select(
                "grid_cell_id",
                "population_1975",
                "population_2020",
                "population_change_absolute",
                "built_surface_total_m2_1975",
                "built_surface_total_m2_2020",
                "built_surface_change_m2",
                "built_fraction_change",
                "settlement_transition",
                "became_urban",
                "became_urban_centre",
            )
            .orderBy(
                F.col(
                    "population_change_absolute"
                ).desc(),
            )
            .show(
                30,
                truncate=False,
            )
        )


        # ====================================================
        # WRITE OUTPUTS TO HDFS
        # ====================================================

        (
            classified_df
            .repartition(4)
            .write
            .mode("overwrite")
            .parquet(CELL_OUTPUT_PATH)
        )

        (
            growth_class_summary_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(CLASS_SUMMARY_PATH)
        )

        (
            settlement_summary_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(SETTLEMENT_SUMMARY_PATH)
        )

        (
            threshold_df
            .coalesce(1)
            .write
            .mode("overwrite")
            .parquet(THRESHOLD_OUTPUT_PATH)
        )

        print("HDFS integrated-change outputs: WRITTEN")


        # ====================================================
        # READ-BACK VALIDATION
        # ====================================================

        cell_readback_df = spark.read.parquet(
            CELL_OUTPUT_PATH
        )

        class_readback_df = spark.read.parquet(
            CLASS_SUMMARY_PATH
        )

        settlement_readback_df = spark.read.parquet(
            SETTLEMENT_SUMMARY_PATH
        )

        threshold_readback_df = spark.read.parquet(
            THRESHOLD_OUTPUT_PATH
        )


        if (
            cell_readback_df.count()
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "Cell-change read-back count mismatch."
            )


        class_cell_total = (
            class_readback_df
            .agg(
                F.sum(
                    "grid_cell_count"
                ).alias("total")
            )
            .first()["total"]
        )

        settlement_cell_total = (
            settlement_readback_df
            .agg(
                F.sum(
                    "grid_cell_count"
                ).alias("total")
            )
            .first()["total"]
        )


        if (
            int(class_cell_total)
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "Growth-class summary does not reconcile "
                "to the full grid."
            )


        if (
            int(settlement_cell_total)
            != EXPECTED_GRID_CELLS
        ):
            raise RuntimeError(
                "Settlement-growth summary does not "
                "reconcile to the full grid."
            )


        if threshold_readback_df.count() != 3:
            raise RuntimeError(
                "Threshold read-back should contain "
                "exactly three metric records."
            )


        # ====================================================
        # FINAL RESULT
        # ====================================================

        class_count = (
            class_readback_df.count()
        )

        print()
        print("=" * 80)
        print("CELL 18 REMOTE RESULT")
        print("=" * 80)

        print("1975 enriched input read: PASS")
        print("2020 enriched input read: PASS")
        print("Endpoint key agreement: PASS")
        print("Endpoint spatial agreement: PASS")
        print("Continuous change metrics: PASS")
        print("Complete-cell quantile thresholds: PASS")
        print("Data-driven growth classification: PASS")
        print("Endpoint total conservation: PASS")
        print("Growth-class summary: PASS")
        print("Settlement-transition summary: PASS")
        print("HDFS output writes: PASS")
        print("HDFS output read-back: PASS")

        print(
            "Classified grid cells:",
            cell_readback_df.count(),
        )

        print(
            "Growth classes produced:",
            class_count,
        )

        print(
            "Threshold base complete cells:",
            threshold_base_count,
        )

        print(
            "Population q75 threshold:",
            pop_q75,
        )

        print(
            "Built-surface q75 threshold m2:",
            built_q75,
        )

        print(
            "Cell output:",
            CELL_OUTPUT_PATH,
        )

        print(
            "Class summary output:",
            CLASS_SUMMARY_PATH,
        )

        print(
            "Settlement summary output:",
            SETTLEMENT_SUMMARY_PATH,
        )

        print(
            "Threshold output:",
            THRESHOLD_OUTPUT_PATH,
        )

        print("Source GeoTIFF reread: NO")
        print("Python UDF used: NO")
        print("NumPy used: NO")
        print("Rasterio used: NO")
        print("CELL 18: SUCCESS")

    finally:
        if spark is not None:
            spark.stop()
    '''
).strip() + "\n"


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

encoded_program = base64.b64encode(
    cell18_program.encode("utf-8")
).decode("ascii")

remote_script = (
    "/home/ubuntu/"
    "ghsl_notebook04_cell18_integrated_change.py"
)

remote_command = (
    "set -e; "
    "source /home/ubuntu/ghsl-sedona-env.sh; "

    f"echo '{encoded_program}' "
    f"| base64 -d > {remote_script}; "

    f"python3 -m py_compile {remote_script}; "
    "echo 'Remote Python syntax: PASS'; "

    "spark-submit "
    "--master local[1] "
    "--driver-memory 2g "
    "--conf spark.ui.enabled=false "
    "--conf spark.sql.shuffle.partitions=4 "
    "--conf spark.default.parallelism=2 "
    "--conf spark.sql.adaptive.enabled=false "
    "--conf spark.driver.maxResultSize=256m "
    f"{remote_script}"
)


result = run_remote(
    remote_command,
    timeout=1200,
    show_output=True,
)

print()
print("=" * 80)
print("CELL 18 NOTEBOOK RESULT")
print("=" * 80)
print("Remote return code:", result.returncode)
print("CELL 18: SUCCESS")

Remote Python syntax: PASS
CELL 18: INTEGRATED ENDPOINT CHANGE ANALYSIS
1975 input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=1975
2020 input: hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/enriched_grid_year/epoch=2020
Output root: hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020
Endpoint key agreement: PASS
Endpoint spatial agreement: PASS

DATA-DRIVEN COMPLETE-CELL THRESHOLDS
Population absolute-change quantiles: {'q25': 28.18177643418312, 'median': 267.9460836574435, 'q75': 1040.5260162549093}
Built-surface change m2 quantiles: {'q25': 12388.0, 'median': 35942.0, 'q75': 101618.0}
Built-fraction change quantiles: {'q25': 0.012388, 'median': 0.035942, 'q75': 0.101618}
Endpoint total conservation: PASS

GROWTH-CLASS SUMMARY
+----------------------------+---------------+------------------+------------------+--------------------------+---------------------------+---------------------------+----------

Cell 18 succeeded fully. The integrated population plus built-surface classification is now persistent in HDFS, with all 3,658 cells classified and all validation checks passing.

Data-driven thresholds

Thresholds were learned from the 3,480 complete cells, excluding partial pilot-boundary cells from threshold estimation:

Metric	25th percentile	Median	75th percentilePopulation change	+28	+268	+1,041 people
Built-surface change	+12,388 m²	+35,942 m²	+101,618 m²
Built-fraction change	+1.24 pp	+3.59 pp	+10.16 pp

These are relative thresholds within this Bengaluru pilot, not universal urban-growth standards.

Growth classification
Growth class	Cells	Population change	Built-surface changeModerate combined growth	1,715	+2.69M	+115.88 km²
Population decline	568	−150,432	+9.47 km²
Combined high growth	544	+8.11M	+126.73 km²
Mixed or other	438	+39,603	+5.87 km²
Stable low change	242	+1,627	+0.77 km²
Built-surface-led expansion	106	+17,348	+15.45 km²
Population-led densification	45	+834,540	+1.17 km²

Strongest findings
The 544 combined-high-growth cells generated about 70.2% of the total population increase and roughly 46.0% of the total built-surface increase.
The 45 population-led densification cells added approximately 834,540 people while adding only 1.17 km² of built surface. This is the strongest evidence of population intensification within relatively constrained physical expansion.
The 106 built-surface-led expansion cells added approximately 15.45 km² of built surface but only about 17,348 people, indicating low-population physical expansion, infrastructure, industrial/commercial development, or early-stage conversion.
Among rural-to-urban-centre transitions, 303 cells were combined-high-growth cells. Those cells added approximately 3.63 million people and 73.79 km² of built surface.
Among stable urban-centre cells, 85 cells were combined-high-growth and 24 cells were population-led densification, showing that the established core experienced both physical redevelopment and strong intensification.
Important caution

The population_decline class contains 568 cells, but those cells still show an aggregate increase of about 9.47 km² in built surface. This could reflect:

redistribution of population toward newer urban areas;
industrial/commercial development;
changes in residential occupancy;
GHSL estimation changes;
local classification or boundary effects.

So it should be interpreted as “population decline between endpoint estimates,” not necessarily actual abandonment.

Persistent outputs
Cell-level integrated changes:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/cell_change
Growth-class summary:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/growth_class_summary
Growth by settlement transition:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/growth_by_settlement_transition
Learned thresholds:
Plain Text
hdfs://localhost:9000/data/ghsl/analytics/pilots/bengaluru/integrated_change_1975_2020/thresholds

What this means for project completion

The Bengaluru 1975-to-2020 endpoint prototype is now functionally complete for the three core products:

Plain Text
Population + SMOD + built surface

The next engineering step should not be another endpoint calculation. We should now parameterize the validated pipeline across the eight intermediate historical epochs:

Plain Text
1980, 1985, 1990, 1995,
2000, 2005, 2010, 2015

That will let us determine when each grid cell transitioned, not merely the difference between 1975 and 2020.